# ProductLens 3.0: Stage 2 Teacher Evaluation on Gold Benchmark
> **Hardware Target:** Kaggle Tesla T4 GPU (16 GB VRAM)  
> **Model:** `Qwen/Qwen3.5-9B` (4-bit via `bitsandbytes`, `dtype=float16`, `language_model_only=True`, `max_model_len=2048`)  
> **Decoding Engine:** vLLM with `StructuredOutputsParams` (strict Pydantic JSON schema)  
> **Goal:** Benchmark upper-bound teacher performance on the 350-review Gold Set before launching the 20k–50k silver labeling run.

In [ ]:
# [Step 1/5] Install dependencies via uv (fast 1-2 min install)
import subprocess, sys

print("Installing uv and dependencies...")
subprocess.run(
    f"{sys.executable} -m pip install -q uv && uv pip install --system --quiet vllm vllm-bnb-plugin bitsandbytes pydantic scikit-learn pandas",
    shell=True,
    check=True
)
print("✅ Dependencies installed successfully via uv!")

In [ ]:
# [Step 2/5] Hardware & Dynamic CUDA Linker Setup
import os, subprocess, torch

print("Configuring dynamic libcuda.so symlinks for FlashInfer JIT linker...")

# 1. Dynamically locate the real CUDA driver library (libcuda.so.1 or libcuda.so)
candidate_paths = [
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/local/cuda/compat/libcuda.so.1",
    "/usr/local/nvidia/lib64/libcuda.so.1",
    "/usr/local/nvidia/lib/libcuda.so.1",
    "/usr/lib64/libcuda.so.1",
    "/usr/lib/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so",
    "/usr/local/cuda/lib64/stubs/libcuda.so",
]
libcuda = next((p for p in candidate_paths if os.path.exists(p)), None)

if not libcuda:
    # Refresh ldconfig cache and retry
    try:
        subprocess.run("ldconfig", shell=True)
        out = subprocess.check_output(
            "ldconfig -p | awk '/libcuda\\.so/ {print $NF; exit}'",
            shell=True,
            text=True,
        ).strip()
        if out and os.path.exists(out):
            libcuda = out
    except Exception:
        pass

if not libcuda:
    # Direct search across system library roots
    try:
        out = subprocess.check_output(
            "find /usr /lib* /usr/local/nvidia -name 'libcuda.so*' 2>/dev/null | head -n 1",
            shell=True,
            text=True,
        ).strip()
        if out and os.path.exists(out):
            libcuda = out
    except Exception:
        pass

assert libcuda and os.path.exists(libcuda), f"libcuda driver library not found! Inspected: {candidate_paths}"
print(f"Found driver library at: {libcuda}")

# 2. Symlink libcuda.so into the exact -L directories FlashInfer searches
for dst in [
    "/usr/local/cuda/lib64/stubs/libcuda.so",
    "/usr/local/cuda/lib64/libcuda.so",
    "/usr/lib/x86_64-linux-gnu/libcuda.so",
    "/usr/lib/libcuda.so",
]:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.lexists(dst):
        try:
            os.remove(dst)
        except Exception:
            pass
    try:
        os.symlink(libcuda, dst)
    except Exception as e:
        print(f"Symlink warning for {dst}: {e}")

# 3. 1-second fail-fast linker smoke test (catches linker issues immediately)
subprocess.run(
    "c++ -shared -x c++ /dev/null -L/usr/local/cuda/lib64 -L/usr/local/cuda/lib64/stubs -lcudart -lcuda -o /tmp/test_lcuda.so",
    shell=True,
    check=True,
)
if os.path.exists("/tmp/test_lcuda.so"):
    os.remove("/tmp/test_lcuda.so")
print(f"✅ Linker verified: -lcuda resolved to {libcuda}")

# 4. Keep only valid vLLM env vars (do NOT add /stubs to LD_LIBRARY_PATH)
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# 5. Neutralize unused MiniMax M3 MSA warmup (unconditionally attempts 'import cutlass')
try:
    import vllm
    from pathlib import Path
    vllm_root = Path(vllm.__file__).parent
    
    mm_file = vllm_root / "model_executor" / "warmup" / "minimax_m3_msa_warmup.py"
    if mm_file.exists():
        mm_file.write_text("def minimax_m3_msa_warmup(*args, **kwargs):\n    pass\n")
        print(f"✅ Neutralized MiniMax M3 warmup at: {mm_file}")
        
    ops_file = vllm_root / "models" / "minimax_m3" / "nvidia" / "ops" / "index_decode_score.py"
    if ops_file.exists():
        ops_file.write_text("def minimax_m3_index_decode_score_cutedsl(*args, **kwargs):\n    pass\n")
        print(f"✅ Neutralized MiniMax M3 ops at: {ops_file}")
except Exception as e:
    print(f"Warmup patch note: {e}")

# Optional HF_TOKEN from Kaggle Secrets if available
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("🔑 HF_TOKEN configured from Kaggle Secrets.")
except Exception:
    pass

print(f"CUDA Available: {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"Device Count:   {device_count}")
for i in range(device_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({props.total_memory / (1024**3):.2f} GB VRAM)")

In [ ]:
# [Step 3/5] Setup ProductLens 3.0 Core Modules
import os, sys
from pathlib import Path

# Create package tree in /kaggle/working/productlens
pkg_dir = Path("/kaggle/working/productlens")
(pkg_dir / "schemas").mkdir(parents=True, exist_ok=True)
(pkg_dir / "validator").mkdir(parents=True, exist_ok=True)
(pkg_dir / "evaluation").mkdir(parents=True, exist_ok=True)
(pkg_dir / "gold").mkdir(parents=True, exist_ok=True)

(pkg_dir / "__init__.py").write_text('"""ProductLens 3.0 package."""\n')
(pkg_dir / "schemas" / "__init__.py").write_text('"""Schemas."""\n')
(pkg_dir / "validator" / "__init__.py").write_text('"""Validator."""\n')
(pkg_dir / "evaluation" / "__init__.py").write_text('"""Evaluation."""\n')
(pkg_dir / "gold" / "__init__.py").write_text('"""Gold."""\n')

(pkg_dir / "schemas" / "enums.py").write_text("\"\"\"\nEnumerations for ProductLens 3.0: Sentiments, Verticals, and Canonical Categories.\n\"\"\"\n\nfrom enum import Enum\nfrom typing import Set\n\n\nclass Sentiment(str, Enum):\n    \"\"\"Aspect-level sentiment polarity.\"\"\"\n    POSITIVE = \"positive\"\n    NEGATIVE = \"negative\"\n    NEUTRAL = \"neutral\"\n    MIXED = \"mixed\"\n\n    @classmethod\n    def from_str(cls, value: str) -> \"Sentiment\":\n        val = value.strip().lower()\n        for member in cls:\n            if member.value == val:\n                return member\n        raise ValueError(f\"Invalid sentiment '{value}'. Allowed: {[m.value for m in cls]}\")\n\n\nclass VerticalDomain(str, Enum):\n    \"\"\"Target e-commerce domains for ProductLens v1/v2.\"\"\"\n    ELECTRONICS_TECH = \"electronics_tech\"\n    APPAREL_FASHION = \"apparel_fashion\"\n\n\nclass ElectronicsCategory(str, Enum):\n    \"\"\"Canonical feature taxonomy for Electronics & Tech.\"\"\"\n    DISPLAY = \"DISPLAY\"\n    BATTERY = \"BATTERY\"\n    SOUND = \"SOUND\"\n    PERFORMANCE = \"PERFORMANCE\"\n    BUILD_QUALITY = \"BUILD_QUALITY\"\n    CONNECTIVITY = \"CONNECTIVITY\"\n    SOFTWARE = \"SOFTWARE\"\n    DELIVERY_INSTALLATION = \"DELIVERY_INSTALLATION\"\n    VALUE = \"VALUE\"\n    OTHER = \"OTHER\"\n\n    @classmethod\n    def values(cls) -> Set[str]:\n        return {item.value for item in cls}\n\n\nclass ApparelCategory(str, Enum):\n    \"\"\"Canonical feature taxonomy for Apparel & Fashion.\"\"\"\n    FABRIC_MATERIAL = \"FABRIC_MATERIAL\"\n    FIT_SIZING = \"FIT_SIZING\"\n    COLOR_PRINT = \"COLOR_PRINT\"\n    STITCHING_DURABILITY = \"STITCHING_DURABILITY\"\n    COMFORT = \"COMFORT\"\n    DELIVERY = \"DELIVERY\"\n    VALUE = \"VALUE\"\n    OTHER = \"OTHER\"\n\n    @classmethod\n    def values(cls) -> Set[str]:\n        return {item.value for item in cls}\n\n\ndef normalize_category(category: str, vertical: VerticalDomain | str) -> str:\n    \"\"\"\n    Validates and normalizes category against the vertical's canonical taxonomy.\n    \"\"\"\n    cat_upper = category.strip().upper().replace(\" \", \"_\")\n    vert = VerticalDomain(vertical) if isinstance(vertical, str) else vertical\n\n    if vert == VerticalDomain.ELECTRONICS_TECH:\n        allowed = ElectronicsCategory.values()\n    elif vert == VerticalDomain.APPAREL_FASHION:\n        allowed = ApparelCategory.values()\n    else:\n        raise ValueError(f\"Unsupported vertical domain: {vertical}\")\n\n    if cat_upper in allowed:\n        return cat_upper\n    return \"OTHER\"\n")
(pkg_dir / "schemas" / "quads.py").write_text("\"\"\"\nQuad definitions and compact serialization for ProductLens 3.0.\n\"\"\"\n\nfrom __future__ import annotations\nfrom typing import Optional\nfrom pydantic import BaseModel, Field, model_validator\nfrom productlens.schemas.enums import Sentiment, VerticalDomain, normalize_category\n\n\nclass RawCompactQuad(BaseModel):\n    \"\"\"\n    Unvalidated quad extracted from compact format or LLM generation.\n    Format: aspect|category|sentiment|confidence|evidence\n    \"\"\"\n    aspect: Optional[str] = None\n    category: str\n    sentiment: Sentiment\n    confidence: float = Field(default=1.0, ge=0.0, le=1.0)\n    evidence: str\n\n    @classmethod\n    def from_compact_line(cls, line: str) -> \"RawCompactQuad\":\n        parts = [p.strip() for p in line.split(\"|\")]\n        if len(parts) < 5:\n            raise ValueError(f\"Malformed compact quad line: '{line}'. Expected 5 pipe-delimited fields.\")\n\n        raw_aspect = parts[0]\n        aspect = None if raw_aspect.lower() in {\"\", \"none\", \"null\"} else raw_aspect\n        category = parts[1]\n        sentiment = Sentiment.from_str(parts[2])\n        try:\n            confidence = float(parts[3])\n        except ValueError:\n            confidence = 1.0\n        evidence = \"|\".join(parts[4:]).strip()  # Allow evidence containing pipes\n\n        return cls(\n            aspect=aspect,\n            category=category,\n            sentiment=sentiment,\n            confidence=max(0.0, min(1.0, confidence)),\n            evidence=evidence\n        )\n\n    def to_compact_line(self) -> str:\n        asp_str = self.aspect if self.aspect is not None else \"null\"\n        return f\"{asp_str}|{self.category}|{self.sentiment.value}|{self.confidence:.2f}|{self.evidence}\"\n\n\nclass AspectQuad(BaseModel):\n    \"\"\"\n    Fully validated ABSA quad with exact character offsets strictly verified in review text.\n    \"\"\"\n    aspect: Optional[str] = Field(default=None, description=\"Aspect term or None for implicit aspects\")\n    category: str = Field(..., description=\"Canonical category (e.g. BATTERY, FABRIC_MATERIAL)\")\n    sentiment: Sentiment = Field(..., description=\"positive, negative, neutral, mixed\")\n    confidence: float = Field(default=1.0, ge=0.0, le=1.0, description=\"Confidence score [0.0, 1.0]\")\n    evidence: str = Field(..., description=\"Exact supporting evidence clause in source text\")\n\n    # Character offsets in the parent review text\n    aspect_start: Optional[int] = Field(default=None, ge=0)\n    aspect_end: Optional[int] = Field(default=None, ge=0)\n    evidence_start: int = Field(..., ge=0)\n    evidence_end: int = Field(..., ge=0)\n\n    @property\n    def is_implicit(self) -> bool:\n        return self.aspect is None\n\n    @model_validator(mode=\"after\")\n    def validate_offsets(self) -> \"AspectQuad\":\n        if self.evidence_end <= self.evidence_start:\n            raise ValueError(\n                f\"Invalid evidence offsets: start ({self.evidence_start}) >= end ({self.evidence_end})\"\n            )\n        if self.aspect is not None:\n            if self.aspect_start is None or self.aspect_end is None:\n                raise ValueError(\"Explicit aspect requires non-null aspect_start and aspect_end offsets.\")\n            if self.aspect_end <= self.aspect_start:\n                raise ValueError(\n                    f\"Invalid aspect offsets: start ({self.aspect_start}) >= end ({self.aspect_end})\"\n                )\n        else:\n            if self.aspect_start is not None or self.aspect_end is not None:\n                raise ValueError(\"Implicit aspect (aspect=None) must have null aspect offsets.\")\n        return self\n\n    def to_compact_line(self) -> str:\n        asp_str = self.aspect if self.aspect is not None else \"null\"\n        return f\"{asp_str}|{self.category}|{self.sentiment.value}|{self.confidence:.2f}|{self.evidence}\"\n")
(pkg_dir / "validator" / "substring.py").write_text("\"\"\"\nDeterministic Substring Locator and Offset Calculator for ProductLens 3.0.\nSolves the offset mismatch / hallucination bug by strictly computing start_char and end_char in code.\nIncludes multi-tier normalized snapping and clause alignment.\n\"\"\"\n\nfrom __future__ import annotations\nimport re\nfrom typing import List, Optional, Tuple\nfrom productlens.schemas.enums import VerticalDomain, normalize_category\nfrom productlens.schemas.quads import AspectQuad, RawCompactQuad\n\n\ndef find_all_occurrences(text: str, sub: str, case_sensitive: bool = True) -> List[Tuple[int, int]]:\n    \"\"\"\n    Finds all [start, end) character spans of sub in text.\n    \"\"\"\n    if not sub:\n        return []\n\n    spans: List[Tuple[int, int]] = []\n    if case_sensitive:\n        start = 0\n        while True:\n            idx = text.find(sub, start)\n            if idx == -1:\n                break\n            spans.append((idx, idx + len(sub)))\n            start = idx + 1\n    else:\n        pattern = re.escape(sub)\n        for match in re.finditer(pattern, text, re.IGNORECASE):\n            spans.append((match.start(), match.end()))\n\n    return spans\n\n\ndef find_normalized_spans(text: str, query: str, is_evidence: bool = False) -> List[Tuple[int, int]]:\n    \"\"\"\n    Finds character spans of query in text with multi-tier deterministic fallback:\n    1. Exact match (case-sensitive)\n    2. Case-insensitive match\n    3. Punctuation/quote-stripped match: strips edge quotes, brackets, and terminal punctuation\n    4. Collapsed whitespace match: regex matching allowing multiple spaces/newlines\n    5. Clause-level substring alignment (if is_evidence=True and evidence spans multiple clauses)\n    \"\"\"\n    if not query:\n        return []\n\n    # Tier 1: Exact case-sensitive\n    spans = find_all_occurrences(text, query, case_sensitive=True)\n    if spans:\n        return spans\n\n    # Tier 2: Case-insensitive\n    spans = find_all_occurrences(text, query, case_sensitive=False)\n    if spans:\n        return spans\n\n    # Tier 3: Edge punctuation / quote stripping\n    clean_query = query.strip(\" \\t\\n\\r\\\"'.,;:!?-\u2013\u2014()[]{}*`\")\n    if clean_query:\n        spans = find_all_occurrences(text, clean_query, case_sensitive=False)\n        if spans:\n            return spans\n\n    # Tier 4: Collapsed whitespace matching\n    if clean_query:\n        words = [re.escape(w) for w in clean_query.split() if w]\n        if len(words) > 1:\n            pattern = r\"\\s+\".join(words)\n            spans = [(m.start(), m.end()) for m in re.finditer(pattern, text, re.IGNORECASE)]\n            if spans:\n                return spans\n\n    # Tier 5: Clause-level alignment for evidence\n    if is_evidence and clean_query:\n        # Split on clause punctuation (., ;, ,)\n        clauses = re.split(r\"[,;.]\\s*\", clean_query)\n        for clause in clauses:\n            c_clean = clause.strip(\" \\t\\n\\r\\\"'.,;:!?-\u2013\u2014()[]{}*`\")\n            words = [re.escape(w) for w in c_clean.split() if w]\n            if len(words) >= 2:\n                pattern = r\"\\s+\".join(words)\n                spans = [(m.start(), m.end()) for m in re.finditer(pattern, text, re.IGNORECASE)]\n                if spans:\n                    return spans\n\n    return []\n\n\nCATEGORY_KEYWORDS = {\n    \"DISPLAY\": [\"display\", \"screen\", \"panel\", \"oled\", \"brightness\", \"refresh rate\", \"resolution\", \"pixel\"],\n    \"BATTERY\": [\"battery\", \"charger\", \"charging\", \"battery backup\", \"backup\", \"power\"],\n    \"SOUND\": [\"sound\", \"audio\", \"speaker\", \"speakers\", \"volume\", \"bass\", \"mic\", \"microphone\"],\n    \"PERFORMANCE\": [\"performance\", \"processor\", \"speed\", \"lag\", \"cpu\", \"gpu\", \"ssd\", \"ram\", \"thermal\", \"heating\"],\n    \"BUILD_QUALITY\": [\"build\", \"quality\", \"construction\", \"body\", \"hinge\", \"chassis\", \"material\", \"finish\"],\n    \"CONNECTIVITY\": [\"connectivity\", \"bluetooth\", \"wifi\", \"ports\", \"port\", \"usb\", \"hdmi\", \"headphone jack\"],\n    \"SOFTWARE\": [\"software\", \"os\", \"app\", \"firmware\", \"windows\", \"driver\", \"ui\"],\n    \"DELIVERY_INSTALLATION\": [\"delivery\", \"installation\", \"shipping\", \"courier\", \"package\"],\n    \"VALUE\": [\"price\", \"value\", \"money\", \"worth\", \"cost\", \"pricing\"],\n    \"FABRIC_MATERIAL\": [\"fabric\", \"material\", \"cloth\", \"cotton\", \"polyester\", \"wool\", \"silk\"],\n    \"FIT_SIZING\": [\"sizing\", \"size\", \"fit\", \"fitting\", \"cut\"],\n    \"COLOR_PRINT\": [\"color\", \"colour\", \"print\", \"shade\", \"dye\", \"pattern\"],\n    \"STITCHING_DURABILITY\": [\"stitching\", \"stitch\", \"durability\", \"seam\", \"thread\", \"dhaaga\"],\n    \"COMFORT\": [\"comfort\", \"comfortable\", \"feel\"],\n    \"DELIVERY\": [\"delivery\", \"shipping\", \"courier\", \"package\"],\n}\n\n\nclass SubstringValidator:\n    \"\"\"\n    Deterministic validator that resolves exact character spans for aspect and evidence.\n    Applies multi-tier normalized snapping, clause alignment, and enum-fallback before hard rejection.\n    \"\"\"\n\n    @classmethod\n    def validate_quad(\n        cls,\n        text: str,\n        raw: RawCompactQuad,\n        vertical: VerticalDomain | str,\n        case_sensitive: bool = True,\n    ) -> Tuple[Optional[AspectQuad], Optional[str]]:\n        \"\"\"\n        Locates evidence and aspect within text, validates containment,\n        and constructs a fully verified AspectQuad.\n        Returns: (AspectQuad, None) on success or (None, error_str) on failure.\n        \"\"\"\n        normalized_category = normalize_category(raw.category, vertical)\n\n        # 0. Enum-to-Noun Fallback: If raw.aspect is an uppercase category tag not present in text,\n        # extract the matching aspect noun from evidence or treat as implicit\n        aspect_query = raw.aspect\n        if aspect_query and aspect_query.isupper() and not find_normalized_spans(text, aspect_query):\n            cat_key = str(raw.category).upper()\n            found_kw = None\n            ev_text = raw.evidence or \"\"\n            for kw in CATEGORY_KEYWORDS.get(cat_key, []):\n                if re.search(r\"\\b\" + re.escape(kw) + r\"\\b\", ev_text, re.IGNORECASE):\n                    found_kw = kw\n                    break\n            if found_kw:\n                aspect_query = found_kw\n            elif cat_key in [\"VALUE\", \"COMFORT\"]:\n                aspect_query = None\n\n        # 1. Locate Evidence (exact, case-insensitive, edge-stripped, or clause-aligned)\n        if case_sensitive:\n            evidence_spans = find_all_occurrences(text, raw.evidence, case_sensitive=True)\n            if not evidence_spans:\n                evidence_spans = find_normalized_spans(text, raw.evidence, is_evidence=True)\n        else:\n            evidence_spans = find_normalized_spans(text, raw.evidence, is_evidence=True)\n\n        if not evidence_spans:\n            return None, f\"Evidence '{raw.evidence}' not found in review text.\"\n\n        # 2. Implicit Aspect case (aspect is None)\n        if aspect_query is None:\n            best_ev_start, best_ev_end = evidence_spans[0]\n            exact_evidence = text[best_ev_start:best_ev_end]\n            quad = AspectQuad(\n                aspect=None,\n                category=normalized_category,\n                sentiment=raw.sentiment,\n                confidence=raw.confidence,\n                evidence=exact_evidence,\n                aspect_start=None,\n                aspect_end=None,\n                evidence_start=best_ev_start,\n                evidence_end=best_ev_end,\n            )\n            return quad, None\n\n        # 3. Explicit Aspect case: locate aspect within evidence span\n        if case_sensitive:\n            aspect_spans = find_all_occurrences(text, aspect_query, case_sensitive=True)\n            if not aspect_spans:\n                aspect_spans = find_normalized_spans(text, aspect_query, is_evidence=False)\n        else:\n            aspect_spans = find_normalized_spans(text, aspect_query, is_evidence=False)\n\n        if not aspect_spans:\n            return None, f\"Aspect '{aspect_query}' not found in review text.\"\n\n        # Disambiguate: Find pair where aspect is strictly inside evidence span\n        matched_pair: Optional[Tuple[Tuple[int, int], Tuple[int, int]]] = None\n        for ev_s, ev_e in evidence_spans:\n            for asp_s, asp_e in aspect_spans:\n                if ev_s <= asp_s and asp_e <= ev_e:\n                    matched_pair = ((asp_s, asp_e), (ev_s, ev_e))\n                    break\n            if matched_pair:\n                break\n\n        # Fallback if aspect and evidence are distinct clauses in review text\n        if matched_pair is None:\n            # Pick first occurrence of each\n            matched_pair = (aspect_spans[0], evidence_spans[0])\n\n        (asp_start, asp_end), (ev_start, ev_end) = matched_pair\n        exact_aspect = text[asp_start:asp_end]\n        exact_evidence = text[ev_start:ev_end]\n\n        try:\n            quad = AspectQuad(\n                aspect=exact_aspect,\n                category=normalized_category,\n                sentiment=raw.sentiment,\n                confidence=raw.confidence,\n                evidence=exact_evidence,\n                aspect_start=asp_start,\n                aspect_end=asp_end,\n                evidence_start=ev_start,\n                evidence_end=ev_end,\n            )\n            return quad, None\n        except Exception as e:\n            return None, f\"Quad construction failed: {str(e)}\"\n")
(pkg_dir / "evaluation" / "metrics.py").write_text("\"\"\"\nEvaluation metrics for ProductLens 3.0:\nExplicit Aspect F1 (multiset Counter), Implicit Category F1, Sentiment Macro-F1 (explicit + implicit),\nOffset Validity, Compact Parse Rate, ECE, and ARTS Accuracy.\n\"\"\"\n\nfrom __future__ import annotations\nfrom collections import Counter\nfrom typing import Dict, List, Optional, Tuple\nimport numpy as np\nfrom sklearn.metrics import f1_score\nfrom productlens.schemas.enums import Sentiment\nfrom productlens.schemas.quads import AspectQuad\n\n\ndef compute_char_iou(span1: Tuple[int, int], span2: Tuple[int, int]) -> float:\n    \"\"\"\n    Computes Character-level Intersection over Union (IoU) between two [start, end) character spans.\n    \"\"\"\n    start1, end1 = span1\n    start2, end2 = span2\n    inter = max(0, min(end1, end2) - max(start1, start2))\n    union = max(end1, end2) - min(start1, start2)\n    return inter / union if union > 0 else 0.0\n\n\ndef compute_explicit_aspect_spans_f1(\n    gold_quads_list: List[List[AspectQuad]],\n    pred_quads_list: List[List[AspectQuad]],\n    iou_threshold: float = 0.5,\n) -> Dict[str, any]:\n    \"\"\"\n    Computes Precision, Recall, and F1 for explicit aspects:\n      1. Strict Exact Offset Matching: multiset matching by (aspect.lower().strip(), start, end)\n      2. Overlap IoU Matching: greedy bipartite matching where character IoU >= iou_threshold\n    \"\"\"\n    tp = 0\n    fp = 0\n    fn = 0\n    overlap_tp = 0\n    total_gold_explicit = 0\n    total_pred_explicit = 0\n\n    for gold_quads, pred_quads in zip(gold_quads_list, pred_quads_list):\n        gold_explicit = [q for q in gold_quads if not q.is_implicit and q.aspect is not None]\n        pred_explicit = [q for q in pred_quads if not q.is_implicit and q.aspect is not None]\n\n        total_gold_explicit += len(gold_explicit)\n        total_pred_explicit += len(pred_explicit)\n\n        # 1. Strict exact offset matching\n        gold_counts = Counter(\n            (q.aspect.lower().strip(), q.aspect_start, q.aspect_end)\n            for q in gold_explicit\n        )\n        pred_counts = Counter(\n            (q.aspect.lower().strip(), q.aspect_start, q.aspect_end)\n            for q in pred_explicit\n        )\n\n        matched_counts = gold_counts & pred_counts\n        matched_in_review = sum(matched_counts.values())\n\n        tp += matched_in_review\n        fp += sum((pred_counts - gold_counts).values())\n        fn += sum((gold_counts - pred_counts).values())\n\n        # 2. Overlap IoU >= 0.5 greedy matching\n        candidate_pairs = []\n        for g_idx, gq in enumerate(gold_explicit):\n            if gq.aspect_start is None or gq.aspect_end is None:\n                continue\n            for p_idx, pq in enumerate(pred_explicit):\n                if pq.aspect_start is None or pq.aspect_end is None:\n                    continue\n                iou = compute_char_iou((gq.aspect_start, gq.aspect_end), (pq.aspect_start, pq.aspect_end))\n                if iou >= iou_threshold:\n                    candidate_pairs.append((iou, g_idx, p_idx))\n\n        candidate_pairs.sort(key=lambda x: x[0], reverse=True)\n        used_gold = set()\n        used_pred = set()\n        for iou, g_idx, p_idx in candidate_pairs:\n            if g_idx not in used_gold and p_idx not in used_pred:\n                used_gold.add(g_idx)\n                used_pred.add(p_idx)\n                overlap_tp += 1\n\n    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0\n\n    overlap_prec = overlap_tp / total_pred_explicit if total_pred_explicit > 0 else 0.0\n    overlap_rec = overlap_tp / total_gold_explicit if total_gold_explicit > 0 else 0.0\n    overlap_f1 = (\n        2 * overlap_prec * overlap_rec / (overlap_prec + overlap_rec)\n        if (overlap_prec + overlap_rec) > 0\n        else 0.0\n    )\n\n    return {\n        \"aspect_precision\": round(precision, 4),\n        \"aspect_recall\": round(recall, 4),\n        \"aspect_f1\": round(f1, 4),\n        \"strict_aspect_f1\": round(f1, 4),\n        \"overlap_aspect_precision\": round(overlap_prec, 4),\n        \"overlap_aspect_recall\": round(overlap_rec, 4),\n        \"overlap_aspect_f1\": round(overlap_f1, 4),\n        \"explicit_matched_count\": tp,\n        \"explicit_overlap_matched_count\": overlap_tp,\n        \"total_gold_explicit\": total_gold_explicit,\n        \"total_pred_explicit\": total_pred_explicit,\n    }\n\n\ndef compute_implicit_category_f1(\n    gold_quads_list: List[List[AspectQuad]],\n    pred_quads_list: List[List[AspectQuad]],\n) -> Dict[str, any]:\n    \"\"\"\n    Computes Precision, Recall, and F1 for implicit aspects (aspect: null)\n    matched by canonical category.\n    \"\"\"\n    tp = 0\n    fp = 0\n    fn = 0\n    total_gold_implicit = 0\n    total_pred_implicit = 0\n\n    for gold_quads, pred_quads in zip(gold_quads_list, pred_quads_list):\n        gold_implicit = [q for q in gold_quads if q.is_implicit]\n        pred_implicit = [q for q in pred_quads if q.is_implicit]\n\n        total_gold_implicit += len(gold_implicit)\n        total_pred_implicit += len(pred_implicit)\n\n        gold_counts = Counter(q.category for q in gold_implicit)\n        pred_counts = Counter(q.category for q in pred_implicit)\n\n        matched_counts = gold_counts & pred_counts\n        matched_in_review = sum(matched_counts.values())\n\n        tp += matched_in_review\n        fp += sum((pred_counts - gold_counts).values())\n        fn += sum((gold_counts - pred_counts).values())\n\n    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0\n\n    return {\n        \"implicit_category_precision\": round(precision, 4),\n        \"implicit_category_recall\": round(recall, 4),\n        \"implicit_category_f1\": round(f1, 4),\n        \"implicit_matched_count\": tp,\n        \"total_gold_implicit\": total_gold_implicit,\n        \"total_pred_implicit\": total_pred_implicit,\n    }\n\n\ndef compute_sentiment_macro_f1(\n    gold_quads_list: List[List[AspectQuad]],\n    pred_quads_list: List[List[AspectQuad]],\n) -> Dict[str, any]:\n    \"\"\"\n    Computes Sentiment Macro-F1 across all matched aspect spans (both explicit and implicit)\n    across all 4 classes: positive, negative, neutral, mixed.\n    \"\"\"\n    y_true: List[str] = []\n    y_pred: List[str] = []\n\n    for gold_quads, pred_quads in zip(gold_quads_list, pred_quads_list):\n        # 1. Match explicit aspects: multiset matching by (aspect.lower(), start, end)\n        pred_explicit_pool = [q for q in pred_quads if not q.is_implicit and q.aspect is not None]\n        for gq in [q for q in gold_quads if not q.is_implicit and q.aspect is not None]:\n            g_key = (gq.aspect.lower().strip(), gq.aspect_start, gq.aspect_end)\n            for idx, pq in enumerate(pred_explicit_pool):\n                p_key = (pq.aspect.lower().strip(), pq.aspect_start, pq.aspect_end)\n                if g_key == p_key:\n                    y_true.append(gq.sentiment.value)\n                    y_pred.append(pq.sentiment.value)\n                    pred_explicit_pool.pop(idx)\n                    break\n\n        # 2. Match implicit aspects: match by category\n        pred_implicit_pool = [q for q in pred_quads if q.is_implicit]\n        for gq in [q for q in gold_quads if q.is_implicit]:\n            for idx, pq in enumerate(pred_implicit_pool):\n                if gq.category == pq.category:\n                    y_true.append(gq.sentiment.value)\n                    y_pred.append(pq.sentiment.value)\n                    pred_implicit_pool.pop(idx)\n                    break\n\n    if not y_true:\n        return {\"sentiment_macro_f1\": 0.0, \"matched_aspect_count\": 0}\n\n    present_classes = sorted(list(set(y_true) | set(y_pred)))\n    macro_f1 = f1_score(y_true, y_pred, labels=present_classes, average=\"macro\", zero_division=0)\n\n    # Per-class F1 for all defined sentiments\n    all_classes = [s.value for s in Sentiment]\n    per_class = f1_score(y_true, y_pred, labels=all_classes, average=None, zero_division=0)\n    per_class_f1 = {f\"f1_{cls_name}\": round(float(score), 4) for cls_name, score in zip(all_classes, per_class)}\n\n    res = {\n        \"sentiment_macro_f1\": round(float(macro_f1), 4),\n        \"matched_aspect_count\": len(y_true),\n    }\n    res.update(per_class_f1)\n    return res\n\n\ndef compute_expected_calibration_error(\n    confidences: List[float],\n    correctness: List[int],\n    n_bins: int = 10,\n) -> float:\n    \"\"\"\n    Computes Expected Calibration Error (ECE) with equal-width confidence bins.\n    \"\"\"\n    if not confidences or not correctness or len(confidences) != len(correctness):\n        return 0.0\n\n    conf_arr = np.array(confidences)\n    corr_arr = np.array(correctness)\n    bin_boundaries = np.linspace(0, 1, n_bins + 1)\n    ece = 0.0\n\n    for i in range(n_bins):\n        bin_lower = bin_boundaries[i]\n        bin_upper = bin_boundaries[i + 1]\n        in_bin = (conf_arr > bin_lower) & (conf_arr <= bin_upper) if i > 0 else (conf_arr >= bin_lower) & (conf_arr <= bin_upper)\n        prop_in_bin = np.mean(in_bin)\n\n        if prop_in_bin > 0:\n            avg_confidence = np.mean(conf_arr[in_bin])\n            avg_accuracy = np.mean(corr_arr[in_bin])\n            ece += np.abs(avg_accuracy - avg_confidence) * prop_in_bin\n\n    return round(float(ece), 4)\n\n\ndef compute_parse_and_offset_rates(\n    total_raw_lines: int,\n    parsed_lines: int,\n    validated_quads: int,\n) -> Dict[str, float]:\n    \"\"\"\n    Calculates parse rate and deterministic offset validity rate.\n    \"\"\"\n    parse_rate = (parsed_lines / total_raw_lines) if total_raw_lines > 0 else 1.0\n    offset_validity_rate = (validated_quads / parsed_lines) if parsed_lines > 0 else 1.0\n\n    return {\n        \"parse_rate\": round(parse_rate, 4),\n        \"offset_validity_rate\": round(offset_validity_rate, 4),\n    }\n")
(pkg_dir / "gold" / "schema.py").write_text("\"\"\"\nGold Set Schema, Partitioning, and Hard Case definitions.\nSupports English and Hinglish (Latin-script Hindi + English) reviews with negation, sarcasm, and implicit aspects.\n\"\"\"\n\nfrom __future__ import annotations\nfrom typing import List, Optional\nfrom pydantic import BaseModel, Field\nfrom productlens.schemas.enums import Sentiment, VerticalDomain\nfrom productlens.schemas.quads import AspectQuad\n\n\nclass GoldReview(BaseModel):\n    \"\"\"A curated, gold-standard review for benchmarking.\"\"\"\n    review_id: str\n    product_id: str\n    text: str\n    vertical: VerticalDomain\n    language: str = Field(default=\"en\", description=\"'en' for English or 'hi-en' for Hinglish\")\n    is_hard_case: bool = False\n    hard_case_type: Optional[str] = Field(\n        default=None,\n        description=\"negation, sarcasm, implicit_aspect, cross_clause_contrast, etc.\"\n    )\n    quads: List[AspectQuad] = Field(default_factory=list)\n\n\nclass GoldSetManifest(BaseModel):\n    \"\"\"Manifest summary for the gold set.\"\"\"\n    total_reviews: int\n    english_reviews: int\n    hinglish_reviews: int\n    hard_cases: int\n    total_quads: int\n    products_count: int\n")
(pkg_dir / "gold" / "manager.py").write_text("\"\"\"\nGold Set Manager: Loads, validates, and partitions the curated gold benchmark set.\n\"\"\"\n\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Dict, List\nfrom productlens.gold.schema import GoldReview, GoldSetManifest\nfrom productlens.schemas.enums import VerticalDomain\n\n\nDEFAULT_GOLD_PATH = Path(__file__).parent / \"gold_dataset.jsonl\"\n\n\nclass GoldSetManager:\n    \"\"\"Manages the curated gold benchmark dataset.\"\"\"\n\n    @staticmethod\n    def load_reviews(path: Path | str = DEFAULT_GOLD_PATH) -> List[GoldReview]:\n        filepath = Path(path)\n        if not filepath.exists():\n            raise FileNotFoundError(f\"Gold set file does not exist at {filepath}\")\n\n        reviews: List[GoldReview] = []\n        with open(filepath, \"r\", encoding=\"utf-8\") as f:\n            for line in f:\n                line = line.strip()\n                if line:\n                    reviews.append(GoldReview.model_validate_json(line))\n        return reviews\n\n    @classmethod\n    def get_manifest(cls, reviews: List[GoldReview]) -> GoldSetManifest:\n        products = {r.product_id for r in reviews}\n        english_count = sum(1 for r in reviews if r.language == \"en\")\n        hinglish_count = sum(1 for r in reviews if r.language == \"hi-en\")\n        hard_count = sum(1 for r in reviews if r.is_hard_case)\n        quad_count = sum(len(r.quads) for r in reviews)\n\n        return GoldSetManifest(\n            total_reviews=len(reviews),\n            english_reviews=english_count,\n            hinglish_reviews=hinglish_count,\n            hard_cases=hard_count,\n            total_quads=quad_count,\n            products_count=len(products),\n        )\n\n    @classmethod\n    def split_by_language(cls, reviews: List[GoldReview]) -> Dict[str, List[GoldReview]]:\n        return {\n            \"en\": [r for r in reviews if r.language == \"en\"],\n            \"hi-en\": [r for r in reviews if r.language == \"hi-en\"],\n        }\n")
(pkg_dir / "evaluation" / "runner.py").write_text("\"\"\"\nStage 1 Evaluation Runner for ProductLens 3.0.\nExecutes benchmarks against gold set, disaggregating English vs. Hinglish and hard cases.\nSupports explicit multiset matching and implicit category tracking.\n\"\"\"\n\nfrom __future__ import annotations\nimport time\nfrom typing import Callable, Dict, List, Optional\nfrom productlens.gold.schema import GoldReview\nfrom productlens.gold.manager import GoldSetManager\nfrom productlens.schemas.quads import AspectQuad\nfrom productlens.evaluation.metrics import (\n    compute_explicit_aspect_spans_f1,\n    compute_implicit_category_f1,\n    compute_sentiment_macro_f1,\n    compute_expected_calibration_error,\n    compute_parse_and_offset_rates,\n)\n\n\nclass Stage1Evaluator:\n    \"\"\"Evaluates ABSA systems and compact parses on the Stage 1 Gold Set.\"\"\"\n\n    def __init__(self, reviews: Optional[List[GoldReview]] = None):\n        self.reviews = reviews if reviews is not None else GoldSetManager.load_reviews()\n\n    def evaluate_predictions(\n        self,\n        predict_fn: Callable[[GoldReview], List[AspectQuad]],\n    ) -> Dict[str, any]:\n        \"\"\"\n        Runs prediction function over the gold set and calculates comprehensive Stage 1 metrics.\n        \"\"\"\n        start_time = time.perf_counter()\n\n        gold_quads_all: List[List[AspectQuad]] = []\n        pred_quads_all: List[List[AspectQuad]] = []\n\n        # Subsets for disaggregated evaluation\n        lang_groups: Dict[str, Dict[str, List[List[AspectQuad]]]] = {\n            \"en\": {\"gold\": [], \"pred\": []},\n            \"hi-en\": {\"gold\": [], \"pred\": []},\n        }\n        hard_case_groups: Dict[str, Dict[str, List[List[AspectQuad]]]] = {\n            \"hard\": {\"gold\": [], \"pred\": []},\n            \"standard\": {\"gold\": [], \"pred\": []},\n        }\n\n        confidences: List[float] = []\n        correctness: List[int] = []\n\n        for r in self.reviews:\n            gold_quads = r.quads\n            pred_quads = predict_fn(r)\n\n            gold_quads_all.append(gold_quads)\n            pred_quads_all.append(pred_quads)\n\n            # Route by language\n            lang_key = r.language if r.language in lang_groups else \"en\"\n            lang_groups[lang_key][\"gold\"].append(gold_quads)\n            lang_groups[lang_key][\"pred\"].append(pred_quads)\n\n            # Route by hard case\n            hard_key = \"hard\" if r.is_hard_case else \"standard\"\n            hard_case_groups[hard_key][\"gold\"].append(gold_quads)\n            hard_case_groups[hard_key][\"pred\"].append(pred_quads)\n\n            # Calibration data: collect on matched explicit spans and implicit categories\n            pred_explicit_pool = [q for q in pred_quads if not q.is_implicit and q.aspect is not None]\n            for gq in [q for q in gold_quads if not q.is_implicit and q.aspect is not None]:\n                g_key = (gq.aspect.lower().strip(), gq.aspect_start, gq.aspect_end)\n                for idx, pq in enumerate(pred_explicit_pool):\n                    p_key = (pq.aspect.lower().strip(), pq.aspect_start, pq.aspect_end)\n                    if g_key == p_key:\n                        confidences.append(pq.confidence)\n                        correctness.append(1 if pq.sentiment == gq.sentiment else 0)\n                        pred_explicit_pool.pop(idx)\n                        break\n\n            pred_implicit_pool = [q for q in pred_quads if q.is_implicit]\n            for gq in [q for q in gold_quads if q.is_implicit]:\n                for idx, pq in enumerate(pred_implicit_pool):\n                    if gq.category == pq.category:\n                        confidences.append(pq.confidence)\n                        correctness.append(1 if pq.sentiment == gq.sentiment else 0)\n                        pred_implicit_pool.pop(idx)\n                        break\n\n        elapsed = time.perf_counter() - start_time\n        throughput_sec_per_500 = (elapsed / len(self.reviews)) * 500 if self.reviews else 0.0\n\n        # Helper to compute composite bundle for any group\n        def evaluate_group(gold_list: List[List[AspectQuad]], pred_list: List[List[AspectQuad]]) -> Dict[str, any]:\n            exp_metrics = compute_explicit_aspect_spans_f1(gold_list, pred_list)\n            imp_metrics = compute_implicit_category_f1(gold_list, pred_list)\n            sent_metrics = compute_sentiment_macro_f1(gold_list, pred_list)\n            return {\n                **exp_metrics,\n                **imp_metrics,\n                **sent_metrics,\n                \"total_matched_quads\": exp_metrics[\"explicit_matched_count\"] + imp_metrics[\"implicit_matched_count\"],\n            }\n\n        overall_metrics = evaluate_group(gold_quads_all, pred_quads_all)\n        ece = compute_expected_calibration_error(confidences, correctness)\n\n        en_metrics = evaluate_group(lang_groups[\"en\"][\"gold\"], lang_groups[\"en\"][\"pred\"])\n        hi_metrics = evaluate_group(lang_groups[\"hi-en\"][\"gold\"], lang_groups[\"hi-en\"][\"pred\"])\n        hard_metrics = evaluate_group(hard_case_groups[\"hard\"][\"gold\"], hard_case_groups[\"hard\"][\"pred\"])\n\n        return {\n            \"total_reviews\": len(self.reviews),\n            \"total_matched_quads\": overall_metrics[\"total_matched_quads\"],\n            \"explicit_matched_count\": overall_metrics[\"explicit_matched_count\"],\n            \"implicit_matched_count\": overall_metrics[\"implicit_matched_count\"],\n            \"overall\": {\n                **overall_metrics,\n                \"expected_calibration_error\": ece,\n            },\n            \"english\": en_metrics,\n            \"hinglish\": hi_metrics,\n            \"hard_cases\": hard_metrics,\n            \"latency\": {\n                \"elapsed_seconds\": round(elapsed, 4),\n                \"seconds_per_500_reviews\": round(throughput_sec_per_500, 4),\n            },\n        }\n")

# Add working directory to sys.path
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

print("✅ ProductLens 3.0 core modules staged successfully!")

In [ ]:
# [Step 4/5] Stage the 350-Review Gold Benchmark Dataset
gold_records = ["{\"review_id\": \"gold_rev_0001\", \"product_id\": \"PROD_ELEC_001\", \"text\": \"The OLED display is stunning and vibrant, but the battery drains in under 3 hours.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"OLED display\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"The OLED display is stunning and vibrant\", \"aspect_start\": 4, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 40}, {\"aspect\": \"battery\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the battery drains in under 3 hours\", \"aspect_start\": 50, \"aspect_end\": 57, \"evidence_start\": 46, \"evidence_end\": 81}]}", "{\"review_id\": \"gold_rev_0002\", \"product_id\": \"PROD_ELEC_001\", \"text\": \"Sound quality is rich and immersive, yet the companion software crashes constantly.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Sound quality\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Sound quality is rich and immersive\", \"aspect_start\": 0, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 35}, {\"aspect\": \"companion software\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the companion software crashes constantly\", \"aspect_start\": 45, \"aspect_end\": 63, \"evidence_start\": 41, \"evidence_end\": 82}]}", "{\"review_id\": \"gold_rev_0003\", \"product_id\": \"PROD_APP_001\", \"text\": \"The fabric feels exceptionally soft, but the stitching around the collar tore after one wash.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"The fabric feels exceptionally soft\", \"aspect_start\": 4, \"aspect_end\": 10, \"evidence_start\": 0, \"evidence_end\": 35}, {\"aspect\": \"stitching\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the stitching around the collar tore after one wash\", \"aspect_start\": 45, \"aspect_end\": 54, \"evidence_start\": 41, \"evidence_end\": 92}]}", "{\"review_id\": \"gold_rev_0004\", \"product_id\": \"PROD_APP_002\", \"text\": \"Color is exactly as pictured in photos, however the sizing runs uncomfortably tight.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Color is exactly as pictured in photos\", \"aspect_start\": 0, \"aspect_end\": 5, \"evidence_start\": 0, \"evidence_end\": 38}, {\"aspect\": \"sizing\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the sizing runs uncomfortably tight\", \"aspect_start\": 52, \"aspect_end\": 58, \"evidence_start\": 48, \"evidence_end\": 83}]}", "{\"review_id\": \"gold_rev_0005\", \"product_id\": \"PROD_ELEC_002\", \"text\": \"Do not be fooled by the high refresh rate; the panel does not eliminate ghosting.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"refresh rate\", \"category\": \"DISPLAY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"Do not be fooled by the high refresh rate\", \"aspect_start\": 29, \"aspect_end\": 41, \"evidence_start\": 0, \"evidence_end\": 41}, {\"aspect\": \"panel\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the panel does not eliminate ghosting\", \"aspect_start\": 47, \"aspect_end\": 52, \"evidence_start\": 43, \"evidence_end\": 80}]}", "{\"review_id\": \"gold_rev_0006\", \"product_id\": \"PROD_ELEC_002\", \"text\": \"The charger does not charge quickly at all despite advertised fast charging.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"charger\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The charger does not charge quickly at all\", \"aspect_start\": 4, \"aspect_end\": 11, \"evidence_start\": 0, \"evidence_end\": 42}]}", "{\"review_id\": \"gold_rev_0007\", \"product_id\": \"PROD_APP_002\", \"text\": \"The cotton material is not breathable and does not feel natural against the skin.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"cotton material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The cotton material is not breathable and does not feel natural\", \"aspect_start\": 4, \"aspect_end\": 19, \"evidence_start\": 0, \"evidence_end\": 63}]}", "{\"review_id\": \"gold_rev_0008\", \"product_id\": \"PROD_APP_003\", \"text\": \"The t-shirt is definitely not pre-shrunk as claimed; it shrank two full sizes.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"t-shirt\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"it shrank two full sizes\", \"aspect_start\": 4, \"aspect_end\": 11, \"evidence_start\": 53, \"evidence_end\": 77}]}", "{\"review_id\": \"gold_rev_0009\", \"product_id\": \"PROD_ELEC_003\", \"text\": \"Great, another gaming laptop where the cooling fan sounds like a Boeing 747 taking off.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"sarcasm\", \"quads\": [{\"aspect\": \"cooling fan\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"cooling fan sounds like a Boeing 747 taking off\", \"aspect_start\": 39, \"aspect_end\": 50, \"evidence_start\": 39, \"evidence_end\": 86}]}", "{\"review_id\": \"gold_rev_0010\", \"product_id\": \"PROD_ELEC_003\", \"text\": \"Oh wonderful, Bluetooth connectivity drops every 10 seconds right in the middle of meetings.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"sarcasm\", \"quads\": [{\"aspect\": \"Bluetooth connectivity\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Bluetooth connectivity drops every 10 seconds\", \"aspect_start\": 14, \"aspect_end\": 36, \"evidence_start\": 14, \"evidence_end\": 59}]}", "{\"review_id\": \"gold_rev_0011\", \"product_id\": \"PROD_APP_003\", \"text\": \"Brilliant design if you enjoy transparent fabric that exposes everything in daylight.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"sarcasm\", \"quads\": [{\"aspect\": \"fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"transparent fabric that exposes everything in daylight\", \"aspect_start\": 42, \"aspect_end\": 48, \"evidence_start\": 30, \"evidence_end\": 84}]}", "{\"review_id\": \"gold_rev_0012\", \"product_id\": \"PROD_ELEC_004\", \"text\": \"Completely overpriced for what it provides compared to rival models.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"implicit_aspect\", \"quads\": [{\"aspect\": null, \"category\": \"VALUE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Completely overpriced for what it provides\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 0, \"evidence_end\": 42}]}", "{\"review_id\": \"gold_rev_0013\", \"product_id\": \"PROD_ELEC_004\", \"text\": \"Worth every single penny and feels premium straight out of the box.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"implicit_aspect\", \"quads\": [{\"aspect\": null, \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Worth every single penny\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 0, \"evidence_end\": 24}, {\"aspect\": null, \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"feels premium straight out of the box\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 29, \"evidence_end\": 66}]}", "{\"review_id\": \"gold_rev_0014\", \"product_id\": \"PROD_APP_004\", \"text\": \"Total waste of money, faded instantly upon the first cold rinse.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"implicit_aspect\", \"quads\": [{\"aspect\": null, \"category\": \"VALUE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Total waste of money\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 0, \"evidence_end\": 20}, {\"aspect\": null, \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"faded instantly upon the first cold rinse\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 22, \"evidence_end\": 63}]}", "{\"review_id\": \"gold_rev_0015\", \"product_id\": \"PROD_ELEC_005\", \"text\": \"Screen quality ekdum zabardast hai par battery backup bilkul bekaar hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"Screen quality\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Screen quality ekdum zabardast hai\", \"aspect_start\": 0, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 34}, {\"aspect\": \"battery backup\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"battery backup bilkul bekaar hai\", \"aspect_start\": 39, \"aspect_end\": 53, \"evidence_start\": 39, \"evidence_end\": 71}]}", "{\"review_id\": \"gold_rev_0016\", \"product_id\": \"PROD_ELEC_005\", \"text\": \"Sound bohot clear hai aur volume bhi kafi loud hai bina kisi distortion ke.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"Sound\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Sound bohot clear hai\", \"aspect_start\": 0, \"aspect_end\": 5, \"evidence_start\": 0, \"evidence_end\": 21}, {\"aspect\": \"volume\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"volume bhi kafi loud hai\", \"aspect_start\": 26, \"aspect_end\": 32, \"evidence_start\": 26, \"evidence_end\": 50}]}", "{\"review_id\": \"gold_rev_0017\", \"product_id\": \"PROD_ELEC_006\", \"text\": \"Display toh mast hai lekin processor heavy games me bohot garam ho jata hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Display\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Display toh mast hai\", \"aspect_start\": 0, \"aspect_end\": 7, \"evidence_start\": 0, \"evidence_end\": 20}, {\"aspect\": \"processor\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"processor heavy games me bohot garam ho jata hai\", \"aspect_start\": 27, \"aspect_end\": 36, \"evidence_start\": 27, \"evidence_end\": 75}]}", "{\"review_id\": \"gold_rev_0018\", \"product_id\": \"PROD_ELEC_006\", \"text\": \"Fast delivery mili Flipkart se par installation team do din late aayi.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"delivery\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Fast delivery mili Flipkart se\", \"aspect_start\": 5, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 30}, {\"aspect\": \"installation\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"installation team do din late aayi\", \"aspect_start\": 35, \"aspect_end\": 47, \"evidence_start\": 35, \"evidence_end\": 69}]}", "{\"review_id\": \"gold_rev_0019\", \"product_id\": \"PROD_APP_005\", \"text\": \"T-shirt ka fabric bohot soft aur comfortable hai daily use ke liye.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"fabric bohot soft aur comfortable hai\", \"aspect_start\": 11, \"aspect_end\": 17, \"evidence_start\": 11, \"evidence_end\": 48}]}", "{\"review_id\": \"gold_rev_0020\", \"product_id\": \"PROD_APP_005\", \"text\": \"Color toh photo jaisa hi bright hai par size fitting thodi loose aayi hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Color toh photo jaisa hi bright hai\", \"aspect_start\": 0, \"aspect_end\": 5, \"evidence_start\": 0, \"evidence_end\": 35}, {\"aspect\": \"size fitting\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"size fitting thodi loose aayi hai\", \"aspect_start\": 40, \"aspect_end\": 52, \"evidence_start\": 40, \"evidence_end\": 73}]}", "{\"review_id\": \"gold_rev_0021\", \"product_id\": \"PROD_APP_006\", \"text\": \"Stitching bilkul acchi nahi hai, ek wash ke baad hi dhaaga nikal gaya.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Stitching\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Stitching bilkul acchi nahi hai\", \"aspect_start\": 0, \"aspect_end\": 9, \"evidence_start\": 0, \"evidence_end\": 31}]}", "{\"review_id\": \"gold_rev_0022\", \"product_id\": \"PROD_APP_006\", \"text\": \"Paisa vasool item hai boss, is price me aisi t-shirt milna mushkil hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"implicit_aspect\", \"quads\": [{\"aspect\": null, \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Paisa vasool item hai boss\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 0, \"evidence_end\": 26}]}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I like the Mini Mac it was easy to setup and install , but I am learning as I go and could use a tutorial to learn how to use some of the features I was use to on the PC especially the right mouse click menu .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"setup\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I like the Mini Mac it was easy to setup and install , but I am learning as I go and could use a tutorial to learn how to use some of the features I was use to on the PC especially the right mouse click menu .\", \"aspect_start\": 35, \"aspect_end\": 40, \"evidence_start\": 0, \"evidence_end\": 209}, {\"aspect\": \"install\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I like the Mini Mac it was easy to setup and install , but I am learning as I go and could use a tutorial to learn how to use some of the features I was use to on the PC especially the right mouse click menu .\", \"aspect_start\": 45, \"aspect_end\": 52, \"evidence_start\": 0, \"evidence_end\": 209}, {\"aspect\": \"tutorial\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I like the Mini Mac it was easy to setup and install , but I am learning as I go and could use a tutorial to learn how to use some of the features I was use to on the PC especially the right mouse click menu .\", \"aspect_start\": 97, \"aspect_end\": 105, \"evidence_start\": 0, \"evidence_end\": 209}, {\"aspect\": \"features\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I like the Mini Mac it was easy to setup and install , but I am learning as I go and could use a tutorial to learn how to use some of the features I was use to on the PC especially the right mouse click menu .\", \"aspect_start\": 138, \"aspect_end\": 146, \"evidence_start\": 0, \"evidence_end\": 209}, {\"aspect\": \"right mouse click menu\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I like the Mini Mac it was easy to setup and install , but I am learning as I go and could use a tutorial to learn how to use some of the features I was use to on the PC especially the right mouse click menu .\", \"aspect_start\": 185, \"aspect_end\": 207, \"evidence_start\": 0, \"evidence_end\": 209}], \"review_id\": \"gold_rev_0023\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"big and confusing air mouse is not there but an option is there but its bit laggy picture 4/5 audio 4.5/5 connectivity 5/5 remote 3/5 battery provided so a good thing mira cast 3/5 plus as bonus Google play store and aptoide both working seemlessly update after a month use its a great tv in limited budget section go for it without second thought.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"big and confusing air mouse is not there but an option is there but its bit laggy picture 4/5 audio 4.5/5 connectivity 5/5 remote 3/5 battery provided so a good thing mira cast 3/5 plus as bonus Google play store and aptoide both working seemlessly update after a month use its a great tv in limited budget section go for it without second thought.\", \"aspect_start\": 82, \"aspect_end\": 89, \"evidence_start\": 0, \"evidence_end\": 348}, {\"aspect\": \"audio\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"big and confusing air mouse is not there but an option is there but its bit laggy picture 4/5 audio 4.5/5 connectivity 5/5 remote 3/5 battery provided so a good thing mira cast 3/5 plus as bonus Google play store and aptoide both working seemlessly update after a month use its a great tv in limited budget section go for it without second thought.\", \"aspect_start\": 94, \"aspect_end\": 99, \"evidence_start\": 0, \"evidence_end\": 348}, {\"aspect\": \"connectivity\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"big and confusing air mouse is not there but an option is there but its bit laggy picture 4/5 audio 4.5/5 connectivity 5/5 remote 3/5 battery provided so a good thing mira cast 3/5 plus as bonus Google play store and aptoide both working seemlessly update after a month use its a great tv in limited budget section go for it without second thought.\", \"aspect_start\": 106, \"aspect_end\": 118, \"evidence_start\": 0, \"evidence_end\": 348}, {\"aspect\": \"remote\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"big and confusing air mouse is not there but an option is there but its bit laggy picture 4/5 audio 4.5/5 connectivity 5/5 remote 3/5 battery provided so a good thing mira cast 3/5 plus as bonus Google play store and aptoide both working seemlessly update after a month use its a great tv in limited budget section go for it without second thought.\", \"aspect_start\": 123, \"aspect_end\": 129, \"evidence_start\": 0, \"evidence_end\": 348}, {\"aspect\": \"mira cast\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"big and confusing air mouse is not there but an option is there but its bit laggy picture 4/5 audio 4.5/5 connectivity 5/5 remote 3/5 battery provided so a good thing mira cast 3/5 plus as bonus Google play store and aptoide both working seemlessly update after a month use its a great tv in limited budget section go for it without second thought.\", \"aspect_start\": 167, \"aspect_end\": 176, \"evidence_start\": 0, \"evidence_end\": 348}], \"review_id\": \"gold_rev_0024\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"A veryimportant feature is Firewire 800 which in my experience works better then USB3 -LRB- in PC enabled with USB3 -RRB- I was not originally sold on the MAC OS I felt it was inferior in many ways To Windows 7 .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"USB3\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"A veryimportant feature is Firewire 800 which in my experience works better then USB3 -LRB- in PC enabled with USB3 -RRB- I was not originally sold on the MAC OS I felt it was inferior in many ways To Windows 7 .\", \"aspect_start\": 81, \"aspect_end\": 85, \"evidence_start\": 0, \"evidence_end\": 212}, {\"aspect\": \"USB3\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"A veryimportant feature is Firewire 800 which in my experience works better then USB3 -LRB- in PC enabled with USB3 -RRB- I was not originally sold on the MAC OS I felt it was inferior in many ways To Windows 7 .\", \"aspect_start\": 81, \"aspect_end\": 85, \"evidence_start\": 0, \"evidence_end\": 212}, {\"aspect\": \"MAC OS\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"A veryimportant feature is Firewire 800 which in my experience works better then USB3 -LRB- in PC enabled with USB3 -RRB- I was not originally sold on the MAC OS I felt it was inferior in many ways To Windows 7 .\", \"aspect_start\": 155, \"aspect_end\": 161, \"evidence_start\": 0, \"evidence_end\": 212}, {\"aspect\": \"Windows 7\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"A veryimportant feature is Firewire 800 which in my experience works better then USB3 -LRB- in PC enabled with USB3 -RRB- I was not originally sold on the MAC OS I felt it was inferior in many ways To Windows 7 .\", \"aspect_start\": 201, \"aspect_end\": 210, \"evidence_start\": 0, \"evidence_end\": 212}], \"review_id\": \"gold_rev_0025\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I had to buy a wireless mouse to go with it , as I am old school and hate the pad , but knew that before I bought it , now it works great , need to get adjusted to the key board , as I am used to a bigger one and pounding .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"wireless mouse\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I had to buy a wireless mouse to go with it , as I am old school and hate the pad , but knew that before I bought it , now it works great , need to get adjusted to the key board , as I am used to a bigger one and pounding .\", \"aspect_start\": 15, \"aspect_end\": 29, \"evidence_start\": 0, \"evidence_end\": 223}, {\"aspect\": \"pad\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"I had to buy a wireless mouse to go with it , as I am old school and hate the pad , but knew that before I bought it , now it works great , need to get adjusted to the key board , as I am used to a bigger one and pounding .\", \"aspect_start\": 78, \"aspect_end\": 81, \"evidence_start\": 0, \"evidence_end\": 223}, {\"aspect\": \"works\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I had to buy a wireless mouse to go with it , as I am old school and hate the pad , but knew that before I bought it , now it works great , need to get adjusted to the key board , as I am used to a bigger one and pounding .\", \"aspect_start\": 126, \"aspect_end\": 131, \"evidence_start\": 0, \"evidence_end\": 223}, {\"aspect\": \"key board\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I had to buy a wireless mouse to go with it , as I am old school and hate the pad , but knew that before I bought it , now it works great , need to get adjusted to the key board , as I am used to a bigger one and pounding .\", \"aspect_start\": 168, \"aspect_end\": 177, \"evidence_start\": 0, \"evidence_end\": 223}], \"review_id\": \"gold_rev_0026\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"with in purchase period of 10-11 months there were spots visible at the back side of the screen and the executive who came from the service center confirmed that the display had manufacturing defect and will be replaced however post lots of follow ups, complaints and escalation the display panel was not replaced and they have installed back the same panel after cleaning it.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"screen\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"with in purchase period of 10-11 months there were spots visible at the back side of the screen and the executive who came from the service center confirmed that the display had manufacturing defect and will be replaced however post lots of follow ups, complaints and escalation the display panel was not replaced and they have installed back the same panel after cleaning it.\", \"aspect_start\": 89, \"aspect_end\": 95, \"evidence_start\": 0, \"evidence_end\": 376}, {\"aspect\": \"display\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"with in purchase period of 10-11 months there were spots visible at the back side of the screen and the executive who came from the service center confirmed that the display had manufacturing defect and will be replaced however post lots of follow ups, complaints and escalation the display panel was not replaced and they have installed back the same panel after cleaning it.\", \"aspect_start\": 166, \"aspect_end\": 173, \"evidence_start\": 0, \"evidence_end\": 376}, {\"aspect\": \"display panel\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"with in purchase period of 10-11 months there were spots visible at the back side of the screen and the executive who came from the service center confirmed that the display had manufacturing defect and will be replaced however post lots of follow ups, complaints and escalation the display panel was not replaced and they have installed back the same panel after cleaning it.\", \"aspect_start\": 283, \"aspect_end\": 296, \"evidence_start\": 0, \"evidence_end\": 376}, {\"aspect\": \"panel\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"with in purchase period of 10-11 months there were spots visible at the back side of the screen and the executive who came from the service center confirmed that the display had manufacturing defect and will be replaced however post lots of follow ups, complaints and escalation the display panel was not replaced and they have installed back the same panel after cleaning it.\", \"aspect_start\": 291, \"aspect_end\": 296, \"evidence_start\": 0, \"evidence_end\": 376}], \"review_id\": \"gold_rev_0027\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"color contrast is good and fabric is comfortable yet looks great and feels premium .\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"color contrast\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"color contrast is good and fabric is comfortable yet looks great and feels premium .\", \"aspect_start\": 0, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 84}, {\"aspect\": \"fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"color contrast is good and fabric is comfortable yet looks great and feels premium .\", \"aspect_start\": 27, \"aspect_end\": 33, \"evidence_start\": 0, \"evidence_end\": 84}, {\"aspect\": \"looks\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"color contrast is good and fabric is comfortable yet looks great and feels premium .\", \"aspect_start\": 53, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 84}, {\"aspect\": \"feels\", \"category\": \"COMFORT\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"color contrast is good and fabric is comfortable yet looks great and feels premium .\", \"aspect_start\": 69, \"aspect_end\": 74, \"evidence_start\": 0, \"evidence_end\": 84}], \"review_id\": \"gold_rev_0028\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Additionally , there is barely a ventilation system in the computer , and even the simple activity of watching videos let alone playing steam games causes the laptop to get very very hot , and in fact impossible to keep on lap .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"ventilation system\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Additionally , there is barely a ventilation system in the computer , and even the simple activity of watching videos let alone playing steam games causes the laptop to get very very hot , and in fact impossible to keep on lap .\", \"aspect_start\": 33, \"aspect_end\": 51, \"evidence_start\": 0, \"evidence_end\": 228}, {\"aspect\": \"watching videos\", \"category\": \"SOFTWARE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"Additionally , there is barely a ventilation system in the computer , and even the simple activity of watching videos let alone playing steam games causes the laptop to get very very hot , and in fact impossible to keep on lap .\", \"aspect_start\": 102, \"aspect_end\": 117, \"evidence_start\": 0, \"evidence_end\": 228}, {\"aspect\": \"playing steam games\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"Additionally , there is barely a ventilation system in the computer , and even the simple activity of watching videos let alone playing steam games causes the laptop to get very very hot , and in fact impossible to keep on lap .\", \"aspect_start\": 128, \"aspect_end\": 147, \"evidence_start\": 0, \"evidence_end\": 228}], \"review_id\": \"gold_rev_0029\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I was a little worry at first because I do n't have a lot of experience with os.x and windows has always been second nature to me after many years of using windows .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"os.x\", \"category\": \"SOFTWARE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I was a little worry at first because I do n't have a lot of experience with os.x and windows has always been second nature to me after many years of using windows .\", \"aspect_start\": 77, \"aspect_end\": 81, \"evidence_start\": 0, \"evidence_end\": 165}, {\"aspect\": \"windows\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I was a little worry at first because I do n't have a lot of experience with os.x and windows has always been second nature to me after many years of using windows .\", \"aspect_start\": 86, \"aspect_end\": 93, \"evidence_start\": 0, \"evidence_end\": 165}, {\"aspect\": \"windows\", \"category\": \"SOFTWARE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I was a little worry at first because I do n't have a lot of experience with os.x and windows has always been second nature to me after many years of using windows .\", \"aspect_start\": 86, \"aspect_end\": 93, \"evidence_start\": 0, \"evidence_end\": 165}], \"review_id\": \"gold_rev_0030\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"It also comes with 4G of RAM but if you 're like me you want to max that out so I immediately put 8G of RAM in her and I 've never used a computer that performs better .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"4G of RAM\", \"category\": \"PERFORMANCE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"It also comes with 4G of RAM but if you 're like me you want to max that out so I immediately put 8G of RAM in her and I 've never used a computer that performs better .\", \"aspect_start\": 19, \"aspect_end\": 28, \"evidence_start\": 0, \"evidence_end\": 169}, {\"aspect\": \"8G of RAM\", \"category\": \"PERFORMANCE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"It also comes with 4G of RAM but if you 're like me you want to max that out so I immediately put 8G of RAM in her and I 've never used a computer that performs better .\", \"aspect_start\": 98, \"aspect_end\": 107, \"evidence_start\": 0, \"evidence_end\": 169}, {\"aspect\": \"performs\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"It also comes with 4G of RAM but if you 're like me you want to max that out so I immediately put 8G of RAM in her and I 've never used a computer that performs better .\", \"aspect_start\": 152, \"aspect_end\": 160, \"evidence_start\": 0, \"evidence_end\": 169}], \"review_id\": \"gold_rev_0031\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"One more thing , this mac does NOT come with restore disks and I am not sure if you can make them direct from the mac like you can with newer PC 's , also the charging cables are made of the same cheap material as the iPhone/iPod touch cables .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"restore disks\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"One more thing , this mac does NOT come with restore disks and I am not sure if you can make them direct from the mac like you can with newer PC 's , also the charging cables are made of the same cheap material as the iPhone/iPod touch cables .\", \"aspect_start\": 45, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 244}, {\"aspect\": \"charging cables\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"One more thing , this mac does NOT come with restore disks and I am not sure if you can make them direct from the mac like you can with newer PC 's , also the charging cables are made of the same cheap material as the iPhone/iPod touch cables .\", \"aspect_start\": 159, \"aspect_end\": 174, \"evidence_start\": 0, \"evidence_end\": 244}, {\"aspect\": \"material\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"One more thing , this mac does NOT come with restore disks and I am not sure if you can make them direct from the mac like you can with newer PC 's , also the charging cables are made of the same cheap material as the iPhone/iPod touch cables .\", \"aspect_start\": 202, \"aspect_end\": 210, \"evidence_start\": 0, \"evidence_end\": 244}], \"review_id\": \"gold_rev_0032\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Other than not being a fan of click pads -LRB- industry standard these days -RRB- and the lousy internal speakers , it 's hard for me to find things about this notebook I do n't like , especially considering the $ 350 price tag .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"internal speakers\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Other than not being a fan of click pads -LRB- industry standard these days -RRB- and the lousy internal speakers , it 's hard for me to find things about this notebook I do n't like , especially considering the $ 350 price tag .\", \"aspect_start\": 96, \"aspect_end\": 113, \"evidence_start\": 0, \"evidence_end\": 229}, {\"aspect\": \"price tag\", \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Other than not being a fan of click pads -LRB- industry standard these days -RRB- and the lousy internal speakers , it 's hard for me to find things about this notebook I do n't like , especially considering the $ 350 price tag .\", \"aspect_start\": 218, \"aspect_end\": 227, \"evidence_start\": 0, \"evidence_end\": 229}, {\"aspect\": \"click pads\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Other than not being a fan of click pads -LRB- industry standard these days -RRB- and the lousy internal speakers , it 's hard for me to find things about this notebook I do n't like , especially considering the $ 350 price tag .\", \"aspect_start\": 30, \"aspect_end\": 40, \"evidence_start\": 0, \"evidence_end\": 229}], \"review_id\": \"gold_rev_0033\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The SD card reader is slightly recessed and upside down -LRB- the nail slot on the card can not be accessed -RRB- , if this was a self ejecting slot this would not be an issue , but its not .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"SD card reader\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The SD card reader is slightly recessed and upside down -LRB- the nail slot on the card can not be accessed -RRB- , if this was a self ejecting slot this would not be an issue , but its not .\", \"aspect_start\": 4, \"aspect_end\": 18, \"evidence_start\": 0, \"evidence_end\": 191}, {\"aspect\": \"nail slot on the card\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The SD card reader is slightly recessed and upside down -LRB- the nail slot on the card can not be accessed -RRB- , if this was a self ejecting slot this would not be an issue , but its not .\", \"aspect_start\": 66, \"aspect_end\": 87, \"evidence_start\": 0, \"evidence_end\": 191}, {\"aspect\": \"slot\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The SD card reader is slightly recessed and upside down -LRB- the nail slot on the card can not be accessed -RRB- , if this was a self ejecting slot this would not be an issue , but its not .\", \"aspect_start\": 71, \"aspect_end\": 75, \"evidence_start\": 0, \"evidence_end\": 191}], \"review_id\": \"gold_rev_0034\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The power plug has to be connected to the power adaptor to charge the battery but wo n't stay connected .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"power plug\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The power plug has to be connected to the power adaptor to charge the battery but wo n't stay connected .\", \"aspect_start\": 4, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 105}, {\"aspect\": \"power adaptor\", \"category\": \"BATTERY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"The power plug has to be connected to the power adaptor to charge the battery but wo n't stay connected .\", \"aspect_start\": 42, \"aspect_end\": 55, \"evidence_start\": 0, \"evidence_end\": 105}, {\"aspect\": \"battery\", \"category\": \"BATTERY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"The power plug has to be connected to the power adaptor to charge the battery but wo n't stay connected .\", \"aspect_start\": 70, \"aspect_end\": 77, \"evidence_start\": 0, \"evidence_end\": 105}], \"review_id\": \"gold_rev_0035\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"this Mac Mini does not have a built-in mic , and it would seem that its Mac OS 10.9 does not handle external microphones properly .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"built-in mic\", \"category\": \"SOUND\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"this Mac Mini does not have a built-in mic , and it would seem that its Mac OS 10.9 does not handle external microphones properly .\", \"aspect_start\": 30, \"aspect_end\": 42, \"evidence_start\": 0, \"evidence_end\": 131}, {\"aspect\": \"Mac OS 10.9\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"this Mac Mini does not have a built-in mic , and it would seem that its Mac OS 10.9 does not handle external microphones properly .\", \"aspect_start\": 72, \"aspect_end\": 83, \"evidence_start\": 0, \"evidence_end\": 131}, {\"aspect\": \"external microphones\", \"category\": \"SOUND\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"this Mac Mini does not have a built-in mic , and it would seem that its Mac OS 10.9 does not handle external microphones properly .\", \"aspect_start\": 100, \"aspect_end\": 120, \"evidence_start\": 0, \"evidence_end\": 131}], \"review_id\": \"gold_rev_0036\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"compare to vu visuals also not good sound quality is better video quality is too bad.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"compare to vu visuals also not good sound quality is better video quality is too bad.\", \"aspect_start\": 36, \"aspect_end\": 49, \"evidence_start\": 0, \"evidence_end\": 85}, {\"aspect\": \"visuals\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"compare to vu visuals also not good sound quality is better video quality is too bad.\", \"aspect_start\": 14, \"aspect_end\": 21, \"evidence_start\": 0, \"evidence_end\": 85}, {\"aspect\": \"video quality\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"compare to vu visuals also not good sound quality is better video quality is too bad.\", \"aspect_start\": 60, \"aspect_end\": 73, \"evidence_start\": 0, \"evidence_end\": 85}], \"review_id\": \"gold_rev_0037\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"everything else is compromisable with little slower micro controller and not so good menu interface with few customisation and in my opinion you probably wont require many customisation on this tv with naturally good picture and sound.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"micro controller\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"everything else is compromisable with little slower micro controller and not so good menu interface with few customisation and in my opinion you probably wont require many customisation on this tv with naturally good picture and sound.\", \"aspect_start\": 52, \"aspect_end\": 68, \"evidence_start\": 0, \"evidence_end\": 235}, {\"aspect\": \"menu interface\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"everything else is compromisable with little slower micro controller and not so good menu interface with few customisation and in my opinion you probably wont require many customisation on this tv with naturally good picture and sound.\", \"aspect_start\": 85, \"aspect_end\": 99, \"evidence_start\": 0, \"evidence_end\": 235}, {\"aspect\": \"picture\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"everything else is compromisable with little slower micro controller and not so good menu interface with few customisation and in my opinion you probably wont require many customisation on this tv with naturally good picture and sound.\", \"aspect_start\": 217, \"aspect_end\": 224, \"evidence_start\": 0, \"evidence_end\": 235}], \"review_id\": \"gold_rev_0038\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"good product good clarity i am satisfied with the quality of the product 2 hdmi, 1usb but the screen resolution is really good and same day installation go for it thank you Flipkart\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"clarity\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"good product good clarity i am satisfied with the quality of the product 2 hdmi, 1usb but the screen resolution is really good and same day installation go for it thank you Flipkart\", \"aspect_start\": 18, \"aspect_end\": 25, \"evidence_start\": 0, \"evidence_end\": 181}, {\"aspect\": \"screen resolution\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"good product good clarity i am satisfied with the quality of the product 2 hdmi, 1usb but the screen resolution is really good and same day installation go for it thank you Flipkart\", \"aspect_start\": 94, \"aspect_end\": 111, \"evidence_start\": 0, \"evidence_end\": 181}, {\"aspect\": \"installation\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"good product good clarity i am satisfied with the quality of the product 2 hdmi, 1usb but the screen resolution is really good and same day installation go for it thank you Flipkart\", \"aspect_start\": 140, \"aspect_end\": 152, \"evidence_start\": 0, \"evidence_end\": 181}], \"review_id\": \"gold_rev_0039\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"hi good quality but display small line showing customer service not responsible.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"hi good quality but display small line showing customer service not responsible.\", \"aspect_start\": 8, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 80}, {\"aspect\": \"display\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"hi good quality but display small line showing customer service not responsible.\", \"aspect_start\": 20, \"aspect_end\": 27, \"evidence_start\": 0, \"evidence_end\": 80}, {\"aspect\": \"customer service\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"hi good quality but display small line showing customer service not responsible.\", \"aspect_start\": 47, \"aspect_end\": 63, \"evidence_start\": 0, \"evidence_end\": 80}], \"review_id\": \"gold_rev_0040\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture clarity is too good and sound is also but in sound bass is not being good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture clarity\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture clarity is too good and sound is also but in sound bass is not being good.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 82}, {\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture clarity is too good and sound is also but in sound bass is not being good.\", \"aspect_start\": 32, \"aspect_end\": 37, \"evidence_start\": 0, \"evidence_end\": 82}, {\"aspect\": \"bass\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture clarity is too good and sound is also but in sound bass is not being good.\", \"aspect_start\": 59, \"aspect_end\": 63, \"evidence_start\": 0, \"evidence_end\": 82}], \"review_id\": \"gold_rev_0041\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"tv was received on 6th sept 2017, installed on 08 th sept 2017 however demo has not been given till date even after calling the demo personal over telephone 4-5 times.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"demo personal\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"tv was received on 6th sept 2017, installed on 08 th sept 2017 however demo has not been given till date even after calling the demo personal over telephone 4-5 times.\", \"aspect_start\": 128, \"aspect_end\": 141, \"evidence_start\": 0, \"evidence_end\": 167}, {\"aspect\": \"installed\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"tv was received on 6th sept 2017, installed on 08 th sept 2017 however demo has not been given till date even after calling the demo personal over telephone 4-5 times.\", \"aspect_start\": 34, \"aspect_end\": 43, \"evidence_start\": 0, \"evidence_end\": 167}, {\"aspect\": \"demo\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"tv was received on 6th sept 2017, installed on 08 th sept 2017 however demo has not been given till date even after calling the demo personal over telephone 4-5 times.\", \"aspect_start\": 71, \"aspect_end\": 75, \"evidence_start\": 0, \"evidence_end\": 167}], \"review_id\": \"gold_rev_0042\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"value for money for this product is great and picture quality great but should be have refresh rate 100hz always in big size model neither sometime flickering problem have shown playing in usb video, and one more things that should support more audio and video codec that is mkv or h265 codec in this type of big size that is 32 inch or more.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"sarcasm\", \"quads\": [{\"aspect\": \"refresh rate\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"value for money for this product is great and picture quality great but should be have refresh rate 100hz always in big size model neither sometime flickering problem have shown playing in usb video, and one more things that should support more audio and video codec that is mkv or h265 codec in this type of big size that is 32 inch or more.\", \"aspect_start\": 87, \"aspect_end\": 99, \"evidence_start\": 0, \"evidence_end\": 342}, {\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"value for money for this product is great and picture quality great but should be have refresh rate 100hz always in big size model neither sometime flickering problem have shown playing in usb video, and one more things that should support more audio and video codec that is mkv or h265 codec in this type of big size that is 32 inch or more.\", \"aspect_start\": 46, \"aspect_end\": 61, \"evidence_start\": 0, \"evidence_end\": 342}, {\"aspect\": \"audio and video codec\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"value for money for this product is great and picture quality great but should be have refresh rate 100hz always in big size model neither sometime flickering problem have shown playing in usb video, and one more things that should support more audio and video codec that is mkv or h265 codec in this type of big size that is 32 inch or more.\", \"aspect_start\": 245, \"aspect_end\": 266, \"evidence_start\": 0, \"evidence_end\": 342}], \"review_id\": \"gold_rev_0043\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"video display is very poor and color reproduction is very bad, not able to watch sd or hd channels clearly as compare to any basic resolution tvs.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"video display\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"video display is very poor and color reproduction is very bad, not able to watch sd or hd channels clearly as compare to any basic resolution tvs.\", \"aspect_start\": 0, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 146}, {\"aspect\": \"color reproduction\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"video display is very poor and color reproduction is very bad, not able to watch sd or hd channels clearly as compare to any basic resolution tvs.\", \"aspect_start\": 31, \"aspect_end\": 49, \"evidence_start\": 0, \"evidence_end\": 146}, {\"aspect\": \"hd channels\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"video display is very poor and color reproduction is very bad, not able to watch sd or hd channels clearly as compare to any basic resolution tvs.\", \"aspect_start\": 87, \"aspect_end\": 98, \"evidence_start\": 0, \"evidence_end\": 146}], \"review_id\": \"gold_rev_0044\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"cheap quality product logo looks really bad cloth is good but colour is also different .!\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"cloth\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"cheap quality product logo looks really bad cloth is good but colour is also different .!\", \"aspect_start\": 44, \"aspect_end\": 49, \"evidence_start\": 0, \"evidence_end\": 89}, {\"aspect\": \"product logo\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"cheap quality product logo looks really bad cloth is good but colour is also different .!\", \"aspect_start\": 14, \"aspect_end\": 26, \"evidence_start\": 0, \"evidence_end\": 89}, {\"aspect\": \"colour\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"cheap quality product logo looks really bad cloth is good but colour is also different .!\", \"aspect_start\": 62, \"aspect_end\": 68, \"evidence_start\": 0, \"evidence_end\": 89}], \"review_id\": \"gold_rev_0045\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"fabric quality and finishing not so good for the price stitching also not clean, metronaut brand quality is better than this.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"fabric quality\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"fabric quality and finishing not so good for the price stitching also not clean, metronaut brand quality is better than this.\", \"aspect_start\": 0, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 125}, {\"aspect\": \"finishing\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"fabric quality and finishing not so good for the price stitching also not clean, metronaut brand quality is better than this.\", \"aspect_start\": 19, \"aspect_end\": 28, \"evidence_start\": 0, \"evidence_end\": 125}, {\"aspect\": \"stitching\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"fabric quality and finishing not so good for the price stitching also not clean, metronaut brand quality is better than this.\", \"aspect_start\": 55, \"aspect_end\": 64, \"evidence_start\": 0, \"evidence_end\": 125}], \"review_id\": \"gold_rev_0046\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"good pattern but shirt material and print quality not as good as expected.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"pattern\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"good pattern but shirt material and print quality not as good as expected.\", \"aspect_start\": 5, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 74}, {\"aspect\": \"shirt material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"good pattern but shirt material and print quality not as good as expected.\", \"aspect_start\": 17, \"aspect_end\": 31, \"evidence_start\": 0, \"evidence_end\": 74}, {\"aspect\": \"print quality\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"good pattern but shirt material and print quality not as good as expected.\", \"aspect_start\": 36, \"aspect_end\": 49, \"evidence_start\": 0, \"evidence_end\": 74}], \"review_id\": \"gold_rev_0047\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"nice material and good fitting , finishing is not that good what it should be.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"nice material and good fitting , finishing is not that good what it should be.\", \"aspect_start\": 5, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 78}, {\"aspect\": \"fitting\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"nice material and good fitting , finishing is not that good what it should be.\", \"aspect_start\": 23, \"aspect_end\": 30, \"evidence_start\": 0, \"evidence_end\": 78}, {\"aspect\": \"finishing\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"nice material and good fitting , finishing is not that good what it should be.\", \"aspect_start\": 33, \"aspect_end\": 42, \"evidence_start\": 0, \"evidence_end\": 78}], \"review_id\": \"gold_rev_0048\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"not completely satisfied, tshirt design and fitting is good but quality is not up to the mark, little thickness in fabric should be given\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"tshirt design\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"not completely satisfied, tshirt design and fitting is good but quality is not up to the mark, little thickness in fabric should be given\", \"aspect_start\": 26, \"aspect_end\": 39, \"evidence_start\": 0, \"evidence_end\": 137}, {\"aspect\": \"fitting\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"not completely satisfied, tshirt design and fitting is good but quality is not up to the mark, little thickness in fabric should be given\", \"aspect_start\": 44, \"aspect_end\": 51, \"evidence_start\": 0, \"evidence_end\": 137}, {\"aspect\": \"fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"not completely satisfied, tshirt design and fitting is good but quality is not up to the mark, little thickness in fabric should be given\", \"aspect_start\": 115, \"aspect_end\": 121, \"evidence_start\": 0, \"evidence_end\": 137}], \"review_id\": \"gold_rev_0049\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"only design best but material quality and fitting third class.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"design\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"only design best but material quality and fitting third class.\", \"aspect_start\": 5, \"aspect_end\": 11, \"evidence_start\": 0, \"evidence_end\": 62}, {\"aspect\": \"material quality\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"only design best but material quality and fitting third class.\", \"aspect_start\": 21, \"aspect_end\": 37, \"evidence_start\": 0, \"evidence_end\": 62}, {\"aspect\": \"fitting\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"only design best but material quality and fitting third class.\", \"aspect_start\": 42, \"aspect_end\": 49, \"evidence_start\": 0, \"evidence_end\": 62}], \"review_id\": \"gold_rev_0050\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"very nice design and color but poor quality color fades and border line stretched with 3 to 4 washes.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"design\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"very nice design and color but poor quality color fades and border line stretched with 3 to 4 washes.\", \"aspect_start\": 10, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 101}, {\"aspect\": \"color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"very nice design and color but poor quality color fades and border line stretched with 3 to 4 washes.\", \"aspect_start\": 21, \"aspect_end\": 26, \"evidence_start\": 0, \"evidence_end\": 101}, {\"aspect\": \"quality\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"very nice design and color but poor quality color fades and border line stretched with 3 to 4 washes.\", \"aspect_start\": 36, \"aspect_end\": 43, \"evidence_start\": 0, \"evidence_end\": 101}], \"review_id\": \"gold_rev_0051\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"A lot of features and shortcuts on the MBP that I was never exposed to on a normal PC .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"features\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"A lot of features and shortcuts on the MBP that I was never exposed to on a normal PC .\", \"aspect_start\": 9, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 87}, {\"aspect\": \"shortcuts\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"A lot of features and shortcuts on the MBP that I was never exposed to on a normal PC .\", \"aspect_start\": 22, \"aspect_end\": 31, \"evidence_start\": 0, \"evidence_end\": 87}], \"review_id\": \"gold_rev_0052\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"As far as user type - I dabble in everything from games -LRB- WoW -RRB- to Photoshop , but nothing professionally .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"games\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"As far as user type - I dabble in everything from games -LRB- WoW -RRB- to Photoshop , but nothing professionally .\", \"aspect_start\": 50, \"aspect_end\": 55, \"evidence_start\": 0, \"evidence_end\": 115}, {\"aspect\": \"Photoshop\", \"category\": \"SOFTWARE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"As far as user type - I dabble in everything from games -LRB- WoW -RRB- to Photoshop , but nothing professionally .\", \"aspect_start\": 75, \"aspect_end\": 84, \"evidence_start\": 0, \"evidence_end\": 115}], \"review_id\": \"gold_rev_0053\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"But with this laptop , the bass is very weak and the sound comes out sounding tinny .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"bass\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"But with this laptop , the bass is very weak and the sound comes out sounding tinny .\", \"aspect_start\": 27, \"aspect_end\": 31, \"evidence_start\": 0, \"evidence_end\": 85}, {\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"But with this laptop , the bass is very weak and the sound comes out sounding tinny .\", \"aspect_start\": 53, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 85}], \"review_id\": \"gold_rev_0054\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Did not enjoy the new Windows 8 and touchscreen functions .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Windows 8\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Did not enjoy the new Windows 8 and touchscreen functions .\", \"aspect_start\": 22, \"aspect_end\": 31, \"evidence_start\": 0, \"evidence_end\": 59}, {\"aspect\": \"touchscreen functions\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Did not enjoy the new Windows 8 and touchscreen functions .\", \"aspect_start\": 36, \"aspect_end\": 57, \"evidence_start\": 0, \"evidence_end\": 59}], \"review_id\": \"gold_rev_0055\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Here are the things that made me confident with my purchase : Build Quality - Seriously , you ca n't beat a unibody construction .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Build Quality\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Here are the things that made me confident with my purchase : Build Quality - Seriously , you ca n't beat a unibody construction .\", \"aspect_start\": 62, \"aspect_end\": 75, \"evidence_start\": 0, \"evidence_end\": 130}, {\"aspect\": \"unibody construction\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Here are the things that made me confident with my purchase : Build Quality - Seriously , you ca n't beat a unibody construction .\", \"aspect_start\": 108, \"aspect_end\": 128, \"evidence_start\": 0, \"evidence_end\": 130}], \"review_id\": \"gold_rev_0056\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"However , it did not have any scratches , ZERO battery cycle count -LRB- pretty surprised -RRB- , and all the hardware seemed to be working perfectly .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"battery cycle count\", \"category\": \"BATTERY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"However , it did not have any scratches , ZERO battery cycle count -LRB- pretty surprised -RRB- , and all the hardware seemed to be working perfectly .\", \"aspect_start\": 47, \"aspect_end\": 66, \"evidence_start\": 0, \"evidence_end\": 151}, {\"aspect\": \"hardware\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"However , it did not have any scratches , ZERO battery cycle count -LRB- pretty surprised -RRB- , and all the hardware seemed to be working perfectly .\", \"aspect_start\": 110, \"aspect_end\": 118, \"evidence_start\": 0, \"evidence_end\": 151}], \"review_id\": \"gold_rev_0057\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"However , the experience was great since the OS does not become unstable and the application will simply shutdown and reopen .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"OS\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"However , the experience was great since the OS does not become unstable and the application will simply shutdown and reopen .\", \"aspect_start\": 45, \"aspect_end\": 47, \"evidence_start\": 0, \"evidence_end\": 126}, {\"aspect\": \"application\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"However , the experience was great since the OS does not become unstable and the application will simply shutdown and reopen .\", \"aspect_start\": 81, \"aspect_end\": 92, \"evidence_start\": 0, \"evidence_end\": 126}], \"review_id\": \"gold_rev_0058\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I ca n't believe how quiet the hard drive is and how quick this thing boots up .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"hard drive\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I ca n't believe how quiet the hard drive is and how quick this thing boots up .\", \"aspect_start\": 31, \"aspect_end\": 41, \"evidence_start\": 0, \"evidence_end\": 80}, {\"aspect\": \"boots up\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I ca n't believe how quiet the hard drive is and how quick this thing boots up .\", \"aspect_start\": 70, \"aspect_end\": 78, \"evidence_start\": 0, \"evidence_end\": 80}], \"review_id\": \"gold_rev_0059\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I know some people complained about HDMI issues but they released a firmware patch to address that issue .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"HDMI\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I know some people complained about HDMI issues but they released a firmware patch to address that issue .\", \"aspect_start\": 36, \"aspect_end\": 40, \"evidence_start\": 0, \"evidence_end\": 106}, {\"aspect\": \"firmware patch\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I know some people complained about HDMI issues but they released a firmware patch to address that issue .\", \"aspect_start\": 68, \"aspect_end\": 82, \"evidence_start\": 0, \"evidence_end\": 106}], \"review_id\": \"gold_rev_0060\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I like coming back to Mac OS but this laptop is lacking in speaker quality compared to my $ 400 old HP laptop .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Mac OS\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I like coming back to Mac OS but this laptop is lacking in speaker quality compared to my $ 400 old HP laptop .\", \"aspect_start\": 22, \"aspect_end\": 28, \"evidence_start\": 0, \"evidence_end\": 111}, {\"aspect\": \"speaker quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"I like coming back to Mac OS but this laptop is lacking in speaker quality compared to my $ 400 old HP laptop .\", \"aspect_start\": 59, \"aspect_end\": 74, \"evidence_start\": 0, \"evidence_end\": 111}], \"review_id\": \"gold_rev_0061\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I like the design and its features but there are somethings I think needs to be improved .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"design\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I like the design and its features but there are somethings I think needs to be improved .\", \"aspect_start\": 11, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 90}, {\"aspect\": \"features\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I like the design and its features but there are somethings I think needs to be improved .\", \"aspect_start\": 26, \"aspect_end\": 34, \"evidence_start\": 0, \"evidence_end\": 90}], \"review_id\": \"gold_rev_0062\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I opted for the SquareTrade 3-Year Computer Accidental Protection Warranty -LRB- $ 1500-2000 -RRB- which also support `` accidents '' like drops and spills that are NOT covered by AppleCare .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"SquareTrade 3-Year Computer Accidental Protection Warranty\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I opted for the SquareTrade 3-Year Computer Accidental Protection Warranty -LRB- $ 1500-2000 -RRB- which also support `` accidents '' like drops and spills that are NOT covered by AppleCare .\", \"aspect_start\": 16, \"aspect_end\": 74, \"evidence_start\": 0, \"evidence_end\": 191}, {\"aspect\": \"AppleCare\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"I opted for the SquareTrade 3-Year Computer Accidental Protection Warranty -LRB- $ 1500-2000 -RRB- which also support `` accidents '' like drops and spills that are NOT covered by AppleCare .\", \"aspect_start\": 180, \"aspect_end\": 189, \"evidence_start\": 0, \"evidence_end\": 191}], \"review_id\": \"gold_rev_0063\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"It is made of such solid construction and since I have never had a Mac using my iPhone helped me get used to the system a bit .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"construction\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"It is made of such solid construction and since I have never had a Mac using my iPhone helped me get used to the system a bit .\", \"aspect_start\": 25, \"aspect_end\": 37, \"evidence_start\": 0, \"evidence_end\": 127}, {\"aspect\": \"system\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"It is made of such solid construction and since I have never had a Mac using my iPhone helped me get used to the system a bit .\", \"aspect_start\": 113, \"aspect_end\": 119, \"evidence_start\": 0, \"evidence_end\": 127}], \"review_id\": \"gold_rev_0064\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Just beware no DVD slot so when I went to install software I had on CD I could n't .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"install software\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Just beware no DVD slot so when I went to install software I had on CD I could n't .\", \"aspect_start\": 42, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 84}, {\"aspect\": \"DVD slot\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"Just beware no DVD slot so when I went to install software I had on CD I could n't .\", \"aspect_start\": 15, \"aspect_end\": 23, \"evidence_start\": 0, \"evidence_end\": 84}], \"review_id\": \"gold_rev_0065\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"My laptop with Windows 7 crashed and I did not want Windows 8 .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Windows 7\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"My laptop with Windows 7 crashed and I did not want Windows 8 .\", \"aspect_start\": 15, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 63}, {\"aspect\": \"Windows 8\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"My laptop with Windows 7 crashed and I did not want Windows 8 .\", \"aspect_start\": 52, \"aspect_end\": 61, \"evidence_start\": 0, \"evidence_end\": 63}], \"review_id\": \"gold_rev_0066\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Not too expense and has enough storage for most users and many ports .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"storage\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Not too expense and has enough storage for most users and many ports .\", \"aspect_start\": 31, \"aspect_end\": 38, \"evidence_start\": 0, \"evidence_end\": 70}, {\"aspect\": \"ports\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Not too expense and has enough storage for most users and many ports .\", \"aspect_start\": 63, \"aspect_end\": 68, \"evidence_start\": 0, \"evidence_end\": 70}], \"review_id\": \"gold_rev_0067\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Note , however , that any existing MagSafe accessories you have will not work with the MagSafe 2 connection .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"MagSafe accessories\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"Note , however , that any existing MagSafe accessories you have will not work with the MagSafe 2 connection .\", \"aspect_start\": 35, \"aspect_end\": 54, \"evidence_start\": 0, \"evidence_end\": 109}, {\"aspect\": \"MagSafe 2 connection\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Note , however , that any existing MagSafe accessories you have will not work with the MagSafe 2 connection .\", \"aspect_start\": 87, \"aspect_end\": 107, \"evidence_start\": 0, \"evidence_end\": 109}], \"review_id\": \"gold_rev_0068\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Screen - although some people might complain about low res which I think is ridiculous .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Screen\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Screen - although some people might complain about low res which I think is ridiculous .\", \"aspect_start\": 0, \"aspect_end\": 6, \"evidence_start\": 0, \"evidence_end\": 88}, {\"aspect\": \"res\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Screen - although some people might complain about low res which I think is ridiculous .\", \"aspect_start\": 55, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 88}], \"review_id\": \"gold_rev_0069\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Strong build though which really adds to its durability .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"durability\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Strong build though which really adds to its durability .\", \"aspect_start\": 45, \"aspect_end\": 55, \"evidence_start\": 0, \"evidence_end\": 57}, {\"aspect\": \"build\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Strong build though which really adds to its durability .\", \"aspect_start\": 7, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 57}], \"review_id\": \"gold_rev_0070\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The Mac mini is about 8x smaller than my old computer which is a huge bonus and runs very quiet , actually the fans are n't audible unlike my old pc\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"runs\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"The Mac mini is about 8x smaller than my old computer which is a huge bonus and runs very quiet , actually the fans are n't audible unlike my old pc\", \"aspect_start\": 80, \"aspect_end\": 84, \"evidence_start\": 0, \"evidence_end\": 148}, {\"aspect\": \"fans\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"The Mac mini is about 8x smaller than my old computer which is a huge bonus and runs very quiet , actually the fans are n't audible unlike my old pc\", \"aspect_start\": 111, \"aspect_end\": 115, \"evidence_start\": 0, \"evidence_end\": 148}], \"review_id\": \"gold_rev_0071\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The Mountain Lion OS is not hard to figure out if you are familiar with Microsoft Windows .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Mountain Lion OS\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"The Mountain Lion OS is not hard to figure out if you are familiar with Microsoft Windows .\", \"aspect_start\": 4, \"aspect_end\": 20, \"evidence_start\": 0, \"evidence_end\": 91}, {\"aspect\": \"Microsoft Windows\", \"category\": \"SOUND\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"The Mountain Lion OS is not hard to figure out if you are familiar with Microsoft Windows .\", \"aspect_start\": 72, \"aspect_end\": 89, \"evidence_start\": 0, \"evidence_end\": 91}], \"review_id\": \"gold_rev_0072\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The case is now slightly larger than the previous generation , but the lack of an external power supply justifies the small increase in size .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"case\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The case is now slightly larger than the previous generation , but the lack of an external power supply justifies the small increase in size .\", \"aspect_start\": 4, \"aspect_end\": 8, \"evidence_start\": 0, \"evidence_end\": 142}, {\"aspect\": \"external power supply\", \"category\": \"BATTERY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"The case is now slightly larger than the previous generation , but the lack of an external power supply justifies the small increase in size .\", \"aspect_start\": 82, \"aspect_end\": 103, \"evidence_start\": 0, \"evidence_end\": 142}], \"review_id\": \"gold_rev_0073\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Then the system would many times not power down without a forced power-off .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"system\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Then the system would many times not power down without a forced power-off .\", \"aspect_start\": 9, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 76}, {\"aspect\": \"power down\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Then the system would many times not power down without a forced power-off .\", \"aspect_start\": 37, \"aspect_end\": 47, \"evidence_start\": 0, \"evidence_end\": 76}], \"review_id\": \"gold_rev_0074\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"They do n't just look good ; they deliver excellent performance .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"look\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"They do n't just look good ; they deliver excellent performance .\", \"aspect_start\": 17, \"aspect_end\": 21, \"evidence_start\": 0, \"evidence_end\": 65}, {\"aspect\": \"performance\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"They do n't just look good ; they deliver excellent performance .\", \"aspect_start\": 52, \"aspect_end\": 63, \"evidence_start\": 0, \"evidence_end\": 65}], \"review_id\": \"gold_rev_0075\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"When considering a Mac , look at the total cost of ownership and not just the initial price tag .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"cost of ownership\", \"category\": \"SOFTWARE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"When considering a Mac , look at the total cost of ownership and not just the initial price tag .\", \"aspect_start\": 43, \"aspect_end\": 60, \"evidence_start\": 0, \"evidence_end\": 97}, {\"aspect\": \"price tag\", \"category\": \"VALUE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"When considering a Mac , look at the total cost of ownership and not just the initial price tag .\", \"aspect_start\": 86, \"aspect_end\": 95, \"evidence_start\": 0, \"evidence_end\": 97}], \"review_id\": \"gold_rev_0076\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"air has higher resolution but the fonts are small .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"resolution\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"air has higher resolution but the fonts are small .\", \"aspect_start\": 15, \"aspect_end\": 25, \"evidence_start\": 0, \"evidence_end\": 51}, {\"aspect\": \"fonts\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"air has higher resolution but the fonts are small .\", \"aspect_start\": 34, \"aspect_end\": 39, \"evidence_start\": 0, \"evidence_end\": 51}], \"review_id\": \"gold_rev_0077\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"the volume is really low to low for a laptopwas not expectin t volume to be so lowan i hate that about this computer\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"volume\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the volume is really low to low for a laptopwas not expectin t volume to be so lowan i hate that about this computer\", \"aspect_start\": 4, \"aspect_end\": 10, \"evidence_start\": 0, \"evidence_end\": 116}, {\"aspect\": \"volume\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the volume is really low to low for a laptopwas not expectin t volume to be so lowan i hate that about this computer\", \"aspect_start\": 4, \"aspect_end\": 10, \"evidence_start\": 0, \"evidence_end\": 116}], \"review_id\": \"gold_rev_0078\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"- sound is very low out of the box, but turning off all the stupid dbx processing in the sound settings makes it much louder and much better in short, this is a tv that is future proof with all the required connections / technologies that will keep you satisfied with its amazing picture quality for a very long time.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"- sound is very low out of the box, but turning off all the stupid dbx processing in the sound settings makes it much louder and much better in short, this is a tv that is future proof with all the required connections / technologies that will keep you satisfied with its amazing picture quality for a very long time.\", \"aspect_start\": 2, \"aspect_end\": 7, \"evidence_start\": 0, \"evidence_end\": 317}, {\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"- sound is very low out of the box, but turning off all the stupid dbx processing in the sound settings makes it much louder and much better in short, this is a tv that is future proof with all the required connections / technologies that will keep you satisfied with its amazing picture quality for a very long time.\", \"aspect_start\": 280, \"aspect_end\": 295, \"evidence_start\": 0, \"evidence_end\": 317}], \"review_id\": \"gold_rev_0079\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"> low power consumption 80w on the web page, but didn't have the tv long enough to look at the back label > hdmi worked correctly ( with laptop 1366x768 resolution) with audio.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"power consumption\", \"category\": \"BATTERY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"> low power consumption 80w on the web page, but didn't have the tv long enough to look at the back label > hdmi worked correctly ( with laptop 1366x768 resolution) with audio.\", \"aspect_start\": 6, \"aspect_end\": 23, \"evidence_start\": 0, \"evidence_end\": 176}, {\"aspect\": \"hdmi\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"> low power consumption 80w on the web page, but didn't have the tv long enough to look at the back label > hdmi worked correctly ( with laptop 1366x768 resolution) with audio.\", \"aspect_start\": 108, \"aspect_end\": 112, \"evidence_start\": 0, \"evidence_end\": 176}], \"review_id\": \"gold_rev_0080\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"also the installed apps was not opening like Hotstar youtube.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Hotstar\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"also the installed apps was not opening like Hotstar youtube.\", \"aspect_start\": 45, \"aspect_end\": 52, \"evidence_start\": 0, \"evidence_end\": 61}, {\"aspect\": \"apps\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"also the installed apps was not opening like Hotstar youtube.\", \"aspect_start\": 19, \"aspect_end\": 23, \"evidence_start\": 0, \"evidence_end\": 61}], \"review_id\": \"gold_rev_0081\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"and i have converted this tv as smart tv by placing tanix tx3 android box and one more thing the audio out from 3.5 mm jack is not controlled by tv remote so except these two things this is unbelievable product under this price range.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"audio out\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"and i have converted this tv as smart tv by placing tanix tx3 android box and one more thing the audio out from 3.5 mm jack is not controlled by tv remote so except these two things this is unbelievable product under this price range.\", \"aspect_start\": 97, \"aspect_end\": 106, \"evidence_start\": 0, \"evidence_end\": 234}, {\"aspect\": \"remote\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"and i have converted this tv as smart tv by placing tanix tx3 android box and one more thing the audio out from 3.5 mm jack is not controlled by tv remote so except these two things this is unbelievable product under this price range.\", \"aspect_start\": 148, \"aspect_end\": 154, \"evidence_start\": 0, \"evidence_end\": 234}], \"review_id\": \"gold_rev_0082\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"and the installation is quick from the side of jeeve service and one thing is so good that the extended warranty gives me a headaceless till 3 years which may not other brands gives.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"installation\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"and the installation is quick from the side of jeeve service and one thing is so good that the extended warranty gives me a headaceless till 3 years which may not other brands gives.\", \"aspect_start\": 8, \"aspect_end\": 20, \"evidence_start\": 0, \"evidence_end\": 182}, {\"aspect\": \"warranty\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"and the installation is quick from the side of jeeve service and one thing is so good that the extended warranty gives me a headaceless till 3 years which may not other brands gives.\", \"aspect_start\": 104, \"aspect_end\": 112, \"evidence_start\": 0, \"evidence_end\": 182}], \"review_id\": \"gold_rev_0083\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"as far as sound quality is concerned then everybody who is going to purchase led tv of any brand any model must bear one thing in mind that due to the slim design of led tvs it is practically not possible to incorporate good quality speakers into them because good quality speakers consume space and are always wide ( voluminous).\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"quality speakers\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"as far as sound quality is concerned then everybody who is going to purchase led tv of any brand any model must bear one thing in mind that due to the slim design of led tvs it is practically not possible to incorporate good quality speakers into them because good quality speakers consume space and are always wide ( voluminous).\", \"aspect_start\": 225, \"aspect_end\": 241, \"evidence_start\": 0, \"evidence_end\": 330}, {\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"as far as sound quality is concerned then everybody who is going to purchase led tv of any brand any model must bear one thing in mind that due to the slim design of led tvs it is practically not possible to incorporate good quality speakers into them because good quality speakers consume space and are always wide ( voluminous).\", \"aspect_start\": 10, \"aspect_end\": 23, \"evidence_start\": 0, \"evidence_end\": 330}], \"review_id\": \"gold_rev_0084\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"awesome product but only drawback is not having appstore, and remote is air remote .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"air remote\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"awesome product but only drawback is not having appstore, and remote is air remote .\", \"aspect_start\": 72, \"aspect_end\": 82, \"evidence_start\": 0, \"evidence_end\": 84}, {\"aspect\": \"remote\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"awesome product but only drawback is not having appstore, and remote is air remote .\", \"aspect_start\": 62, \"aspect_end\": 68, \"evidence_start\": 0, \"evidence_end\": 84}], \"review_id\": \"gold_rev_0085\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"bough this for my another room & except little bit sound quality difference which is not so huge, picture quality is same quality.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"bough this for my another room & except little bit sound quality difference which is not so huge, picture quality is same quality.\", \"aspect_start\": 51, \"aspect_end\": 64, \"evidence_start\": 0, \"evidence_end\": 130}, {\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"bough this for my another room & except little bit sound quality difference which is not so huge, picture quality is same quality.\", \"aspect_start\": 98, \"aspect_end\": 113, \"evidence_start\": 0, \"evidence_end\": 130}], \"review_id\": \"gold_rev_0086\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"build quality is not good, when tv is on boot light near bazel is show 2 and 3 places, i bought 3 vu led tv( 43\\\", two 32\\\") on this festive season but all of three has same problem\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"build quality\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"build quality is not good, when tv is on boot light near bazel is show 2 and 3 places, i bought 3 vu led tv( 43\\\", two 32\\\") on this festive season but all of three has same problem\", \"aspect_start\": 0, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 179}, {\"aspect\": \"boot light\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"build quality is not good, when tv is on boot light near bazel is show 2 and 3 places, i bought 3 vu led tv( 43\\\", two 32\\\") on this festive season but all of three has same problem\", \"aspect_start\": 41, \"aspect_end\": 51, \"evidence_start\": 0, \"evidence_end\": 179}], \"review_id\": \"gold_rev_0087\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but Flipkart packaging n delivery are below average.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"packaging\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but Flipkart packaging n delivery are below average.\", \"aspect_start\": 13, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 52}, {\"aspect\": \"delivery\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but Flipkart packaging n delivery are below average.\", \"aspect_start\": 25, \"aspect_end\": 33, \"evidence_start\": 0, \"evidence_end\": 52}], \"review_id\": \"gold_rev_0088\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"clarity good, sound good, bad only some format sound not work.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"clarity\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"clarity good, sound good, bad only some format sound not work.\", \"aspect_start\": 0, \"aspect_end\": 7, \"evidence_start\": 0, \"evidence_end\": 62}, {\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"clarity good, sound good, bad only some format sound not work.\", \"aspect_start\": 14, \"aspect_end\": 19, \"evidence_start\": 0, \"evidence_end\": 62}], \"review_id\": \"gold_rev_0089\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"installation and service is very poor no sign of them yet, keeps on rescheduling from past 3 days.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"installation\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"installation and service is very poor no sign of them yet, keeps on rescheduling from past 3 days.\", \"aspect_start\": 0, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 98}, {\"aspect\": \"service\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"installation and service is very poor no sign of them yet, keeps on rescheduling from past 3 days.\", \"aspect_start\": 17, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 98}], \"review_id\": \"gold_rev_0090\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"nice product in low price,but delivery is to delay picture quality is good and sound also.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"nice product in low price,but delivery is to delay picture quality is good and sound also.\", \"aspect_start\": 51, \"aspect_end\": 66, \"evidence_start\": 0, \"evidence_end\": 90}, {\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"nice product in low price,but delivery is to delay picture quality is good and sound also.\", \"aspect_start\": 79, \"aspect_end\": 84, \"evidence_start\": 0, \"evidence_end\": 90}], \"review_id\": \"gold_rev_0091\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"no optical output but still ( coaxial/hdmi arc alternative can be used) cons: remote quality bad audio output extremely very bad.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"remote quality\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"no optical output but still ( coaxial/hdmi arc alternative can be used) cons: remote quality bad audio output extremely very bad.\", \"aspect_start\": 78, \"aspect_end\": 92, \"evidence_start\": 0, \"evidence_end\": 129}, {\"aspect\": \"audio output\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"no optical output but still ( coaxial/hdmi arc alternative can be used) cons: remote quality bad audio output extremely very bad.\", \"aspect_start\": 97, \"aspect_end\": 109, \"evidence_start\": 0, \"evidence_end\": 129}], \"review_id\": \"gold_rev_0092\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"not compactable with keyboard, and mirror cast is not working, and no response from Flipkart help centre\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"mirror cast\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"not compactable with keyboard, and mirror cast is not working, and no response from Flipkart help centre\", \"aspect_start\": 35, \"aspect_end\": 46, \"evidence_start\": 0, \"evidence_end\": 104}, {\"aspect\": \"Flipkart help centre\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"not compactable with keyboard, and mirror cast is not working, and no response from Flipkart help centre\", \"aspect_start\": 84, \"aspect_end\": 104, \"evidence_start\": 0, \"evidence_end\": 104}], \"review_id\": \"gold_rev_0093\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"only bad thing about the tv is it is not supporting subtitles while playing movies from pen drive, hope vu will update the player in future\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"subtitles\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"only bad thing about the tv is it is not supporting subtitles while playing movies from pen drive, hope vu will update the player in future\", \"aspect_start\": 52, \"aspect_end\": 61, \"evidence_start\": 0, \"evidence_end\": 139}, {\"aspect\": \"player\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"only bad thing about the tv is it is not supporting subtitles while playing movies from pen drive, hope vu will update the player in future\", \"aspect_start\": 123, \"aspect_end\": 129, \"evidence_start\": 0, \"evidence_end\": 139}], \"review_id\": \"gold_rev_0094\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"only the concern is it's placement when you hang tv on wall few hdmi port & vga port is not accessible.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"hdmi port\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"only the concern is it's placement when you hang tv on wall few hdmi port & vga port is not accessible.\", \"aspect_start\": 64, \"aspect_end\": 73, \"evidence_start\": 0, \"evidence_end\": 103}, {\"aspect\": \"vga port\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"only the concern is it's placement when you hang tv on wall few hdmi port & vga port is not accessible.\", \"aspect_start\": 76, \"aspect_end\": 84, \"evidence_start\": 0, \"evidence_end\": 103}], \"review_id\": \"gold_rev_0095\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture clarity is nice but audio is very poor.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture clarity\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture clarity is nice but audio is very poor.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 47}, {\"aspect\": \"audio\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture clarity is nice but audio is very poor.\", \"aspect_start\": 28, \"aspect_end\": 33, \"evidence_start\": 0, \"evidence_end\": 47}], \"review_id\": \"gold_rev_0096\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture clarity is not so good and audio is not clear.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"picture clarity\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture clarity is not so good and audio is not clear.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 54}, {\"aspect\": \"audio\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture clarity is not so good and audio is not clear.\", \"aspect_start\": 35, \"aspect_end\": 40, \"evidence_start\": 0, \"evidence_end\": 54}], \"review_id\": \"gold_rev_0097\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality is draw back but all features are good\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture quality is draw back but all features are good\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 54}, {\"aspect\": \"features\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture quality is draw back but all features are good\", \"aspect_start\": 37, \"aspect_end\": 45, \"evidence_start\": 0, \"evidence_end\": 54}], \"review_id\": \"gold_rev_0098\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality is good , sound quality may lag but good value for money overall.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture quality is good , sound quality may lag but good value for money overall.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 81}, {\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture quality is good , sound quality may lag but good value for money overall.\", \"aspect_start\": 26, \"aspect_end\": 39, \"evidence_start\": 0, \"evidence_end\": 81}], \"review_id\": \"gold_rev_0099\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality is good but audio not up to the level.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture quality is good but audio not up to the level.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 54}, {\"aspect\": \"audio\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture quality is good but audio not up to the level.\", \"aspect_start\": 28, \"aspect_end\": 33, \"evidence_start\": 0, \"evidence_end\": 54}], \"review_id\": \"gold_rev_0100\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality is good, but the sound is too low for a spacious living room, i am satisfied with my buy though.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture quality is good, but the sound is too low for a spacious living room, i am satisfied with my buy though.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 112}, {\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture quality is good, but the sound is too low for a spacious living room, i am satisfied with my buy though.\", \"aspect_start\": 33, \"aspect_end\": 38, \"evidence_start\": 0, \"evidence_end\": 112}], \"review_id\": \"gold_rev_0101\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality is very good but sound quality is some less then other companies like sony, lg etc.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture quality is very good but sound quality is some less then other companies like sony, lg etc.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 99}, {\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture quality is very good but sound quality is some less then other companies like sony, lg etc.\", \"aspect_start\": 33, \"aspect_end\": 46, \"evidence_start\": 0, \"evidence_end\": 99}], \"review_id\": \"gold_rev_0102\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture sound features all excellent but no 3d.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture sound features all excellent but no 3d.\", \"aspect_start\": 0, \"aspect_end\": 7, \"evidence_start\": 0, \"evidence_end\": 47}, {\"aspect\": \"sound features\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture sound features all excellent but no 3d.\", \"aspect_start\": 8, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 47}], \"review_id\": \"gold_rev_0103\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"product delivered in right time but installation not completed still.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"installation\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"product delivered in right time but installation not completed still.\", \"aspect_start\": 36, \"aspect_end\": 48, \"evidence_start\": 0, \"evidence_end\": 69}, {\"aspect\": \"delivered\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"product delivered in right time but installation not completed still.\", \"aspect_start\": 8, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 69}], \"review_id\": \"gold_rev_0104\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"samsang led tv good quality , picture quality is very good but sound quality not good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"samsang led tv good quality , picture quality is very good but sound quality not good.\", \"aspect_start\": 30, \"aspect_end\": 45, \"evidence_start\": 0, \"evidence_end\": 86}, {\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"samsang led tv good quality , picture quality is very good but sound quality not good.\", \"aspect_start\": 63, \"aspect_end\": 76, \"evidence_start\": 0, \"evidence_end\": 86}], \"review_id\": \"gold_rev_0105\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"sound good installation mount not good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"sound good installation mount not good.\", \"aspect_start\": 0, \"aspect_end\": 5, \"evidence_start\": 0, \"evidence_end\": 39}, {\"aspect\": \"installation mount\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"sound good installation mount not good.\", \"aspect_start\": 11, \"aspect_end\": 29, \"evidence_start\": 0, \"evidence_end\": 39}], \"review_id\": \"gold_rev_0106\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"sound quality good but volume is low.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"sound quality good but volume is low.\", \"aspect_start\": 0, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 37}, {\"aspect\": \"volume\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"sound quality good but volume is low.\", \"aspect_start\": 23, \"aspect_end\": 29, \"evidence_start\": 0, \"evidence_end\": 37}], \"review_id\": \"gold_rev_0107\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"the picture quality is fantastic, not great viewing angles\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"the picture quality is fantastic, not great viewing angles\", \"aspect_start\": 4, \"aspect_end\": 19, \"evidence_start\": 0, \"evidence_end\": 58}, {\"aspect\": \"viewing angles\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the picture quality is fantastic, not great viewing angles\", \"aspect_start\": 44, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 58}], \"review_id\": \"gold_rev_0108\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"this product god picture quality very god but sund quality not god\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"this product god picture quality very god but sund quality not god\", \"aspect_start\": 17, \"aspect_end\": 32, \"evidence_start\": 0, \"evidence_end\": 66}, {\"aspect\": \"sund quality\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"this product god picture quality very god but sund quality not god\", \"aspect_start\": 46, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 66}], \"review_id\": \"gold_rev_0109\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"tv stop working on next day from the installation, blank screen coming on tv, remote is not working, panel looking like fix by tape.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"remote\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"tv stop working on next day from the installation, blank screen coming on tv, remote is not working, panel looking like fix by tape.\", \"aspect_start\": 78, \"aspect_end\": 84, \"evidence_start\": 0, \"evidence_end\": 132}, {\"aspect\": \"panel\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"tv stop working on next day from the installation, blank screen coming on tv, remote is not working, panel looking like fix by tape.\", \"aspect_start\": 101, \"aspect_end\": 106, \"evidence_start\": 0, \"evidence_end\": 132}], \"review_id\": \"gold_rev_0110\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"very very poor product vu led after 2 months got repair mother board fault 6 days left but till date mother board not change by vu don't buy this product waste of money\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"mother board\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"very very poor product vu led after 2 months got repair mother board fault 6 days left but till date mother board not change by vu don't buy this product waste of money\", \"aspect_start\": 56, \"aspect_end\": 68, \"evidence_start\": 0, \"evidence_end\": 168}, {\"aspect\": \"mother board\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"very very poor product vu led after 2 months got repair mother board fault 6 days left but till date mother board not change by vu don't buy this product waste of money\", \"aspect_start\": 56, \"aspect_end\": 68, \"evidence_start\": 0, \"evidence_end\": 168}], \"review_id\": \"gold_rev_0111\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"well sound is loud , need homw theater cons contrast not good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"well sound is loud , need homw theater cons contrast not good.\", \"aspect_start\": 5, \"aspect_end\": 10, \"evidence_start\": 0, \"evidence_end\": 62}, {\"aspect\": \"contrast\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"well sound is loud , need homw theater cons contrast not good.\", \"aspect_start\": 44, \"aspect_end\": 52, \"evidence_start\": 0, \"evidence_end\": 62}], \"review_id\": \"gold_rev_0112\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"bit expensive but fabric is awesome fit is awesome !\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"bit expensive but fabric is awesome fit is awesome !\", \"aspect_start\": 18, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 52}, {\"aspect\": \"fit\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"bit expensive but fabric is awesome fit is awesome !\", \"aspect_start\": 36, \"aspect_end\": 39, \"evidence_start\": 0, \"evidence_end\": 52}], \"review_id\": \"gold_rev_0113\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"fitting is good, fabric is a bit thin , but at this price it is acceptable.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"fitting\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"fitting is good, fabric is a bit thin , but at this price it is acceptable.\", \"aspect_start\": 0, \"aspect_end\": 7, \"evidence_start\": 0, \"evidence_end\": 75}, {\"aspect\": \"fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"fitting is good, fabric is a bit thin , but at this price it is acceptable.\", \"aspect_start\": 17, \"aspect_end\": 23, \"evidence_start\": 0, \"evidence_end\": 75}], \"review_id\": \"gold_rev_0114\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"good fitting but color very light average tshirt.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"fitting\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"good fitting but color very light average tshirt.\", \"aspect_start\": 5, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 49}, {\"aspect\": \"color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"good fitting but color very light average tshirt.\", \"aspect_start\": 17, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 49}], \"review_id\": \"gold_rev_0115\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"good material n comfortable to wear,but there was size variation.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"good material n comfortable to wear,but there was size variation.\", \"aspect_start\": 5, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 65}, {\"aspect\": \"size\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"good material n comfortable to wear,but there was size variation.\", \"aspect_start\": 50, \"aspect_end\": 54, \"evidence_start\": 0, \"evidence_end\": 65}], \"review_id\": \"gold_rev_0116\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"it looks and fits good but the fabric could have been a little better.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"fits\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"it looks and fits good but the fabric could have been a little better.\", \"aspect_start\": 13, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 70}, {\"aspect\": \"fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"it looks and fits good but the fabric could have been a little better.\", \"aspect_start\": 31, \"aspect_end\": 37, \"evidence_start\": 0, \"evidence_end\": 70}], \"review_id\": \"gold_rev_0117\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"material 8s good but size are not perfect.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"material 8s good but size are not perfect.\", \"aspect_start\": 0, \"aspect_end\": 8, \"evidence_start\": 0, \"evidence_end\": 42}, {\"aspect\": \"size\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"material 8s good but size are not perfect.\", \"aspect_start\": 21, \"aspect_end\": 25, \"evidence_start\": 0, \"evidence_end\": 42}], \"review_id\": \"gold_rev_0118\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"material is good but after 2 month colour going to fad almost.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"material is good but after 2 month colour going to fad almost.\", \"aspect_start\": 0, \"aspect_end\": 8, \"evidence_start\": 0, \"evidence_end\": 62}, {\"aspect\": \"colour\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"material is good but after 2 month colour going to fad almost.\", \"aspect_start\": 35, \"aspect_end\": 41, \"evidence_start\": 0, \"evidence_end\": 62}], \"review_id\": \"gold_rev_0119\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"product ws good material also good but medium size is too big.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"product ws good material also good but medium size is too big.\", \"aspect_start\": 16, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 62}, {\"aspect\": \"medium size\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"product ws good material also good but medium size is too big.\", \"aspect_start\": 39, \"aspect_end\": 50, \"evidence_start\": 0, \"evidence_end\": 62}], \"review_id\": \"gold_rev_0120\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"quality good but size not perfect.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"quality good but size not perfect.\", \"aspect_start\": 0, \"aspect_end\": 7, \"evidence_start\": 0, \"evidence_end\": 34}, {\"aspect\": \"size\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"quality good but size not perfect.\", \"aspect_start\": 17, \"aspect_end\": 21, \"evidence_start\": 0, \"evidence_end\": 34}], \"review_id\": \"gold_rev_0121\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"size and fit is not good as expected.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"size\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"size and fit is not good as expected.\", \"aspect_start\": 0, \"aspect_end\": 4, \"evidence_start\": 0, \"evidence_end\": 37}, {\"aspect\": \"fit\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"size and fit is not good as expected.\", \"aspect_start\": 9, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 37}], \"review_id\": \"gold_rev_0122\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"size is perfect but color is so dull.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"size\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"size is perfect but color is so dull.\", \"aspect_start\": 0, \"aspect_end\": 4, \"evidence_start\": 0, \"evidence_end\": 37}, {\"aspect\": \"color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"size is perfect but color is so dull.\", \"aspect_start\": 20, \"aspect_end\": 25, \"evidence_start\": 0, \"evidence_end\": 37}], \"review_id\": \"gold_rev_0123\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"the material and product was good but the size did not fit hence returned it.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"the material and product was good but the size did not fit hence returned it.\", \"aspect_start\": 4, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 77}, {\"aspect\": \"size\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the material and product was good but the size did not fit hence returned it.\", \"aspect_start\": 42, \"aspect_end\": 46, \"evidence_start\": 0, \"evidence_end\": 77}], \"review_id\": \"gold_rev_0124\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"the product is good however the price is little bit high but thanks indeed the delivery boy was also so helpful he waited 2 hrs to deliver to me a spedy delivery too thanks\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"price\", \"category\": \"VALUE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the product is good however the price is little bit high but thanks indeed the delivery boy was also so helpful he waited 2 hrs to deliver to me a spedy delivery too thanks\", \"aspect_start\": 32, \"aspect_end\": 37, \"evidence_start\": 0, \"evidence_end\": 172}, {\"aspect\": \"delivery boy\", \"category\": \"DELIVERY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"the product is good however the price is little bit high but thanks indeed the delivery boy was also so helpful he waited 2 hrs to deliver to me a spedy delivery too thanks\", \"aspect_start\": 79, \"aspect_end\": 91, \"evidence_start\": 0, \"evidence_end\": 172}], \"review_id\": \"gold_rev_0125\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"though the cloth is comfort to wear buthe th e color does not match as on the image of the order given.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"cloth\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"though the cloth is comfort to wear buthe th e color does not match as on the image of the order given.\", \"aspect_start\": 11, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 103}, {\"aspect\": \"color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"though the cloth is comfort to wear buthe th e color does not match as on the image of the order given.\", \"aspect_start\": 47, \"aspect_end\": 52, \"evidence_start\": 0, \"evidence_end\": 103}], \"review_id\": \"gold_rev_0126\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"very bad product color fades and cloth is not good.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"very bad product color fades and cloth is not good.\", \"aspect_start\": 17, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 51}, {\"aspect\": \"cloth\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"very bad product color fades and cloth is not good.\", \"aspect_start\": 33, \"aspect_end\": 38, \"evidence_start\": 0, \"evidence_end\": 51}], \"review_id\": \"gold_rev_0127\"}", "{\"product_id\": \"PROD_APP_TSHIRT\", \"text\": \"very comfortable in wearing but the cloth of the t shirt is too light in weight.\", \"vertical\": \"apparel_fashion\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"wearing\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"very comfortable in wearing but the cloth of the t shirt is too light in weight.\", \"aspect_start\": 20, \"aspect_end\": 27, \"evidence_start\": 0, \"evidence_end\": 80}, {\"aspect\": \"cloth of the t shirt\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"very comfortable in wearing but the cloth of the t shirt is too light in weight.\", \"aspect_start\": 36, \"aspect_end\": 56, \"evidence_start\": 0, \"evidence_end\": 80}], \"review_id\": \"gold_rev_0128\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"-LRB- but Office can be purchased -RRB- IF I ever need a laptop again I am for sure purchasing another Toshiba !!\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Office\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"-LRB- but Office can be purchased -RRB- IF I ever need a laptop again I am for sure purchasing another Toshiba !!\", \"aspect_start\": 10, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 113}], \"review_id\": \"gold_rev_0129\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"A veryimportant feature is Firewire 80Negative which in my experience works better then USB3 -LRB- in PC enabled with USB3 -RRB- I was not originally sold on the MAC OS I felt it was inferior in many ways To Windows 7 .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Firewire 80Negative\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"A veryimportant feature is Firewire 80Negative which in my experience works better then USB3 -LRB- in PC enabled with USB3 -RRB- I was not originally sold on the MAC OS I felt it was inferior in many ways To Windows 7 .\", \"aspect_start\": 27, \"aspect_end\": 46, \"evidence_start\": 0, \"evidence_end\": 219}], \"review_id\": \"gold_rev_0130\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"And I may be the only one but I am really liking Windows 8 .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Windows 8\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"And I may be the only one but I am really liking Windows 8 .\", \"aspect_start\": 49, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 60}], \"review_id\": \"gold_rev_0131\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"And mine had broke but I sent it in under warranty - no problems .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"warranty\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"And mine had broke but I sent it in under warranty - no problems .\", \"aspect_start\": 42, \"aspect_end\": 50, \"evidence_start\": 0, \"evidence_end\": 66}], \"review_id\": \"gold_rev_0132\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Apple no longer includes iDVD with the computer and furthermore , Apple does n't even offer it anymore !\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"iDVD\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Apple no longer includes iDVD with the computer and furthermore , Apple does n't even offer it anymore !\", \"aspect_start\": 25, \"aspect_end\": 29, \"evidence_start\": 0, \"evidence_end\": 104}], \"review_id\": \"gold_rev_0133\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"At home and the office it gets plugged into an external 24 '' LCD screen , so built in screen size is not terribly important .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"built in screen size\", \"category\": \"DISPLAY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"At home and the office it gets plugged into an external 24 '' LCD screen , so built in screen size is not terribly important .\", \"aspect_start\": 78, \"aspect_end\": 98, \"evidence_start\": 0, \"evidence_end\": 126}], \"review_id\": \"gold_rev_0134\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"At home and the office it gets plugged into an external 24\\\" LCD screen , so built in screen size is not terribly important .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"external 24\\\" LCD screen\", \"category\": \"DISPLAY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"At home and the office it gets plugged into an external 24\\\" LCD screen , so built in screen size is not terribly important .\", \"aspect_start\": 47, \"aspect_end\": 70, \"evidence_start\": 0, \"evidence_end\": 124}], \"review_id\": \"gold_rev_0135\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"But I do NOT like Win8 .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Win8\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"But I do NOT like Win8 .\", \"aspect_start\": 18, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 24}], \"review_id\": \"gold_rev_0136\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"But I wanted the Pro for the CD/DVD player .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"CD/DVD player\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"But I wanted the Pro for the CD/DVD player .\", \"aspect_start\": 29, \"aspect_end\": 42, \"evidence_start\": 0, \"evidence_end\": 44}], \"review_id\": \"gold_rev_0137\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"But I was completely wrong , this computer is UNBELIEVABLE amazing and easy to use .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"use\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"But I was completely wrong , this computer is UNBELIEVABLE amazing and easy to use .\", \"aspect_start\": 79, \"aspect_end\": 82, \"evidence_start\": 0, \"evidence_end\": 84}], \"review_id\": \"gold_rev_0138\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"But the mountain lion is just too slow .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"mountain lion\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"But the mountain lion is just too slow .\", \"aspect_start\": 8, \"aspect_end\": 21, \"evidence_start\": 0, \"evidence_end\": 40}], \"review_id\": \"gold_rev_0139\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"But the performance of Mac Mini is a huge disappointment .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"performance\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"But the performance of Mac Mini is a huge disappointment .\", \"aspect_start\": 8, \"aspect_end\": 19, \"evidence_start\": 0, \"evidence_end\": 58}], \"review_id\": \"gold_rev_0140\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Does everything I need it to , has a wonderful battery life and I could n't be happier .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"battery life\", \"category\": \"BATTERY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Does everything I need it to , has a wonderful battery life and I could n't be happier .\", \"aspect_start\": 47, \"aspect_end\": 59, \"evidence_start\": 0, \"evidence_end\": 88}], \"review_id\": \"gold_rev_0141\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"HOWEVER I chose two day shipping and it took over a week to arrive .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"shipping\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"HOWEVER I chose two day shipping and it took over a week to arrive .\", \"aspect_start\": 24, \"aspect_end\": 32, \"evidence_start\": 0, \"evidence_end\": 68}], \"review_id\": \"gold_rev_0142\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Having heard from friends and family about how reliable a Mac product is , I never expected to have an application crash within the first month , but I did .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"application\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Having heard from friends and family about how reliable a Mac product is , I never expected to have an application crash within the first month , but I did .\", \"aspect_start\": 103, \"aspect_end\": 114, \"evidence_start\": 0, \"evidence_end\": 157}], \"review_id\": \"gold_rev_0143\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"However , I can refute that OSX is `` FAST '' .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"OSX\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"However , I can refute that OSX is `` FAST '' .\", \"aspect_start\": 28, \"aspect_end\": 31, \"evidence_start\": 0, \"evidence_end\": 47}], \"review_id\": \"gold_rev_0144\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"However , there are MAJOR issues with the touchpad which render the device nearly useless .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"touchpad\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"However , there are MAJOR issues with the touchpad which render the device nearly useless .\", \"aspect_start\": 42, \"aspect_end\": 50, \"evidence_start\": 0, \"evidence_end\": 91}], \"review_id\": \"gold_rev_0145\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I Have been a Pc user for a very long time now but I will get used to this new OS .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"OS\", \"category\": \"SOFTWARE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I Have been a Pc user for a very long time now but I will get used to this new OS .\", \"aspect_start\": 79, \"aspect_end\": 81, \"evidence_start\": 0, \"evidence_end\": 83}], \"review_id\": \"gold_rev_0146\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I do not like too much Windows 8 .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Windows 8\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"I do not like too much Windows 8 .\", \"aspect_start\": 23, \"aspect_end\": 32, \"evidence_start\": 0, \"evidence_end\": 34}], \"review_id\": \"gold_rev_0147\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I have been PC for years but this computer is intuitive and its built in features are a great help\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"built in features\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I have been PC for years but this computer is intuitive and its built in features are a great help\", \"aspect_start\": 64, \"aspect_end\": 81, \"evidence_start\": 0, \"evidence_end\": 98}], \"review_id\": \"gold_rev_0148\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I have n't tried the one with retina display ... Maybe in the future .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"retina display\", \"category\": \"DISPLAY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I have n't tried the one with retina display ... Maybe in the future .\", \"aspect_start\": 30, \"aspect_end\": 44, \"evidence_start\": 0, \"evidence_end\": 70}], \"review_id\": \"gold_rev_0149\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I just bought the new MacBook Pro , the 13 '' model , and I ca n't believe Apple keeps making the same mistake with regard to USB ports .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"USB ports\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"I just bought the new MacBook Pro , the 13 '' model , and I ca n't believe Apple keeps making the same mistake with regard to USB ports .\", \"aspect_start\": 126, \"aspect_end\": 135, \"evidence_start\": 0, \"evidence_end\": 137}], \"review_id\": \"gold_rev_0150\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I never tried any external mics with that iMac .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"external mics\", \"category\": \"SOUND\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I never tried any external mics with that iMac .\", \"aspect_start\": 18, \"aspect_end\": 31, \"evidence_start\": 0, \"evidence_end\": 48}], \"review_id\": \"gold_rev_0151\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I switched to this because I wanted something different , even though I miss windows .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"windows\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I switched to this because I wanted something different , even though I miss windows .\", \"aspect_start\": 77, \"aspect_end\": 84, \"evidence_start\": 0, \"evidence_end\": 86}], \"review_id\": \"gold_rev_0152\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I was a little concerned about the touch pad based on reviews , but I 've found it fine to work with .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"touch pad\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I was a little concerned about the touch pad based on reviews , but I 've found it fine to work with .\", \"aspect_start\": 35, \"aspect_end\": 44, \"evidence_start\": 0, \"evidence_end\": 102}], \"review_id\": \"gold_rev_0153\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I was concerned that the downgrade to the regular hard drive would make it unacceptably slow but I need not have worried - this machine is the fastest I have ever owned ...\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"regular hard drive\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I was concerned that the downgrade to the regular hard drive would make it unacceptably slow but I need not have worried - this machine is the fastest I have ever owned ...\", \"aspect_start\": 42, \"aspect_end\": 60, \"evidence_start\": 0, \"evidence_end\": 172}], \"review_id\": \"gold_rev_0154\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I was considering buying the Air , but in reality , this one has a more powerful performance and seems much more convenient , even though it 's about .20 inch thicker and 2 lbs heavier .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"performance\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"I was considering buying the Air , but in reality , this one has a more powerful performance and seems much more convenient , even though it 's about .20 inch thicker and 2 lbs heavier .\", \"aspect_start\": 81, \"aspect_end\": 92, \"evidence_start\": 0, \"evidence_end\": 186}], \"review_id\": \"gold_rev_0155\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I was having a though time deciding between the 13 '' MacBook Air or the MacBook Pro 13 '' -LRB- Both baseline models , priced at 1,200 $ retail -RRB-\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"priced\", \"category\": \"VALUE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I was having a though time deciding between the 13 '' MacBook Air or the MacBook Pro 13 '' -LRB- Both baseline models , priced at 1,200 $ retail -RRB-\", \"aspect_start\": 120, \"aspect_end\": 126, \"evidence_start\": 0, \"evidence_end\": 150}], \"review_id\": \"gold_rev_0156\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I will not be using that slot again .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"slot\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"I will not be using that slot again .\", \"aspect_start\": 25, \"aspect_end\": 29, \"evidence_start\": 0, \"evidence_end\": 37}], \"review_id\": \"gold_rev_0157\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I work as a designer and coder and I needed a new buddy to work with , not gaming .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"gaming\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"I work as a designer and coder and I needed a new buddy to work with , not gaming .\", \"aspect_start\": 75, \"aspect_end\": 81, \"evidence_start\": 0, \"evidence_end\": 83}], \"review_id\": \"gold_rev_0158\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"I would have given it 5 starts was it not for the fact that it had Windows 8\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Windows 8\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"I would have given it 5 starts was it not for the fact that it had Windows 8\", \"aspect_start\": 67, \"aspect_end\": 76, \"evidence_start\": 0, \"evidence_end\": 76}], \"review_id\": \"gold_rev_0159\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"It 's been a couple weeks since the purchase and I 'm struggle with finding the correct keys -LRB- but that was expected -RRB- .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"keys\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"It 's been a couple weeks since the purchase and I 'm struggle with finding the correct keys -LRB- but that was expected -RRB- .\", \"aspect_start\": 88, \"aspect_end\": 92, \"evidence_start\": 0, \"evidence_end\": 128}], \"review_id\": \"gold_rev_0160\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"It 's not inexpensive but the Hardware performance is impressive for a computer this small .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Hardware performance\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"It 's not inexpensive but the Hardware performance is impressive for a computer this small .\", \"aspect_start\": 30, \"aspect_end\": 50, \"evidence_start\": 0, \"evidence_end\": 92}], \"review_id\": \"gold_rev_0161\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"It 's ok but does n't have a disk drive which I did n't know until after I bought it .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"disk drive\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"It 's ok but does n't have a disk drive which I did n't know until after I bought it .\", \"aspect_start\": 29, \"aspect_end\": 39, \"evidence_start\": 0, \"evidence_end\": 86}], \"review_id\": \"gold_rev_0162\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"It feels cheap , the keyboard is not very sensitive .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"keyboard\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"It feels cheap , the keyboard is not very sensitive .\", \"aspect_start\": 21, \"aspect_end\": 29, \"evidence_start\": 0, \"evidence_end\": 53}], \"review_id\": \"gold_rev_0163\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"It is very fast and has everything that I need except for a word program .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"word program\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"It is very fast and has everything that I need except for a word program .\", \"aspect_start\": 60, \"aspect_end\": 72, \"evidence_start\": 0, \"evidence_end\": 74}], \"review_id\": \"gold_rev_0164\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"It should n't happen like that , I do n't have any design app open or anything .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"design app\", \"category\": \"SOFTWARE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"It should n't happen like that , I do n't have any design app open or anything .\", \"aspect_start\": 51, \"aspect_end\": 61, \"evidence_start\": 0, \"evidence_end\": 80}], \"review_id\": \"gold_rev_0165\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"It suddenly can not work .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"work\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"It suddenly can not work .\", \"aspect_start\": 20, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 26}], \"review_id\": \"gold_rev_0166\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"MAYBE The Mac OS improvement were not The product they Want to offer .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Mac OS\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"MAYBE The Mac OS improvement were not The product they Want to offer .\", \"aspect_start\": 10, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 70}], \"review_id\": \"gold_rev_0167\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"MY TRACKPAD IS NOT WORKING .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"TRACKPAD\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"MY TRACKPAD IS NOT WORKING .\", \"aspect_start\": 3, \"aspect_end\": 11, \"evidence_start\": 0, \"evidence_end\": 28}], \"review_id\": \"gold_rev_0168\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"My last laptop was a 17 '' ASUS gaming machine , which performed admirably , but having since built my own desktop and really settling into the college life , I found myself wanting something smaller and less cumbersome , not to mention that the ASUS had been slowly developing problems ever since I bought it about 4 years ago .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"performed\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"My last laptop was a 17 '' ASUS gaming machine , which performed admirably , but having since built my own desktop and really settling into the college life , I found myself wanting something smaller and less cumbersome , not to mention that the ASUS had been slowly developing problems ever since I bought it about 4 years ago .\", \"aspect_start\": 55, \"aspect_end\": 64, \"evidence_start\": 0, \"evidence_end\": 329}], \"review_id\": \"gold_rev_0169\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"My wife was so excited to open the box , but quickly came to see that it did not function as it should .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"function\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"My wife was so excited to open the box , but quickly came to see that it did not function as it should .\", \"aspect_start\": 81, \"aspect_end\": 89, \"evidence_start\": 0, \"evidence_end\": 104}], \"review_id\": \"gold_rev_0170\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"No startup disk was not included but that may be my fault .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"startup disk\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"No startup disk was not included but that may be my fault .\", \"aspect_start\": 3, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 59}], \"review_id\": \"gold_rev_0171\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Not as fast as I would have expect for an i5 .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"i5\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Not as fast as I would have expect for an i5 .\", \"aspect_start\": 42, \"aspect_end\": 44, \"evidence_start\": 0, \"evidence_end\": 46}], \"review_id\": \"gold_rev_0172\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Not sure on Windows 8 .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Windows 8\", \"category\": \"SOFTWARE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"Not sure on Windows 8 .\", \"aspect_start\": 12, \"aspect_end\": 21, \"evidence_start\": 0, \"evidence_end\": 23}], \"review_id\": \"gold_rev_0173\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Only problem that I had was that the track pad was not very good for me , I only had a problem once or twice with it , But probably my computer was a bit defective .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"track pad\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Only problem that I had was that the track pad was not very good for me , I only had a problem once or twice with it , But probably my computer was a bit defective .\", \"aspect_start\": 37, \"aspect_end\": 46, \"evidence_start\": 0, \"evidence_end\": 165}], \"review_id\": \"gold_rev_0174\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Put a cover on it and is a little better but that is my only complaint .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"cover\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"Put a cover on it and is a little better but that is my only complaint .\", \"aspect_start\": 6, \"aspect_end\": 11, \"evidence_start\": 0, \"evidence_end\": 72}], \"review_id\": \"gold_rev_0175\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Runs good and does the job , ca n't complain about that !\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Runs\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Runs good and does the job , ca n't complain about that !\", \"aspect_start\": 0, \"aspect_end\": 4, \"evidence_start\": 0, \"evidence_end\": 57}], \"review_id\": \"gold_rev_0176\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Sure it 's not light and slim but the features make up for it 100 % .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"features\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Sure it 's not light and slim but the features make up for it 100 % .\", \"aspect_start\": 38, \"aspect_end\": 46, \"evidence_start\": 0, \"evidence_end\": 69}], \"review_id\": \"gold_rev_0177\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The Apple engineers have not yet discovered the delete key .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"delete key\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The Apple engineers have not yet discovered the delete key .\", \"aspect_start\": 48, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 60}], \"review_id\": \"gold_rev_0178\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The Mini 's body has n't changed since late 2010 - and for a good reason .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"body\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"The Mini 's body has n't changed since late 2010 - and for a good reason .\", \"aspect_start\": 12, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 74}], \"review_id\": \"gold_rev_0179\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The battery is not as shown in the product photos .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"battery\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The battery is not as shown in the product photos .\", \"aspect_start\": 4, \"aspect_end\": 11, \"evidence_start\": 0, \"evidence_end\": 51}], \"review_id\": \"gold_rev_0180\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The memory was gone and it was not able to be used .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"memory\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The memory was gone and it was not able to be used .\", \"aspect_start\": 4, \"aspect_end\": 10, \"evidence_start\": 0, \"evidence_end\": 52}], \"review_id\": \"gold_rev_0181\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The new MBP offers great portability and gives us confidence that we are not going to need to purchase a new laptop in 18 months .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"portability\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"The new MBP offers great portability and gives us confidence that we are not going to need to purchase a new laptop in 18 months .\", \"aspect_start\": 25, \"aspect_end\": 36, \"evidence_start\": 0, \"evidence_end\": 130}], \"review_id\": \"gold_rev_0182\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The only task that this computer would not be good enough for would be gaming , otherwise the integrated Intel 4000 graphics work well for other tasks .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"gaming\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The only task that this computer would not be good enough for would be gaming , otherwise the integrated Intel 4000 graphics work well for other tasks .\", \"aspect_start\": 71, \"aspect_end\": 77, \"evidence_start\": 0, \"evidence_end\": 152}], \"review_id\": \"gold_rev_0183\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The settings are not user-friendly either .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"settings\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The settings are not user-friendly either .\", \"aspect_start\": 4, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 43}], \"review_id\": \"gold_rev_0184\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The sound as mentioned earlier is n't the best , but it can be solved with headphones .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"headphones\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"The sound as mentioned earlier is n't the best , but it can be solved with headphones .\", \"aspect_start\": 75, \"aspect_end\": 85, \"evidence_start\": 0, \"evidence_end\": 87}], \"review_id\": \"gold_rev_0185\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The sound is nice and loud ; I do n't have any problems with hearing anything .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"The sound is nice and loud ; I do n't have any problems with hearing anything .\", \"aspect_start\": 4, \"aspect_end\": 9, \"evidence_start\": 0, \"evidence_end\": 79}], \"review_id\": \"gold_rev_0186\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"The technical support was not helpful as well .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"technical support\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"The technical support was not helpful as well .\", \"aspect_start\": 4, \"aspect_end\": 21, \"evidence_start\": 0, \"evidence_end\": 47}], \"review_id\": \"gold_rev_0187\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"This hardware seems to be better than the iMac in that it is n't $ 1400 and smaller .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"hardware\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"This hardware seems to be better than the iMac in that it is n't $ 1400 and smaller .\", \"aspect_start\": 5, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 85}], \"review_id\": \"gold_rev_0188\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"This version of MacBook Pro runs on a third-generation CPU (\\\"Ivy Bridge\\\") , not the latest fourth-generation Haswell CPU the 2013 version has .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"third-generation CPU (\\\"Ivy Bridge\\\")\", \"category\": \"PERFORMANCE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"This version of MacBook Pro runs on a third-generation CPU (\\\"Ivy Bridge\\\") , not the latest fourth-generation Haswell CPU the 2013 version has .\", \"aspect_start\": 38, \"aspect_end\": 73, \"evidence_start\": 0, \"evidence_end\": 143}], \"review_id\": \"gold_rev_0189\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"This version of MacBook Pro runs on a third-generation CPU -LRB- `` Ivy Bridge '' -RRB- , not the latest fourth-generation Haswell CPU the 2013 version has .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"fourth-generation Haswell CPU\", \"category\": \"PERFORMANCE\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"This version of MacBook Pro runs on a third-generation CPU -LRB- `` Ivy Bridge '' -RRB- , not the latest fourth-generation Haswell CPU the 2013 version has .\", \"aspect_start\": 105, \"aspect_end\": 134, \"evidence_start\": 0, \"evidence_end\": 157}], \"review_id\": \"gold_rev_0190\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Though please note that sometimes it crashes , and the sound quality isnt superb .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Though please note that sometimes it crashes , and the sound quality isnt superb .\", \"aspect_start\": 55, \"aspect_end\": 68, \"evidence_start\": 0, \"evidence_end\": 82}], \"review_id\": \"gold_rev_0191\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Was n't sure if I was going to like it much less love it so I went to a local best buy and played around with the IOS system on a Mac Pro and it was totally unique and different .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"IOS system\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Was n't sure if I was going to like it much less love it so I went to a local best buy and played around with the IOS system on a Mac Pro and it was totally unique and different .\", \"aspect_start\": 114, \"aspect_end\": 124, \"evidence_start\": 0, \"evidence_end\": 179}], \"review_id\": \"gold_rev_0192\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Which it did not have , only 3 USB 2 ports .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"USB 2 ports\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"Which it did not have , only 3 USB 2 ports .\", \"aspect_start\": 31, \"aspect_end\": 42, \"evidence_start\": 0, \"evidence_end\": 44}], \"review_id\": \"gold_rev_0193\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Yes , that 's a good thing , but it 's made from aluminum that scratches easily .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"aluminum\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"Yes , that 's a good thing , but it 's made from aluminum that scratches easily .\", \"aspect_start\": 49, \"aspect_end\": 57, \"evidence_start\": 0, \"evidence_end\": 81}], \"review_id\": \"gold_rev_0194\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"Yet my mac continues to function properly .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"function\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"Yet my mac continues to function properly .\", \"aspect_start\": 24, \"aspect_end\": 32, \"evidence_start\": 0, \"evidence_end\": 43}], \"review_id\": \"gold_rev_0195\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"You just can not beat the functionality of an Apple device .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"functionality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"You just can not beat the functionality of an Apple device .\", \"aspect_start\": 26, \"aspect_end\": 39, \"evidence_start\": 0, \"evidence_end\": 60}], \"review_id\": \"gold_rev_0196\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"and its not hard to accidentally bang it against something so i recommend getting a case to protect it .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"case\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"and its not hard to accidentally bang it against something so i recommend getting a case to protect it .\", \"aspect_start\": 84, \"aspect_end\": 88, \"evidence_start\": 0, \"evidence_end\": 104}], \"review_id\": \"gold_rev_0197\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"its as advertised on here ... it works great and is not a waste of money !\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"works\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"its as advertised on here ... it works great and is not a waste of money !\", \"aspect_start\": 33, \"aspect_end\": 38, \"evidence_start\": 0, \"evidence_end\": 74}], \"review_id\": \"gold_rev_0198\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"tech support would not fix the problem unless I bought your plan for $ 150 plus .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"tech support\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"tech support would not fix the problem unless I bought your plan for $ 150 plus .\", \"aspect_start\": 0, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 81}], \"review_id\": \"gold_rev_0199\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"the hardware problems have been so bad , i ca n't wait till it completely dies in 3 years , TOPS !\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"hardware\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"the hardware problems have been so bad , i ca n't wait till it completely dies in 3 years , TOPS !\", \"aspect_start\": 4, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 98}], \"review_id\": \"gold_rev_0200\"}", "{\"product_id\": \"PROD_ELEC_LAPTOP\", \"text\": \"the latest version does not have a disc drive .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"disc drive\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"the latest version does not have a disc drive .\", \"aspect_start\": 35, \"aspect_end\": 45, \"evidence_start\": 0, \"evidence_end\": 47}], \"review_id\": \"gold_rev_0201\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"- excellent aesthetics , brilliant colours, at par to the other lg/samsung brand tv of middle range. -\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"aesthetics\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"- excellent aesthetics , brilliant colours, at par to the other lg/samsung brand tv of middle range. -\", \"aspect_start\": 12, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 102}], \"review_id\": \"gold_rev_0202\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"- excellent aesthetics, brilliant colours , at par to the other lg/samsung brand tv of middle range. -\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"colours\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"- excellent aesthetics, brilliant colours , at par to the other lg/samsung brand tv of middle range. -\", \"aspect_start\": 34, \"aspect_end\": 41, \"evidence_start\": 0, \"evidence_end\": 102}], \"review_id\": \"gold_rev_0203\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"7 + 7wats sound not enough\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"7 + 7wats sound not enough\", \"aspect_start\": 10, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 26}], \"review_id\": \"gold_rev_0204\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"I am very happy this product, but i have find one default in this vu tv is head phone jack not working properly.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"head phone jack\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"I am very happy this product, but i have find one default in this vu tv is head phone jack not working properly.\", \"aspect_start\": 75, \"aspect_end\": 90, \"evidence_start\": 0, \"evidence_end\": 112}], \"review_id\": \"gold_rev_0205\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"absolutely fine led tv except remote .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"remote\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"absolutely fine led tv except remote .\", \"aspect_start\": 30, \"aspect_end\": 36, \"evidence_start\": 0, \"evidence_end\": 38}], \"review_id\": \"gold_rev_0206\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"absolutely fine led tv except remote.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"led tv\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"absolutely fine led tv except remote.\", \"aspect_start\": 16, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 37}], \"review_id\": \"gold_rev_0207\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"also the installed apps was not opening like Hotstar youtube .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"youtube\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"also the installed apps was not opening like Hotstar youtube .\", \"aspect_start\": 53, \"aspect_end\": 60, \"evidence_start\": 0, \"evidence_end\": 62}], \"review_id\": \"gold_rev_0208\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"although aoc calls it a 'razor tv' with an ultra-thin bezel , the promotional images are slightly misleading.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"bezel\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"although aoc calls it a 'razor tv' with an ultra-thin bezel , the promotional images are slightly misleading.\", \"aspect_start\": 54, \"aspect_end\": 59, \"evidence_start\": 0, \"evidence_end\": 109}], \"review_id\": \"gold_rev_0209\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"bad sound , too bright, okayish picture but best in price.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"bad sound , too bright, okayish picture but best in price.\", \"aspect_start\": 4, \"aspect_end\": 9, \"evidence_start\": 0, \"evidence_end\": 58}], \"review_id\": \"gold_rev_0210\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"bad sound, too bright, okayish picture but best in price .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"price\", \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"bad sound, too bright, okayish picture but best in price .\", \"aspect_start\": 51, \"aspect_end\": 56, \"evidence_start\": 0, \"evidence_end\": 58}], \"review_id\": \"gold_rev_0211\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"bad sound, too bright, okayish picture but best in price.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"bad sound, too bright, okayish picture but best in price.\", \"aspect_start\": 31, \"aspect_end\": 38, \"evidence_start\": 0, \"evidence_end\": 57}], \"review_id\": \"gold_rev_0212\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"best in this range but installation not done timely.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"installation\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"best in this range but installation not done timely.\", \"aspect_start\": 23, \"aspect_end\": 35, \"evidence_start\": 0, \"evidence_end\": 52}], \"review_id\": \"gold_rev_0213\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"better: screen mirroring is better but not the best.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"screen mirroring\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"better: screen mirroring is better but not the best.\", \"aspect_start\": 8, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 52}], \"review_id\": \"gold_rev_0214\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but audio quality should be more better.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"audio quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but audio quality should be more better.\", \"aspect_start\": 4, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 40}], \"review_id\": \"gold_rev_0215\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but demo servive is so poor technician has not come\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"demo servive\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but demo servive is so poor technician has not come\", \"aspect_start\": 4, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 51}], \"review_id\": \"gold_rev_0216\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but for this price definitely good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"price\", \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"but for this price definitely good.\", \"aspect_start\": 13, \"aspect_end\": 18, \"evidence_start\": 0, \"evidence_end\": 35}], \"review_id\": \"gold_rev_0217\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but in pc, mode it have a blur on sceeen .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sceeen\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but in pc, mode it have a blur on sceeen .\", \"aspect_start\": 34, \"aspect_end\": 40, \"evidence_start\": 0, \"evidence_end\": 42}], \"review_id\": \"gold_rev_0218\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but sound is little bit low.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but sound is little bit low.\", \"aspect_start\": 4, \"aspect_end\": 9, \"evidence_start\": 0, \"evidence_end\": 28}], \"review_id\": \"gold_rev_0219\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but sound is some low.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but sound is some low.\", \"aspect_start\": 4, \"aspect_end\": 9, \"evidence_start\": 0, \"evidence_end\": 22}], \"review_id\": \"gold_rev_0220\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but sound problem some time.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but sound problem some time.\", \"aspect_start\": 4, \"aspect_end\": 9, \"evidence_start\": 0, \"evidence_end\": 28}], \"review_id\": \"gold_rev_0221\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but sound quality is average.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"but sound quality is average.\", \"aspect_start\": 4, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 29}], \"review_id\": \"gold_rev_0222\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but sound quality is not good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but sound quality is not good.\", \"aspect_start\": 4, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 30}], \"review_id\": \"gold_rev_0223\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but the audio is not up to the mark especially when u are playing any movies from usb or any flash drive.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"audio\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but the audio is not up to the mark especially when u are playing any movies from usb or any flash drive.\", \"aspect_start\": 8, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 105}], \"review_id\": \"gold_rev_0224\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but the remote of the led tv was out of order.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"remote\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but the remote of the led tv was out of order.\", \"aspect_start\": 8, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 46}], \"review_id\": \"gold_rev_0225\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but the sound quality is not satisfied as expectable.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but the sound quality is not satisfied as expectable.\", \"aspect_start\": 8, \"aspect_end\": 21, \"evidence_start\": 0, \"evidence_end\": 53}], \"review_id\": \"gold_rev_0226\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"but there is an issue with usb port , pendrive is not detecting in it.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"usb port\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"but there is an issue with usb port , pendrive is not detecting in it.\", \"aspect_start\": 27, \"aspect_end\": 35, \"evidence_start\": 0, \"evidence_end\": 70}], \"review_id\": \"gold_rev_0227\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"by the way the music output is also not supported in this product.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"music output\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"by the way the music output is also not supported in this product.\", \"aspect_start\": 15, \"aspect_end\": 27, \"evidence_start\": 0, \"evidence_end\": 66}], \"review_id\": \"gold_rev_0228\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"casting is not working on most of the mobiles.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"casting\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"casting is not working on most of the mobiles.\", \"aspect_start\": 0, \"aspect_end\": 7, \"evidence_start\": 0, \"evidence_end\": 46}], \"review_id\": \"gold_rev_0229\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"clarity is also not good will post details review after 10 days observation.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"clarity\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"clarity is also not good will post details review after 10 days observation.\", \"aspect_start\": 0, \"aspect_end\": 7, \"evidence_start\": 0, \"evidence_end\": 76}], \"review_id\": \"gold_rev_0230\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"colour contrast is not good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"colour contrast\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"colour contrast is not good.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 28}], \"review_id\": \"gold_rev_0231\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"cons ui software of tv not equivalent to the competitors - cant expect for tv which is almost 40% of the cost of competitors sound a little low for my liking.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"ui software\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"cons ui software of tv not equivalent to the competitors - cant expect for tv which is almost 40% of the cost of competitors sound a little low for my liking.\", \"aspect_start\": 5, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 158}], \"review_id\": \"gold_rev_0232\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"demonstration was not good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"demonstration\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"demonstration was not good.\", \"aspect_start\": 0, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 27}], \"review_id\": \"gold_rev_0233\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"don't buy this product 4 k videos not play in this and not download cshare .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"4 k videos\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"don't buy this product 4 k videos not play in this and not download cshare .\", \"aspect_start\": 23, \"aspect_end\": 33, \"evidence_start\": 0, \"evidence_end\": 76}], \"review_id\": \"gold_rev_0234\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"downside is the sound output which is not as clear but acceptable.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound output\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"downside is the sound output which is not as clear but acceptable.\", \"aspect_start\": 16, \"aspect_end\": 28, \"evidence_start\": 0, \"evidence_end\": 66}], \"review_id\": \"gold_rev_0235\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"dth hd channels clarity not up to my satisfaction( we have Airtel dth).\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"hd channels clarity\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"dth hd channels clarity not up to my satisfaction( we have Airtel dth).\", \"aspect_start\": 4, \"aspect_end\": 23, \"evidence_start\": 0, \"evidence_end\": 71}], \"review_id\": \"gold_rev_0236\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"everything else is compromisable with little slower micro controller and not so good menu interface with few customisation and in my opinion you probably wont require many customisation on this tv with naturally good picture and sound .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"everything else is compromisable with little slower micro controller and not so good menu interface with few customisation and in my opinion you probably wont require many customisation on this tv with naturally good picture and sound .\", \"aspect_start\": 229, \"aspect_end\": 234, \"evidence_start\": 0, \"evidence_end\": 236}], \"review_id\": \"gold_rev_0237\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"everything is good except Dolby Atmos is not present in it\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Dolby Atmos\", \"category\": \"SOFTWARE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"everything is good except Dolby Atmos is not present in it\", \"aspect_start\": 26, \"aspect_end\": 37, \"evidence_start\": 0, \"evidence_end\": 58}], \"review_id\": \"gold_rev_0238\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"except sound everything is good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"except sound everything is good.\", \"aspect_start\": 7, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 32}], \"review_id\": \"gold_rev_0239\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"getting some backgound noise when connected to speakers ( even in mute and not playing anything).\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"speakers\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"getting some backgound noise when connected to speakers ( even in mute and not playing anything).\", \"aspect_start\": 47, \"aspect_end\": 55, \"evidence_start\": 0, \"evidence_end\": 97}], \"review_id\": \"gold_rev_0240\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"good but live video in Miracast not clear pictures\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Miracast\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"good but live video in Miracast not clear pictures\", \"aspect_start\": 23, \"aspect_end\": 31, \"evidence_start\": 0, \"evidence_end\": 50}], \"review_id\": \"gold_rev_0241\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"good product for the price point, but the speaker volume is a bit low compared to others.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"speaker volume\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"good product for the price point, but the speaker volume is a bit low compared to others.\", \"aspect_start\": 42, \"aspect_end\": 56, \"evidence_start\": 0, \"evidence_end\": 89}], \"review_id\": \"gold_rev_0242\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"good quality, light weight design , it could have 2 usb port but still worth it.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"design\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"good quality, light weight design , it could have 2 usb port but still worth it.\", \"aspect_start\": 27, \"aspect_end\": 33, \"evidence_start\": 0, \"evidence_end\": 80}], \"review_id\": \"gold_rev_0243\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"however chinese tv makers should learn a thing or two about indian audiences, that is, \\\"baby ko bass pasand hai\\\".\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"bass\", \"category\": \"SOUND\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"however chinese tv makers should learn a thing or two about indian audiences, that is, \\\"baby ko bass pasand hai\\\".\", \"aspect_start\": 96, \"aspect_end\": 100, \"evidence_start\": 0, \"evidence_end\": 113}], \"review_id\": \"gold_rev_0244\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"however in this product there is still some scope for improvement in sound quality .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"however in this product there is still some scope for improvement in sound quality .\", \"aspect_start\": 69, \"aspect_end\": 82, \"evidence_start\": 0, \"evidence_end\": 84}], \"review_id\": \"gold_rev_0245\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"however, the delivery time was too long.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"delivery time\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"however, the delivery time was too long.\", \"aspect_start\": 13, \"aspect_end\": 26, \"evidence_start\": 0, \"evidence_end\": 40}], \"review_id\": \"gold_rev_0246\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"i am not happy with delivery team .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"delivery team\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"i am not happy with delivery team .\", \"aspect_start\": 20, \"aspect_end\": 33, \"evidence_start\": 0, \"evidence_end\": 35}], \"review_id\": \"gold_rev_0247\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"i would have given 5 stars but there are some areas of improvement like the processing speed , which i am sure can be addressed by an update.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"processing speed\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"i would have given 5 stars but there are some areas of improvement like the processing speed , which i am sure can be addressed by an update.\", \"aspect_start\": 76, \"aspect_end\": 92, \"evidence_start\": 0, \"evidence_end\": 141}], \"review_id\": \"gold_rev_0248\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"installation not done on time as per their commitment.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"installation\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"installation not done on time as per their commitment.\", \"aspect_start\": 0, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 54}], \"review_id\": \"gold_rev_0249\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"its good, but without remote we cant start tv,, and remote is very delicate and its costs more than 1500.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"remote\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"its good, but without remote we cant start tv,, and remote is very delicate and its costs more than 1500.\", \"aspect_start\": 22, \"aspect_end\": 28, \"evidence_start\": 0, \"evidence_end\": 105}], \"review_id\": \"gold_rev_0250\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"low sound but fit for home.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"low sound but fit for home.\", \"aspect_start\": 4, \"aspect_end\": 9, \"evidence_start\": 0, \"evidence_end\": 27}], \"review_id\": \"gold_rev_0251\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"mirror cast not possible with Hotstar and Tata sky app with android phone.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"mirror cast\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"mirror cast not possible with Hotstar and Tata sky app with android phone.\", \"aspect_start\": 0, \"aspect_end\": 11, \"evidence_start\": 0, \"evidence_end\": 74}], \"review_id\": \"gold_rev_0252\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"mobile mirror option is not available but fast cast app is enough to enjoy d mobile content on tv.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"mobile mirror option\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"mobile mirror option is not available but fast cast app is enough to enjoy d mobile content on tv.\", \"aspect_start\": 0, \"aspect_end\": 20, \"evidence_start\": 0, \"evidence_end\": 98}], \"review_id\": \"gold_rev_0253\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"my television was not installed as scheduled by Jeeves.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"installed\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"my television was not installed as scheduled by Jeeves.\", \"aspect_start\": 22, \"aspect_end\": 31, \"evidence_start\": 0, \"evidence_end\": 55}], \"review_id\": \"gold_rev_0254\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"nice product but wall mount stand should be more good and phone can't connect to tv by using usb wire of phone .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"phone\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"nice product but wall mount stand should be more good and phone can't connect to tv by using usb wire of phone .\", \"aspect_start\": 58, \"aspect_end\": 63, \"evidence_start\": 0, \"evidence_end\": 112}], \"review_id\": \"gold_rev_0255\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"nice product but wall mount stand should be more good and phone can't connect to tv by using usb wire of phone.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"wall mount stand\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"nice product but wall mount stand should be more good and phone can't connect to tv by using usb wire of phone.\", \"aspect_start\": 17, \"aspect_end\": 33, \"evidence_start\": 0, \"evidence_end\": 111}], \"review_id\": \"gold_rev_0256\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"nice product in cost but do not have audio output .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"audio output\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"nice product in cost but do not have audio output .\", \"aspect_start\": 37, \"aspect_end\": 49, \"evidence_start\": 0, \"evidence_end\": 51}], \"review_id\": \"gold_rev_0257\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"nice product in low price ,but delivery is to delay picture quality is good and sound also.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"price\", \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"nice product in low price ,but delivery is to delay picture quality is good and sound also.\", \"aspect_start\": 20, \"aspect_end\": 25, \"evidence_start\": 0, \"evidence_end\": 91}], \"review_id\": \"gold_rev_0258\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"nice tv but connetivity is very poor more lags while playing video and gaming when mirroring screen.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"connetivity\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"nice tv but connetivity is very poor more lags while playing video and gaming when mirroring screen.\", \"aspect_start\": 12, \"aspect_end\": 23, \"evidence_start\": 0, \"evidence_end\": 100}], \"review_id\": \"gold_rev_0259\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"normal cable connection's picture quality is not up to the mark.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"normal cable connection's picture quality is not up to the mark.\", \"aspect_start\": 26, \"aspect_end\": 41, \"evidence_start\": 0, \"evidence_end\": 64}], \"review_id\": \"gold_rev_0260\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"not good poor sound quality .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"not good poor sound quality .\", \"aspect_start\": 14, \"aspect_end\": 27, \"evidence_start\": 0, \"evidence_end\": 29}], \"review_id\": \"gold_rev_0261\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"not satisfied with sound quality .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"not satisfied with sound quality .\", \"aspect_start\": 19, \"aspect_end\": 32, \"evidence_start\": 0, \"evidence_end\": 34}], \"review_id\": \"gold_rev_0262\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"ok but wall mount fitting back side need black rubber bushes two nos but bushes i not received.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"wall mount\", \"category\": \"OTHER\", \"sentiment\": \"neutral\", \"confidence\": 1.0, \"evidence\": \"ok but wall mount fitting back side need black rubber bushes two nos but bushes i not received.\", \"aspect_start\": 7, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 95}], \"review_id\": \"gold_rev_0263\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"only drawback i see is Netflix support but vu can't do anything about it since it is between Google and Netflix.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Netflix support\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"only drawback i see is Netflix support but vu can't do anything about it since it is between Google and Netflix.\", \"aspect_start\": 23, \"aspect_end\": 38, \"evidence_start\": 0, \"evidence_end\": 112}], \"review_id\": \"gold_rev_0264\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"only one thing is not good pandrive usb port not in corner because of this its quite difficult to insert pen drive when tv is wall mouned.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"pandrive usb port\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"only one thing is not good pandrive usb port not in corner because of this its quite difficult to insert pen drive when tv is wall mouned.\", \"aspect_start\": 27, \"aspect_end\": 44, \"evidence_start\": 0, \"evidence_end\": 138}], \"review_id\": \"gold_rev_0265\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"only sound is not up to the mark when compared to other led tv, but it is enough if you are using it for home.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"only sound is not up to the mark when compared to other led tv, but it is enough if you are using it for home.\", \"aspect_start\": 5, \"aspect_end\": 10, \"evidence_start\": 0, \"evidence_end\": 110}], \"review_id\": \"gold_rev_0266\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"packing is not good as per long distance traveling.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"packing\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"packing is not good as per long distance traveling.\", \"aspect_start\": 0, \"aspect_end\": 7, \"evidence_start\": 0, \"evidence_end\": 51}], \"review_id\": \"gold_rev_0267\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality is good but every time select which you want mode last seen memory not available is the negitive and also no indicater on running condition.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"picture quality is good but every time select which you want mode last seen memory not available is the negitive and also no indicater on running condition.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 156}], \"review_id\": \"gold_rev_0268\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality is not good as well as hd ready .\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"hd ready\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture quality is not good as well as hd ready .\", \"aspect_start\": 39, \"aspect_end\": 47, \"evidence_start\": 0, \"evidence_end\": 49}], \"review_id\": \"gold_rev_0269\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality is not good as well as hd ready.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture quality is not good as well as hd ready.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 48}], \"review_id\": \"gold_rev_0270\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality is not good even in Tata sky hd.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture quality is not good even in Tata sky hd.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 48}], \"review_id\": \"gold_rev_0271\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality is not good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture quality is not good.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 28}], \"review_id\": \"gold_rev_0272\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"picture quality not so good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"picture quality not so good.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 28}], \"review_id\": \"gold_rev_0273\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"product is good but installation done by Flipkart was pathetic.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"installation\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"product is good but installation done by Flipkart was pathetic.\", \"aspect_start\": 20, \"aspect_end\": 32, \"evidence_start\": 0, \"evidence_end\": 63}], \"review_id\": \"gold_rev_0274\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"remote button hard and not very smooth to operate.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"remote button\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"remote button hard and not very smooth to operate.\", \"aspect_start\": 0, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 50}], \"review_id\": \"gold_rev_0275\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"screen mirror not working for moto g5s plus mobile apart incredible 4k smart tv in this range.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"screen mirror\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"screen mirror not working for moto g5s plus mobile apart incredible 4k smart tv in this range.\", \"aspect_start\": 0, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 94}], \"review_id\": \"gold_rev_0276\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"screen mirroring not that good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"screen mirroring\", \"category\": \"DISPLAY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"screen mirroring not that good.\", \"aspect_start\": 0, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 31}], \"review_id\": \"gold_rev_0277\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"since the wall mount provided by sony is not that much good and comfortable.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"wall mount\", \"category\": \"OTHER\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"since the wall mount provided by sony is not that much good and comfortable.\", \"aspect_start\": 10, \"aspect_end\": 20, \"evidence_start\": 0, \"evidence_end\": 76}], \"review_id\": \"gold_rev_0278\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"so many buttons in the remote is not working properly.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"buttons in the remote\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"so many buttons in the remote is not working properly.\", \"aspect_start\": 8, \"aspect_end\": 29, \"evidence_start\": 0, \"evidence_end\": 54}], \"review_id\": \"gold_rev_0279\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"solid product par performance no regret sound is crystal clear if settings done in optimum way.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"solid product par performance no regret sound is crystal clear if settings done in optimum way.\", \"aspect_start\": 40, \"aspect_end\": 45, \"evidence_start\": 0, \"evidence_end\": 95}], \"review_id\": \"gold_rev_0280\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"sony clarity and sound is unmatched - i have a samsung and a Panasonic but this takes the cake!.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 1.0, \"evidence\": \"sony clarity and sound is unmatched - i have a samsung and a Panasonic but this takes the cake!.\", \"aspect_start\": 17, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 96}], \"review_id\": \"gold_rev_0281\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"sound is poor but its ok for small room.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sound\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"sound is poor but its ok for small room.\", \"aspect_start\": 0, \"aspect_end\": 5, \"evidence_start\": 0, \"evidence_end\": 40}], \"review_id\": \"gold_rev_0282\"}", "{\"product_id\": \"PROD_ELEC_TV\", \"text\": \"sound quality is not good.\", \"vertical\": \"electronics_tech\", \"language\": \"en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"sound quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 1.0, \"evidence\": \"sound quality is not good.\", \"aspect_start\": 0, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 26}], \"review_id\": \"gold_rev_0283\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Phone ka display bohot crisp hai lekin battery ek din bhi nahi chalti.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"display\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Phone ka display bohot crisp hai lekin battery ek din bhi nahi chalti.\", \"aspect_start\": 9, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 70}, {\"aspect\": \"battery\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Phone ka display bohot crisp hai lekin battery ek din bhi nahi chalti.\", \"aspect_start\": 39, \"aspect_end\": 46, \"evidence_start\": 0, \"evidence_end\": 70}], \"review_id\": \"gold_rev_0284\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Sound quality bilkul cinema hall jaisi hai, par bluetooth connect hone me time leta hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Sound quality\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Sound quality bilkul cinema hall jaisi hai, par bluetooth connect hone me time leta hai.\", \"aspect_start\": 0, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 88}, {\"aspect\": \"bluetooth\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Sound quality bilkul cinema hall jaisi hai, par bluetooth connect hone me time leta hai.\", \"aspect_start\": 48, \"aspect_end\": 57, \"evidence_start\": 0, \"evidence_end\": 88}], \"review_id\": \"gold_rev_0285\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Camera quality acchi hai par low light me bilkul bekaar photos aati hain.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Camera quality\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Camera quality acchi hai par low light me bilkul bekaar photos aati hain.\", \"aspect_start\": 0, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 73}, {\"aspect\": \"low light\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Camera quality acchi hai par low light me bilkul bekaar photos aati hain.\", \"aspect_start\": 29, \"aspect_end\": 38, \"evidence_start\": 0, \"evidence_end\": 73}], \"review_id\": \"gold_rev_0286\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Processor bohot fast hai, heavy games me bhi bilkul lag nahi karta.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Processor\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Processor bohot fast hai, heavy games me bhi bilkul lag nahi karta.\", \"aspect_start\": 0, \"aspect_end\": 9, \"evidence_start\": 0, \"evidence_end\": 67}], \"review_id\": \"gold_rev_0287\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Fast charging support nahi karta charger, bohot slow charge hota hai phone.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"charger\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Fast charging support nahi karta charger, bohot slow charge hota hai phone.\", \"aspect_start\": 33, \"aspect_end\": 40, \"evidence_start\": 0, \"evidence_end\": 75}], \"review_id\": \"gold_rev_0288\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Waah kya phone banaya hai, call aate hi speaker fatne lagta hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"sarcasm\", \"quads\": [{\"aspect\": \"speaker\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Waah kya phone banaya hai, call aate hi speaker fatne lagta hai.\", \"aspect_start\": 40, \"aspect_end\": 47, \"evidence_start\": 0, \"evidence_end\": 64}], \"review_id\": \"gold_rev_0289\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Paisa vasool mobile hai bhai, is range me aisi screen aur speed milna impossible hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"implicit_aspect\", \"quads\": [{\"aspect\": null, \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Paisa vasool mobile hai bhai, is range me aisi screen aur speed milna impossible hai.\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 0, \"evidence_end\": 85}], \"review_id\": \"gold_rev_0290\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Touch screen smooth chal raha hai par fingerprint sensor kabhi kabhi fail ho jata hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Touch screen\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Touch screen smooth chal raha hai par fingerprint sensor kabhi kabhi fail ho jata hai.\", \"aspect_start\": 0, \"aspect_end\": 12, \"evidence_start\": 0, \"evidence_end\": 86}, {\"aspect\": \"fingerprint sensor\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Touch screen smooth chal raha hai par fingerprint sensor kabhi kabhi fail ho jata hai.\", \"aspect_start\": 38, \"aspect_end\": 56, \"evidence_start\": 0, \"evidence_end\": 86}], \"review_id\": \"gold_rev_0291\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Laptop ka build solid hai metal body ke sath lekin heating problem bohot zyada hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"build\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Laptop ka build solid hai metal body ke sath lekin heating problem bohot zyada hai.\", \"aspect_start\": 10, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 83}, {\"aspect\": \"heating\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Laptop ka build solid hai metal body ke sath lekin heating problem bohot zyada hai.\", \"aspect_start\": 51, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 83}], \"review_id\": \"gold_rev_0292\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Keyboard typing experience shaandar hai par trackpad buttons bohot stiff hain.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Keyboard\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Keyboard typing experience shaandar hai par trackpad buttons bohot stiff hain.\", \"aspect_start\": 0, \"aspect_end\": 8, \"evidence_start\": 0, \"evidence_end\": 78}, {\"aspect\": \"trackpad\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Keyboard typing experience shaandar hai par trackpad buttons bohot stiff hain.\", \"aspect_start\": 44, \"aspect_end\": 52, \"evidence_start\": 0, \"evidence_end\": 78}], \"review_id\": \"gold_rev_0293\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Earphones ka bass dhamakedar hai lekin mic quality calls ke liye acchi nahi hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"bass\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Earphones ka bass dhamakedar hai lekin mic quality calls ke liye acchi nahi hai.\", \"aspect_start\": 13, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 80}, {\"aspect\": \"mic quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Earphones ka bass dhamakedar hai lekin mic quality calls ke liye acchi nahi hai.\", \"aspect_start\": 39, \"aspect_end\": 50, \"evidence_start\": 0, \"evidence_end\": 80}], \"review_id\": \"gold_rev_0294\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Wi-Fi reception drop hota rehta hai room me, router ke paas jana padta hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"Wi-Fi reception\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Wi-Fi reception drop hota rehta hai room me, router ke paas jana padta hai.\", \"aspect_start\": 0, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 75}], \"review_id\": \"gold_rev_0295\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"TV ki picture quality behtareen hai par remote control responsive nahi hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"TV ki picture quality behtareen hai par remote control responsive nahi hai.\", \"aspect_start\": 6, \"aspect_end\": 21, \"evidence_start\": 0, \"evidence_end\": 75}, {\"aspect\": \"remote control\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"TV ki picture quality behtareen hai par remote control responsive nahi hai.\", \"aspect_start\": 40, \"aspect_end\": 54, \"evidence_start\": 0, \"evidence_end\": 75}], \"review_id\": \"gold_rev_0296\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Smart features smoothly work karte hain par delivery team ne installation do hafte baad ki.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Smart features\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Smart features smoothly work karte hain par delivery team ne installation do hafte baad ki.\", \"aspect_start\": 0, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 91}, {\"aspect\": \"installation\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Smart features smoothly work karte hain par delivery team ne installation do hafte baad ki.\", \"aspect_start\": 61, \"aspect_end\": 73, \"evidence_start\": 0, \"evidence_end\": 91}], \"review_id\": \"gold_rev_0297\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"SSD speed se system 5 second me boot hota hai, performance top notch hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"SSD speed\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"SSD speed se system 5 second me boot hota hai, performance top notch hai.\", \"aspect_start\": 0, \"aspect_end\": 9, \"evidence_start\": 0, \"evidence_end\": 73}, {\"aspect\": \"performance\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"SSD speed se system 5 second me boot hota hai, performance top notch hai.\", \"aspect_start\": 47, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 73}], \"review_id\": \"gold_rev_0298\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Fabric bohot soft aur breathable cotton hai lekin fitting thodi loose aayi hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Fabric bohot soft aur breathable cotton hai lekin fitting thodi loose aayi hai.\", \"aspect_start\": 0, \"aspect_end\": 6, \"evidence_start\": 0, \"evidence_end\": 79}, {\"aspect\": \"fitting\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Fabric bohot soft aur breathable cotton hai lekin fitting thodi loose aayi hai.\", \"aspect_start\": 50, \"aspect_end\": 57, \"evidence_start\": 0, \"evidence_end\": 79}], \"review_id\": \"gold_rev_0299\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Color wash ke baad fade nahi hota par stitching sides se nikalne lagi hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Color wash ke baad fade nahi hota par stitching sides se nikalne lagi hai.\", \"aspect_start\": 0, \"aspect_end\": 5, \"evidence_start\": 0, \"evidence_end\": 74}, {\"aspect\": \"stitching\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Color wash ke baad fade nahi hota par stitching sides se nikalne lagi hai.\", \"aspect_start\": 38, \"aspect_end\": 47, \"evidence_start\": 0, \"evidence_end\": 74}], \"review_id\": \"gold_rev_0300\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Kurta ka print ekdum vibrant hai par material synthetic lagta hai jo garmi me chipakta hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"print\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Kurta ka print ekdum vibrant hai par material synthetic lagta hai jo garmi me chipakta hai.\", \"aspect_start\": 9, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 91}, {\"aspect\": \"material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Kurta ka print ekdum vibrant hai par material synthetic lagta hai jo garmi me chipakta hai.\", \"aspect_start\": 37, \"aspect_end\": 45, \"evidence_start\": 0, \"evidence_end\": 91}], \"review_id\": \"gold_rev_0301\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Sizing bilkul chart ke according accurate hai, perfect fit mila mujhe.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"Sizing\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Sizing bilkul chart ke according accurate hai, perfect fit mila mujhe.\", \"aspect_start\": 0, \"aspect_end\": 6, \"evidence_start\": 0, \"evidence_end\": 70}, {\"aspect\": \"fit\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Sizing bilkul chart ke according accurate hai, perfect fit mila mujhe.\", \"aspect_start\": 55, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 70}], \"review_id\": \"gold_rev_0302\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Shoes ka sole comfortable aur cushioned hai par upper mesh bohot jaldi phat gaya.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sole\", \"category\": \"COMFORT\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Shoes ka sole comfortable aur cushioned hai par upper mesh bohot jaldi phat gaya.\", \"aspect_start\": 9, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 81}, {\"aspect\": \"upper mesh\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Shoes ka sole comfortable aur cushioned hai par upper mesh bohot jaldi phat gaya.\", \"aspect_start\": 48, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 81}], \"review_id\": \"gold_rev_0303\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Waah Flipkart, pure cotton likh kar polyester thama diya.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"sarcasm\", \"quads\": [{\"aspect\": \"cotton\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Waah Flipkart, pure cotton likh kar polyester thama diya.\", \"aspect_start\": 20, \"aspect_end\": 26, \"evidence_start\": 0, \"evidence_end\": 57}], \"review_id\": \"gold_rev_0304\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Paisa vasool t-shirt hai, roz marra ke use ke liye sabse best.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"implicit_aspect\", \"quads\": [{\"aspect\": null, \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Paisa vasool t-shirt hai, roz marra ke use ke liye sabse best.\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 0, \"evidence_end\": 62}], \"review_id\": \"gold_rev_0305\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Jeans ka denim heavy aur durable hai lekin zip quality bilkul low grade hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"denim\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Jeans ka denim heavy aur durable hai lekin zip quality bilkul low grade hai.\", \"aspect_start\": 9, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 76}, {\"aspect\": \"zip\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Jeans ka denim heavy aur durable hai lekin zip quality bilkul low grade hai.\", \"aspect_start\": 43, \"aspect_end\": 46, \"evidence_start\": 0, \"evidence_end\": 76}], \"review_id\": \"gold_rev_0306\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Collar stitching tight aur clean hai par color wash hone par dull ho gaya.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Collar stitching\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Collar stitching tight aur clean hai par color wash hone par dull ho gaya.\", \"aspect_start\": 0, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 74}, {\"aspect\": \"color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Collar stitching tight aur clean hai par color wash hone par dull ho gaya.\", \"aspect_start\": 41, \"aspect_end\": 46, \"evidence_start\": 0, \"evidence_end\": 74}], \"review_id\": \"gold_rev_0307\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Kapde ka texture premium hai par delivery me pure 10 din lag gaye.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"texture\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Kapde ka texture premium hai par delivery me pure 10 din lag gaye.\", \"aspect_start\": 9, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 66}, {\"aspect\": \"delivery\", \"category\": \"DELIVERY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Kapde ka texture premium hai par delivery me pure 10 din lag gaye.\", \"aspect_start\": 33, \"aspect_end\": 41, \"evidence_start\": 0, \"evidence_end\": 66}], \"review_id\": \"gold_rev_0308\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Fitting bilkul sahi hai par washing ke baad kapda shrink ho gaya.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Fitting\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Fitting bilkul sahi hai par washing ke baad kapda shrink ho gaya.\", \"aspect_start\": 0, \"aspect_end\": 7, \"evidence_start\": 0, \"evidence_end\": 65}, {\"aspect\": \"kapda\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Fitting bilkul sahi hai par washing ke baad kapda shrink ho gaya.\", \"aspect_start\": 44, \"aspect_end\": 49, \"evidence_start\": 0, \"evidence_end\": 65}], \"review_id\": \"gold_rev_0309\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"T-shirt ka print washing ke baad bhi nahi nikla, zabardast quality hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"print\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"T-shirt ka print washing ke baad bhi nahi nikla, zabardast quality hai.\", \"aspect_start\": 11, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 71}, {\"aspect\": \"quality\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"T-shirt ka print washing ke baad bhi nahi nikla, zabardast quality hai.\", \"aspect_start\": 59, \"aspect_end\": 66, \"evidence_start\": 0, \"evidence_end\": 71}], \"review_id\": \"gold_rev_0310\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Pure cotton material hai, garmi ke mausam ke liye bohot hi comfortable.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"cotton material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Pure cotton material hai, garmi ke mausam ke liye bohot hi comfortable.\", \"aspect_start\": 5, \"aspect_end\": 20, \"evidence_start\": 0, \"evidence_end\": 71}], \"review_id\": \"gold_rev_0311\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Color photo se bilkul alag dark shade me aaya hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"Color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Color photo se bilkul alag dark shade me aaya hai.\", \"aspect_start\": 0, \"aspect_end\": 5, \"evidence_start\": 0, \"evidence_end\": 50}], \"review_id\": \"gold_rev_0312\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Stitching me gaps hain aur buttons loose lage hue hain.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"Stitching\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Stitching me gaps hain aur buttons loose lage hue hain.\", \"aspect_start\": 0, \"aspect_end\": 9, \"evidence_start\": 0, \"evidence_end\": 55}, {\"aspect\": \"buttons\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Stitching me gaps hain aur buttons loose lage hue hain.\", \"aspect_start\": 27, \"aspect_end\": 34, \"evidence_start\": 0, \"evidence_end\": 55}], \"review_id\": \"gold_rev_0313\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Phone ka display bohot crisp hai lekin battery ek din bhi nahi chalti.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"display\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Phone ka display bohot crisp hai lekin battery ek din bhi nahi chalti.\", \"aspect_start\": 17, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 78}, {\"aspect\": \"battery\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Phone ka display bohot crisp hai lekin battery ek din bhi nahi chalti.\", \"aspect_start\": 47, \"aspect_end\": 54, \"evidence_start\": 0, \"evidence_end\": 78}], \"review_id\": \"gold_rev_0314\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Sound quality bilkul cinema hall jaisi hai, par bluetooth connect hone me time leta hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Sound quality\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Sound quality bilkul cinema hall jaisi hai, par bluetooth connect hone me time leta hai.\", \"aspect_start\": 8, \"aspect_end\": 21, \"evidence_start\": 0, \"evidence_end\": 96}, {\"aspect\": \"bluetooth\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Sound quality bilkul cinema hall jaisi hai, par bluetooth connect hone me time leta hai.\", \"aspect_start\": 56, \"aspect_end\": 65, \"evidence_start\": 0, \"evidence_end\": 96}], \"review_id\": \"gold_rev_0315\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Camera quality acchi hai par low light me bilkul bekaar photos aati hain.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Camera quality\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Camera quality acchi hai par low light me bilkul bekaar photos aati hain.\", \"aspect_start\": 8, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 81}, {\"aspect\": \"low light\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Camera quality acchi hai par low light me bilkul bekaar photos aati hain.\", \"aspect_start\": 37, \"aspect_end\": 46, \"evidence_start\": 0, \"evidence_end\": 81}], \"review_id\": \"gold_rev_0316\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Processor bohot fast hai, heavy games me bhi bilkul lag nahi karta.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Processor\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Processor bohot fast hai, heavy games me bhi bilkul lag nahi karta.\", \"aspect_start\": 8, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 75}], \"review_id\": \"gold_rev_0317\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Fast charging support nahi karta charger, bohot slow charge hota hai phone.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"charger\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Fast charging support nahi karta charger, bohot slow charge hota hai phone.\", \"aspect_start\": 41, \"aspect_end\": 48, \"evidence_start\": 0, \"evidence_end\": 83}], \"review_id\": \"gold_rev_0318\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Waah kya phone banaya hai, call aate hi speaker fatne lagta hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"sarcasm\", \"quads\": [{\"aspect\": \"speaker\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Waah kya phone banaya hai, call aate hi speaker fatne lagta hai.\", \"aspect_start\": 48, \"aspect_end\": 55, \"evidence_start\": 0, \"evidence_end\": 72}], \"review_id\": \"gold_rev_0319\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Paisa vasool mobile hai bhai, is range me aisi screen aur speed milna impossible hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"implicit_aspect\", \"quads\": [{\"aspect\": null, \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Paisa vasool mobile hai bhai, is range me aisi screen aur speed milna impossible hai.\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 0, \"evidence_end\": 93}], \"review_id\": \"gold_rev_0320\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Touch screen smooth chal raha hai par fingerprint sensor kabhi kabhi fail ho jata hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Touch screen\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Touch screen smooth chal raha hai par fingerprint sensor kabhi kabhi fail ho jata hai.\", \"aspect_start\": 8, \"aspect_end\": 20, \"evidence_start\": 0, \"evidence_end\": 94}, {\"aspect\": \"fingerprint sensor\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Touch screen smooth chal raha hai par fingerprint sensor kabhi kabhi fail ho jata hai.\", \"aspect_start\": 46, \"aspect_end\": 64, \"evidence_start\": 0, \"evidence_end\": 94}], \"review_id\": \"gold_rev_0321\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Laptop ka build solid hai metal body ke sath lekin heating problem bohot zyada hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"build\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Laptop ka build solid hai metal body ke sath lekin heating problem bohot zyada hai.\", \"aspect_start\": 18, \"aspect_end\": 23, \"evidence_start\": 0, \"evidence_end\": 91}, {\"aspect\": \"heating\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Laptop ka build solid hai metal body ke sath lekin heating problem bohot zyada hai.\", \"aspect_start\": 59, \"aspect_end\": 66, \"evidence_start\": 0, \"evidence_end\": 91}], \"review_id\": \"gold_rev_0322\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Keyboard typing experience shaandar hai par trackpad buttons bohot stiff hain.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Keyboard\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Keyboard typing experience shaandar hai par trackpad buttons bohot stiff hain.\", \"aspect_start\": 8, \"aspect_end\": 16, \"evidence_start\": 0, \"evidence_end\": 86}, {\"aspect\": \"trackpad\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Keyboard typing experience shaandar hai par trackpad buttons bohot stiff hain.\", \"aspect_start\": 52, \"aspect_end\": 60, \"evidence_start\": 0, \"evidence_end\": 86}], \"review_id\": \"gold_rev_0323\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Earphones ka bass dhamakedar hai lekin mic quality calls ke liye acchi nahi hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"bass\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Earphones ka bass dhamakedar hai lekin mic quality calls ke liye acchi nahi hai.\", \"aspect_start\": 21, \"aspect_end\": 25, \"evidence_start\": 0, \"evidence_end\": 88}, {\"aspect\": \"mic quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Earphones ka bass dhamakedar hai lekin mic quality calls ke liye acchi nahi hai.\", \"aspect_start\": 47, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 88}], \"review_id\": \"gold_rev_0324\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Wi-Fi reception drop hota rehta hai room me, router ke paas jana padta hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"Wi-Fi reception\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Wi-Fi reception drop hota rehta hai room me, router ke paas jana padta hai.\", \"aspect_start\": 8, \"aspect_end\": 23, \"evidence_start\": 0, \"evidence_end\": 83}], \"review_id\": \"gold_rev_0325\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: TV ki picture quality behtareen hai par remote control responsive nahi hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"picture quality\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: TV ki picture quality behtareen hai par remote control responsive nahi hai.\", \"aspect_start\": 14, \"aspect_end\": 29, \"evidence_start\": 0, \"evidence_end\": 83}, {\"aspect\": \"remote control\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: TV ki picture quality behtareen hai par remote control responsive nahi hai.\", \"aspect_start\": 48, \"aspect_end\": 62, \"evidence_start\": 0, \"evidence_end\": 83}], \"review_id\": \"gold_rev_0326\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: Smart features smoothly work karte hain par delivery team ne installation do hafte baad ki.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Smart features\", \"category\": \"SOFTWARE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Smart features smoothly work karte hain par delivery team ne installation do hafte baad ki.\", \"aspect_start\": 8, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 99}, {\"aspect\": \"installation\", \"category\": \"DELIVERY_INSTALLATION\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Smart features smoothly work karte hain par delivery team ne installation do hafte baad ki.\", \"aspect_start\": 69, \"aspect_end\": 81, \"evidence_start\": 0, \"evidence_end\": 99}], \"review_id\": \"gold_rev_0327\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"Update: SSD speed se system 5 second me boot hota hai, performance top notch hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"SSD speed\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: SSD speed se system 5 second me boot hota hai, performance top notch hai.\", \"aspect_start\": 8, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 81}, {\"aspect\": \"performance\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: SSD speed se system 5 second me boot hota hai, performance top notch hai.\", \"aspect_start\": 55, \"aspect_end\": 66, \"evidence_start\": 0, \"evidence_end\": 81}], \"review_id\": \"gold_rev_0328\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Fabric bohot soft aur breathable cotton hai lekin fitting thodi loose aayi hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Fabric bohot soft aur breathable cotton hai lekin fitting thodi loose aayi hai.\", \"aspect_start\": 8, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 87}, {\"aspect\": \"fitting\", \"category\": \"FIT_SIZING\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Fabric bohot soft aur breathable cotton hai lekin fitting thodi loose aayi hai.\", \"aspect_start\": 58, \"aspect_end\": 65, \"evidence_start\": 0, \"evidence_end\": 87}], \"review_id\": \"gold_rev_0329\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Color wash ke baad fade nahi hota par stitching sides se nikalne lagi hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Color wash ke baad fade nahi hota par stitching sides se nikalne lagi hai.\", \"aspect_start\": 8, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 82}, {\"aspect\": \"stitching\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Color wash ke baad fade nahi hota par stitching sides se nikalne lagi hai.\", \"aspect_start\": 46, \"aspect_end\": 55, \"evidence_start\": 0, \"evidence_end\": 82}], \"review_id\": \"gold_rev_0330\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Kurta ka print ekdum vibrant hai par material synthetic lagta hai jo garmi me chipakta hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"print\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Kurta ka print ekdum vibrant hai par material synthetic lagta hai jo garmi me chipakta hai.\", \"aspect_start\": 17, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 99}, {\"aspect\": \"material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Kurta ka print ekdum vibrant hai par material synthetic lagta hai jo garmi me chipakta hai.\", \"aspect_start\": 45, \"aspect_end\": 53, \"evidence_start\": 0, \"evidence_end\": 99}], \"review_id\": \"gold_rev_0331\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Sizing bilkul chart ke according accurate hai, perfect fit mila mujhe.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"Sizing\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Sizing bilkul chart ke according accurate hai, perfect fit mila mujhe.\", \"aspect_start\": 8, \"aspect_end\": 14, \"evidence_start\": 0, \"evidence_end\": 78}, {\"aspect\": \"fit\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Sizing bilkul chart ke according accurate hai, perfect fit mila mujhe.\", \"aspect_start\": 63, \"aspect_end\": 66, \"evidence_start\": 0, \"evidence_end\": 78}], \"review_id\": \"gold_rev_0332\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Shoes ka sole comfortable aur cushioned hai par upper mesh bohot jaldi phat gaya.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"sole\", \"category\": \"COMFORT\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Shoes ka sole comfortable aur cushioned hai par upper mesh bohot jaldi phat gaya.\", \"aspect_start\": 17, \"aspect_end\": 21, \"evidence_start\": 0, \"evidence_end\": 89}, {\"aspect\": \"upper mesh\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Shoes ka sole comfortable aur cushioned hai par upper mesh bohot jaldi phat gaya.\", \"aspect_start\": 56, \"aspect_end\": 66, \"evidence_start\": 0, \"evidence_end\": 89}], \"review_id\": \"gold_rev_0333\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Waah Flipkart, pure cotton likh kar polyester thama diya.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"sarcasm\", \"quads\": [{\"aspect\": \"cotton\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Waah Flipkart, pure cotton likh kar polyester thama diya.\", \"aspect_start\": 28, \"aspect_end\": 34, \"evidence_start\": 0, \"evidence_end\": 65}], \"review_id\": \"gold_rev_0334\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Paisa vasool t-shirt hai, roz marra ke use ke liye sabse best.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"implicit_aspect\", \"quads\": [{\"aspect\": null, \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Paisa vasool t-shirt hai, roz marra ke use ke liye sabse best.\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 0, \"evidence_end\": 70}], \"review_id\": \"gold_rev_0335\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Jeans ka denim heavy aur durable hai lekin zip quality bilkul low grade hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"denim\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Jeans ka denim heavy aur durable hai lekin zip quality bilkul low grade hai.\", \"aspect_start\": 17, \"aspect_end\": 22, \"evidence_start\": 0, \"evidence_end\": 84}, {\"aspect\": \"zip\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Jeans ka denim heavy aur durable hai lekin zip quality bilkul low grade hai.\", \"aspect_start\": 51, \"aspect_end\": 54, \"evidence_start\": 0, \"evidence_end\": 84}], \"review_id\": \"gold_rev_0336\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Collar stitching tight aur clean hai par color wash hone par dull ho gaya.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Collar stitching\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Collar stitching tight aur clean hai par color wash hone par dull ho gaya.\", \"aspect_start\": 8, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 82}, {\"aspect\": \"color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Collar stitching tight aur clean hai par color wash hone par dull ho gaya.\", \"aspect_start\": 49, \"aspect_end\": 54, \"evidence_start\": 0, \"evidence_end\": 82}], \"review_id\": \"gold_rev_0337\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Kapde ka texture premium hai par delivery me pure 10 din lag gaye.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"texture\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Kapde ka texture premium hai par delivery me pure 10 din lag gaye.\", \"aspect_start\": 17, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 74}, {\"aspect\": \"delivery\", \"category\": \"DELIVERY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Kapde ka texture premium hai par delivery me pure 10 din lag gaye.\", \"aspect_start\": 41, \"aspect_end\": 49, \"evidence_start\": 0, \"evidence_end\": 74}], \"review_id\": \"gold_rev_0338\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Fitting bilkul sahi hai par washing ke baad kapda shrink ho gaya.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Fitting\", \"category\": \"FIT_SIZING\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Fitting bilkul sahi hai par washing ke baad kapda shrink ho gaya.\", \"aspect_start\": 8, \"aspect_end\": 15, \"evidence_start\": 0, \"evidence_end\": 73}, {\"aspect\": \"kapda\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Fitting bilkul sahi hai par washing ke baad kapda shrink ho gaya.\", \"aspect_start\": 52, \"aspect_end\": 57, \"evidence_start\": 0, \"evidence_end\": 73}], \"review_id\": \"gold_rev_0339\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: T-shirt ka print washing ke baad bhi nahi nikla, zabardast quality hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"print\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: T-shirt ka print washing ke baad bhi nahi nikla, zabardast quality hai.\", \"aspect_start\": 19, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 79}, {\"aspect\": \"quality\", \"category\": \"BUILD_QUALITY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: T-shirt ka print washing ke baad bhi nahi nikla, zabardast quality hai.\", \"aspect_start\": 67, \"aspect_end\": 74, \"evidence_start\": 0, \"evidence_end\": 79}], \"review_id\": \"gold_rev_0340\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Pure cotton material hai, garmi ke mausam ke liye bohot hi comfortable.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"cotton material\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Update: Pure cotton material hai, garmi ke mausam ke liye bohot hi comfortable.\", \"aspect_start\": 13, \"aspect_end\": 28, \"evidence_start\": 0, \"evidence_end\": 79}], \"review_id\": \"gold_rev_0341\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Color photo se bilkul alag dark shade me aaya hai.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"Color\", \"category\": \"COLOR_PRINT\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Color photo se bilkul alag dark shade me aaya hai.\", \"aspect_start\": 8, \"aspect_end\": 13, \"evidence_start\": 0, \"evidence_end\": 58}], \"review_id\": \"gold_rev_0342\"}", "{\"product_id\": \"PROD_APP_HIEN\", \"text\": \"Update: Stitching me gaps hain aur buttons loose lage hue hain.\", \"vertical\": \"apparel_fashion\", \"language\": \"hi-en\", \"is_hard_case\": false, \"hard_case_type\": null, \"quads\": [{\"aspect\": \"Stitching\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Stitching me gaps hain aur buttons loose lage hue hain.\", \"aspect_start\": 8, \"aspect_end\": 17, \"evidence_start\": 0, \"evidence_end\": 63}, {\"aspect\": \"buttons\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"Update: Stitching me gaps hain aur buttons loose lage hue hain.\", \"aspect_start\": 35, \"aspect_end\": 42, \"evidence_start\": 0, \"evidence_end\": 63}], \"review_id\": \"gold_rev_0343\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"My review: Phone ka display bohot crisp hai lekin battery ek din bhi nahi chalti.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"display\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"My review: Phone ka display bohot crisp hai lekin battery ek din bhi nahi chalti.\", \"aspect_start\": 20, \"aspect_end\": 27, \"evidence_start\": 0, \"evidence_end\": 81}, {\"aspect\": \"battery\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"My review: Phone ka display bohot crisp hai lekin battery ek din bhi nahi chalti.\", \"aspect_start\": 50, \"aspect_end\": 57, \"evidence_start\": 0, \"evidence_end\": 81}], \"review_id\": \"gold_rev_0344\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"My review: Sound quality bilkul cinema hall jaisi hai, par bluetooth connect hone me time leta hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Sound quality\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"My review: Sound quality bilkul cinema hall jaisi hai, par bluetooth connect hone me time leta hai.\", \"aspect_start\": 11, \"aspect_end\": 24, \"evidence_start\": 0, \"evidence_end\": 99}, {\"aspect\": \"bluetooth\", \"category\": \"CONNECTIVITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"My review: Sound quality bilkul cinema hall jaisi hai, par bluetooth connect hone me time leta hai.\", \"aspect_start\": 59, \"aspect_end\": 68, \"evidence_start\": 0, \"evidence_end\": 99}], \"review_id\": \"gold_rev_0345\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"My review: Camera quality acchi hai par low light me bilkul bekaar photos aati hain.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"cross_clause_contrast\", \"quads\": [{\"aspect\": \"Camera quality\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"My review: Camera quality acchi hai par low light me bilkul bekaar photos aati hain.\", \"aspect_start\": 11, \"aspect_end\": 25, \"evidence_start\": 0, \"evidence_end\": 84}, {\"aspect\": \"low light\", \"category\": \"PERFORMANCE\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"My review: Camera quality acchi hai par low light me bilkul bekaar photos aati hain.\", \"aspect_start\": 40, \"aspect_end\": 49, \"evidence_start\": 0, \"evidence_end\": 84}], \"review_id\": \"gold_rev_0346\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"My review: Processor bohot fast hai, heavy games me bhi bilkul lag nahi karta.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"Processor\", \"category\": \"PERFORMANCE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"My review: Processor bohot fast hai, heavy games me bhi bilkul lag nahi karta.\", \"aspect_start\": 11, \"aspect_end\": 20, \"evidence_start\": 0, \"evidence_end\": 78}], \"review_id\": \"gold_rev_0347\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"My review: Fast charging support nahi karta charger, bohot slow charge hota hai phone.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"negation\", \"quads\": [{\"aspect\": \"charger\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"My review: Fast charging support nahi karta charger, bohot slow charge hota hai phone.\", \"aspect_start\": 44, \"aspect_end\": 51, \"evidence_start\": 0, \"evidence_end\": 86}], \"review_id\": \"gold_rev_0348\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"My review: Waah kya phone banaya hai, call aate hi speaker fatne lagta hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"sarcasm\", \"quads\": [{\"aspect\": \"speaker\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"My review: Waah kya phone banaya hai, call aate hi speaker fatne lagta hai.\", \"aspect_start\": 51, \"aspect_end\": 58, \"evidence_start\": 0, \"evidence_end\": 75}], \"review_id\": \"gold_rev_0349\"}", "{\"product_id\": \"PROD_ELEC_HIEN\", \"text\": \"My review: Paisa vasool mobile hai bhai, is range me aisi screen aur speed milna impossible hai.\", \"vertical\": \"electronics_tech\", \"language\": \"hi-en\", \"is_hard_case\": true, \"hard_case_type\": \"implicit_aspect\", \"quads\": [{\"aspect\": null, \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"My review: Paisa vasool mobile hai bhai, is range me aisi screen aur speed milna impossible hai.\", \"aspect_start\": null, \"aspect_end\": null, \"evidence_start\": 0, \"evidence_end\": 96}], \"review_id\": \"gold_rev_0350\"}"]

gold_file = Path("/kaggle/working/productlens/gold/gold_dataset.jsonl")
with open(gold_file, "w", encoding="utf-8") as f:
    for line in gold_records:
        f.write(line + "\n")

print(f"✅ Staged {len(gold_records)} gold reviews to {gold_file}")

In [ ]:
# [Step 5/5] Initialize vLLM Teacher & Run Gold Eval
import os, json, time

# Keep only valid vLLM env vars
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

from typing import List, Literal, Optional
from pydantic import BaseModel, Field

from vllm import LLM, SamplingParams
try:
    from vllm.sampling_params import StructuredOutputsParams
except ImportError:
    from vllm.model_executor.guided_decoding.guided_fields import StructuredOutputsParams

from productlens.schemas.enums import Sentiment, VerticalDomain
from productlens.schemas.quads import AspectQuad, RawCompactQuad
from productlens.validator.substring import SubstringValidator
from productlens.gold.manager import GoldSetManager
from productlens.evaluation.runner import Stage1Evaluator

# Model configurations
MODEL_NAME = "Qwen/Qwen3.5-9B"
MAX_MODEL_LEN = 2048
MAX_NUM_SEQS = 64  # Capped to 64 to fit Mamba cache on Tesla T4 (prevents max_num_seqs > available cache blocks)
DTYPE = "float16"
QUANTIZATION = "bitsandbytes"
TP_SIZE = 1  # Set to 2 if running on 2x T4 GPUs

print("=" * 72)
print(f"🚀 Launching vLLM Teacher: {MODEL_NAME}")
print(f"   • Tensor Parallel:      {TP_SIZE}")
print(f"   • Quantization:         {QUANTIZATION}")
print(f"   • Dtype:                {DTYPE}")
print(f"   • Max Model Len:        {MAX_MODEL_LEN}")
print(f"   • Max Num Seqs:         {MAX_NUM_SEQS}")
print("=" * 72)

# Define Pydantic extraction schema
CanonicalCategory = Literal[
    "DISPLAY", "BATTERY", "SOUND", "PERFORMANCE", "BUILD_QUALITY",
    "CONNECTIVITY", "SOFTWARE", "DELIVERY_INSTALLATION", "VALUE",
    "FABRIC_MATERIAL", "FIT_SIZING", "COLOR_PRINT", "STITCHING_DURABILITY",
    "COMFORT", "DELIVERY", "OTHER",
]

class TeacherAspectQuad(BaseModel):
    aspect: Optional[str] = Field(None, description="Explicit aspect term or null if implicit")
    category: CanonicalCategory = Field(..., description="Canonical category")
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(..., description="Sentiment polarity")
    confidence: float = Field(1.0, ge=0.0, le=1.0)
    evidence: str = Field(..., description="Verbatim substring evidence from review")

class TeacherExtractionOutput(BaseModel):
    quads: List[TeacherAspectQuad] = Field(default_factory=list)

schema_json = json.dumps(TeacherExtractionOutput.model_json_schema())

# Initialize LLM with top-level language_model_only=True and max_num_seqs=64
llm_kwargs = {
    "model": MODEL_NAME,
    "tensor_parallel_size": TP_SIZE,
    "dtype": DTYPE,
    "quantization": QUANTIZATION,
    "load_format": "bitsandbytes",
    "trust_remote_code": True,
    "max_model_len": MAX_MODEL_LEN,
    "max_num_seqs": MAX_NUM_SEQS,
    "gpu_memory_utilization": 0.90,
    "language_model_only": True,
}

try:
    llm = LLM(**llm_kwargs)
except TypeError as e:
    err_str = str(e)
    if "language_model_only" in err_str:
        llm_kwargs.pop("language_model_only", None)
    if "load_format" in err_str:
        llm_kwargs.pop("load_format", None)
    llm = LLM(**llm_kwargs)

tokenizer = llm.get_tokenizer()

# Load Gold Set
reviews = GoldSetManager.load_reviews()
print(f"\nLoaded {len(reviews)} gold benchmark reviews.")

SYSTEM_PROMPT = """You are an expert Aspect-Based Sentiment Analysis (ABSA) system specialized in e-commerce customer reviews.
Your task is to analyze customer feedback and extract fine-grained aspect sentiment quads.

Strict Rules:
1. 'aspect': MUST BE an exact verbatim substring noun/entity copied directly from the review text (e.g., "battery life", "fabric quality", "picture clarity", "bass").
   CRITICAL: NEVER set 'aspect' to a category enum name (DO NOT output "DISPLAY", "BATTERY", "SOUND", "FABRIC_MATERIAL", "FIT_SIZING", "VALUE", etc. as the aspect).
2. If an aspect is implicit (no specific noun term is named, e.g., "worth every penny", "paisa vasool", "terrible purchase", "too expensive"), set 'aspect' to null.
3. Map every aspect to the canonical category:
   - Electronics: DISPLAY, BATTERY, SOUND, PERFORMANCE, BUILD_QUALITY, CONNECTIVITY, SOFTWARE, DELIVERY_INSTALLATION, VALUE, OTHER.
   - Apparel: FABRIC_MATERIAL, FIT_SIZING, COLOR_PRINT, STITCHING_DURABILITY, COMFORT, DELIVERY, VALUE, OTHER.
4. Classify sentiment strictly into: "positive", "negative", "neutral", or "mixed".
5. Evidence MUST be an exact verbatim substring from the review text containing the opinion or rationale. Do not paraphrase or add punctuation.
6. Provide calibrated confidence between 0.0 and 1.0."""

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": (
            "Product Domain: electronics_tech\n"
            "Review Text:\n"
            '\"\"\"The OLED screen is incredibly bright and colors pop, but battery life drops quickly under heavy load.\"\"\"\n\n'
            "Extract all aspect quads as JSON adhering to the schema."
        ),
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "quads": [
                {
                    "aspect": "OLED screen",
                    "category": "DISPLAY",
                    "sentiment": "positive",
                    "confidence": 0.98,
                    "evidence": "The OLED screen is incredibly bright and colors pop",
                },
                {
                    "aspect": "battery life",
                    "category": "BATTERY",
                    "sentiment": "negative",
                    "confidence": 0.95,
                    "evidence": "battery life drops quickly under heavy load",
                },
            ]
        }),
    },
    {
        "role": "user",
        "content": (
            "Product Domain: apparel_fashion\n"
            "Review Text:\n"
            '\"\"\"The cotton fabric feels soft against the skin, but stitching came undone along the collar seam after two washes.\"\"\"\n\n'
            "Extract all aspect quads as JSON adhering to the schema."
        ),
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "quads": [
                {
                    "aspect": "cotton fabric",
                    "category": "FABRIC_MATERIAL",
                    "sentiment": "positive",
                    "confidence": 0.98,
                    "evidence": "The cotton fabric feels soft against the skin",
                },
                {
                    "aspect": "stitching",
                    "category": "STITCHING_DURABILITY",
                    "sentiment": "negative",
                    "confidence": 0.95,
                    "evidence": "stitching came undone along the collar seam after two washes",
                },
            ]
        }),
    },
    {
        "role": "user",
        "content": (
            "Product Domain: electronics_tech\n"
            "Review Text:\n"
            '\"\"\"Earphones ka bass dhamakedar hai lekin mic quality calls ke liye bilkul acchi nahi hai.\"\"\"\n\n'
            "Extract all aspect quads as JSON adhering to the schema."
        ),
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "quads": [
                {
                    "aspect": "bass",
                    "category": "SOUND",
                    "sentiment": "positive",
                    "confidence": 0.98,
                    "evidence": "Earphones ka bass dhamakedar hai",
                },
                {
                    "aspect": "mic quality",
                    "category": "SOUND",
                    "sentiment": "negative",
                    "confidence": 0.95,
                    "evidence": "mic quality calls ke liye bilkul acchi nahi hai",
                },
            ]
        }),
    },
    {
        "role": "user",
        "content": (
            "Product Domain: electronics_tech\n"
            "Review Text:\n"
            '\"\"\"Paisa vasool mobile hai bhai, definitely worth every penny.\"\"\"\n\n'
            "Extract all aspect quads as JSON adhering to the schema."
        ),
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "quads": [
                {
                    "aspect": None,
                    "category": "VALUE",
                    "sentiment": "positive",
                    "confidence": 0.95,
                    "evidence": "Paisa vasool mobile hai bhai, definitely worth every penny",
                },
            ]
        }),
    },
]

# Format prompts with enable_thinking=False
formatted_prompts = []
for r in reviews:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *FEW_SHOT_EXAMPLES,
        {"role": "user", "content": f"Product Domain: {r.vertical.value}\nReview Text:\n\"\"\"{r.text}\"\"\"\n\nExtract all aspect quads as JSON adhering to the schema."}
    ]
    prompt_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        chat_template_kwargs={"enable_thinking": False}
    )
    formatted_prompts.append(prompt_str)

# Generate with StructuredOutputsParams
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=384,
    structured_outputs=StructuredOutputsParams(json=schema_json),
)

print(f"\nGenerating structured quads for {len(formatted_prompts)} reviews...")
t0 = time.perf_counter()
outputs = llm.generate(formatted_prompts, sampling_params)
total_inference_sec = time.perf_counter() - t0
print(f"Generation completed in {total_inference_sec:.2f}s ({(total_inference_sec / len(reviews)) * 500:.2f}s per 500 reviews).")

# Deterministic Substring Validation & Saving
predictions_path = Path("/kaggle/working/teacher_gold_predictions.jsonl")
predicted_quads_by_id = {}
total_raw_quads = 0
total_valid_quads = 0

with open(predictions_path, "w", encoding="utf-8") as out_f:
    for review, out in zip(reviews, outputs):
        raw_text = out.outputs[0].text
        try:
            parsed_data = json.loads(raw_text)
            raw_quads = [TeacherAspectQuad.model_validate(q) for q in parsed_data.get("quads", [])]
        except Exception:
            raw_quads = []

        total_raw_quads += len(raw_quads)
        validated = []
        for tq in raw_quads:
            raw_c = RawCompactQuad(
                aspect=tq.aspect,
                category=tq.category,
                sentiment=Sentiment(tq.sentiment),
                confidence=tq.confidence,
                evidence=tq.evidence,
            )
            v_quad, err = SubstringValidator.validate_quad(review.text, raw_c, review.vertical)
            if v_quad is not None:
                validated.append(v_quad)

        total_valid_quads += len(validated)
        predicted_quads_by_id[review.review_id] = validated

        out_f.write(json.dumps({
            "review_id": review.review_id,
            "raw_text": raw_text,
            "validated_quads": [q.model_dump() for q in validated],
        }) + "\n")

validator_pass_rate = (total_valid_quads / total_raw_quads * 100) if total_raw_quads > 0 else 0.0

# Run Stage 1 Evaluation Harness
evaluator = Stage1Evaluator(reviews)
eval_results = evaluator.evaluate_predictions(lambda r: predicted_quads_by_id.get(r.review_id, []))

ov = eval_results["overall"]
print("\n" + "=" * 72)
print("📊 Qwen/Qwen3.5-9B Teacher Evaluation Report on Gold Benchmark")
print("=" * 72)
print(f"Total Reviews:                 {eval_results['total_reviews']}")
print(f"Raw Generated Quads:           {total_raw_quads}")
print(f"Validator Pass Rate:           {validator_pass_rate:.2f}% ({total_valid_quads}/{total_raw_quads})")
print(f"Explicit Matched Count:        {ov['explicit_matched_count']} / {ov['total_gold_explicit']}")
print(f"Implicit Matched Count:        {ov['implicit_matched_count']} / {ov['total_gold_implicit']}")
print(f"Total Matched Quads:           {ov['total_matched_quads']}")
print(f"--- Core Performance Metrics ---")
print(f"Overall Aspect F1 (Strict):    {ov['aspect_f1'] * 100:.2f}% (P: {ov['aspect_precision']*100:.2f}%, R: {ov['aspect_recall']*100:.2f}%)")
print(f"Overall Aspect F1 (IoU>=0.5):  {ov.get('overlap_aspect_f1', 0.0) * 100:.2f}% (P: {ov.get('overlap_aspect_precision', 0.0)*100:.2f}%, R: {ov.get('overlap_aspect_recall', 0.0)*100:.2f}%)")
print(f"Implicit Category F1:          {ov['implicit_category_f1'] * 100:.2f}% (P: {ov['implicit_category_precision']*100:.2f}%, R: {ov['implicit_category_recall']*100:.2f}%)")
print(f"Sentiment Macro-F1:            {ov['sentiment_macro_f1'] * 100:.2f}%")
print(f"Expected Calib Error (ECE):    {ov['expected_calibration_error']:.4f}")
print(f"--- Disaggregated Subsets ---")
print(f"English Aspect F1:             {eval_results['english']['aspect_f1'] * 100:.2f}% (IoU>=0.5: {eval_results['english'].get('overlap_aspect_f1', 0.0)*100:.2f}%) | Sent F1: {eval_results['english']['sentiment_macro_f1'] * 100:.2f}%")
print(f"Hinglish Aspect F1:            {eval_results['hinglish']['aspect_f1'] * 100:.2f}% (IoU>=0.5: {eval_results['hinglish'].get('overlap_aspect_f1', 0.0)*100:.2f}%) | Sent F1: {eval_results['hinglish']['sentiment_macro_f1'] * 100:.2f}%")
print(f"Hard Cases Aspect F1:          {eval_results['hard_cases']['aspect_f1'] * 100:.2f}% (IoU>=0.5: {eval_results['hard_cases'].get('overlap_aspect_f1', 0.0)*100:.2f}%) | Sent F1: {eval_results['hard_cases']['sentiment_macro_f1'] * 100:.2f}%")
print(f"Throughput (sec / 500 revs):   {(total_inference_sec / len(reviews)) * 500:.2f}s")
print("=" * 72)

# Save JSON metrics report to working dir
report_path = Path("/kaggle/working/teacher_gold_eval_results.json")
with open(report_path, "w", encoding="utf-8") as rf:
    json.dump({
        "model": MODEL_NAME,
        "validator_pass_rate": round(validator_pass_rate, 2),
        "total_raw_quads": total_raw_quads,
        "total_valid_quads": total_valid_quads,
        "eval_results": eval_results,
    }, rf, indent=2)

print(f"✅ Evaluation summary saved to {report_path}")